# Phase 25 — TensorFlow Training Delivery and Model API Handoff

This notebook is the single training-delivery notebook for TensorFlow model development, evaluation, export, and Model API handoff. It keeps runtime API servers, backend persistence, auth, job hydration, and external GenAI calls outside the notebook.


## Step 25.1 — Requirement and contract matrix

### Purpose
Map project requirements and backend API contracts before training cells are added, so each later notebook section has a clear deliverable and owner.

### Required input
- `REQUIREMENT.md` for TensorFlow, export, inference, API, GenAI, and deliverable requirements.
- `GAP_MODEL_TRAINING.md` for model-owned vs wrapper-owned output boundaries.
- `references/docs/generated/openapi.json` for `AnalyzeCvMultipartRequest` and `CvAnalysis.analysisResult` fields.
- Phase 23 model API contract validation artifacts for training-core handoff rules.

### Action
Create a compact requirement matrix and OpenAPI boundary table. Validate required OpenAPI schemas and fields directly from the local contract snapshot.

### Expected output
- Human-readable requirement and contract tables in the notebook.
- `reports/phase_25_requirement_contract_matrix.json`.
- `artifacts/phase_25_tensorflow_training_delivery/requirement_contract_matrix.json`.

### Verification
The executable cell fails if required OpenAPI schemas, request fields, response fields, score bounds, language enums, or owner mappings are missing.


## Step 25.2 — Single-notebook reproducibility setup

### Purpose
Define deterministic runtime setup before training, export, and handoff cells rely on hidden state.

### Required input
- Project root files: `GAP_MODEL_TRAINING.md`, `REQUIREMENT.md`, `GAP_MODEL_TRAINING.md`.
- Frozen contracts and reports from Phases 12-24.
- Training artifacts such as `artifacts/pairs_v2.parquet` and manual validation labels.

### Action
Resolve the repo root, seed Python/NumPy/TensorFlow when available, check dependencies and runtime versions, hash required datasets/contracts, define versioned artifact paths, and record clean-worktree status.

### Expected output
- Compact setup summary table in the notebook.
- `reports/phase_25_reproducibility_setup.json`.
- `artifacts/phase_25_tensorflow_training_delivery/reproducibility_setup.json`.

### Verification
The executable cell fails fast when required local files, required setup dependencies, hashes, version fields, or split seed are missing. Dirty worktree is recorded as a warning and blocks production-ready export later.


In [47]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from the repository or a child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_ROOT = ROOT / 'artifacts'
PHASE25_ARTIFACT_DIR = ARTIFACT_ROOT / 'phase_25_tensorflow_training_delivery'
PHASE25_MODEL_DIR = ARTIFACT_ROOT / 'models/phase_25_tensorflow_training_delivery'
PHASE25_TENSORBOARD_DIR = ARTIFACT_ROOT / 'tensorboard/phase_25_tensorflow_training_delivery'
PHASE25_FIXTURE_DIR = PHASE25_ARTIFACT_DIR / 'fixtures'
PHASE25_EXPORT_DIR = PHASE25_ARTIFACT_DIR / 'export'

for directory in [REPORTS, PHASE25_ARTIFACT_DIR, PHASE25_MODEL_DIR, PHASE25_TENSORBOARD_DIR, PHASE25_FIXTURE_DIR, PHASE25_EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SETUP_SCHEMA_VERSION = 'phase-25-reproducibility-setup-v1'
MODEL_VERSION = 'jobfit-tensorflow-phase25-v1'
FEATURE_CONFIG_VERSION = 'phase-25-feature-config-v1'
DATASET_MANIFEST_VERSION = 'phase-25-dataset-manifest-v1'
LABEL_MANIFEST_VERSION = 'phase-25-label-manifest-v1'
PYTHON_SEED = 202625
NUMPY_SEED = 202625
TENSORFLOW_SEED = 202625
GENERATED_AT = datetime.now(timezone.utc).isoformat()

os.environ['PYTHONHASHSEED'] = str(PYTHON_SEED)
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')
random.seed(PYTHON_SEED)


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def module_info(import_name: str, dist_name: str | None = None, required_now: bool = False) -> dict[str, Any]:
    spec = importlib.util.find_spec(import_name)
    version = None
    if spec is not None:
        try:
            version = importlib_metadata.version(dist_name or import_name)
        except importlib_metadata.PackageNotFoundError:
            version = 'installed-version-unknown'
    return {
        'importName': import_name,
        'distribution': dist_name or import_name,
        'installed': spec is not None,
        'version': version,
        'requiredNow': required_now,
    }


def git_output(args: list[str]) -> str:
    result = subprocess.run(['git', *args], cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)
    if result.returncode != 0:
        return f'git-error: {result.stderr.strip()}'
    return result.stdout.strip()


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print('(no rows)')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    sep = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))

# Seed NumPy and TensorFlow if available. TensorFlow is allowed to be missing until Step 25.5,
# but its absence is recorded here so the training gate cannot be missed silently.
dependency_specs = [
    ('numpy', 'numpy', True),
    ('pandas', 'pandas', True),
    ('pyarrow', 'pyarrow', True),
    ('sklearn', 'scikit-learn', True),
    ('scipy', 'scipy', True),
    ('tensorflow', 'tensorflow', False),
    ('tensorboard', 'tensorboard', False),
    ('sentence_transformers', 'sentence-transformers', False),
]
dependencies = [module_info(import_name, dist_name, required_now) for import_name, dist_name, required_now in dependency_specs]

if importlib.util.find_spec('numpy') is not None:
    import numpy as np
    np.random.seed(NUMPY_SEED)

TENSORFLOW_AVAILABLE = importlib.util.find_spec('tensorflow') is not None
if TENSORFLOW_AVAILABLE:
    import tensorflow as tf
    tf.random.set_seed(TENSORFLOW_SEED)

required_files = {
    'phase_plan': 'GAP_MODEL_TRAINING.md',
    'requirement': 'REQUIREMENT.md',
    'gap_model_training': 'GAP_MODEL_TRAINING.md',
    'openapi': 'references/docs/generated/openapi.json',
    'phase12_configs': 'reports/phase_12_notebook_configs.json',
    'phase13_snapshot_manifest': 'reports/phase_13_snapshot_manifests.json',
    'phase14_feature_quality': 'reports/phase_14_feature_quality_report.json',
    'phase15_pairs': 'artifacts/pairs_v2.parquet',
    'phase15_leakage': 'reports/phase_15_leakage_report.json',
    'phase16_label_manifest': 'reports/phase_16_label_manifest.json',
    'phase16_human_labels': 'artifacts/manual_validation/phase_16_human_labels_frozen.csv',
    'phase17_embedding_manifest': 'reports/phase_17_embedding_manifest.json',
    'phase18_embedding_manifest': 'reports/phase_18_embedding_manifest.json',
    'phase19_ats_labels': 'reports/phase_19_ats_issue_labels.json',
    'phase19_cv_benchmark': 'reports/phase_19_cv_benchmark_manifest.json',
    'phase21_candidate_reranking': 'reports/phase_21_backend_candidate_reranking.json',
    'phase22_feature_config': 'artifacts/phase_22_calibration_model_card_export/feature_config.json',
    'phase22_label_manifest': 'artifacts/phase_22_calibration_model_card_export/label_manifest.json',
    'phase22_artifact_manifest': 'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json',
    'phase23_contract': 'artifacts/phase_23_model_api_contract_validation/model_core_contract.json',
    'phase24_final_gate': 'reports/phase_24_reproducibility_final_gate.json',
}

file_manifest = []
for role, rel_path in required_files.items():
    path = ROOT / rel_path
    file_manifest.append({
        'role': role,
        'path': rel_path,
        'exists': path.exists(),
        'sha256': sha256(path),
    })

notebook_configs = load_json(ROOT / required_files['phase12_configs'])
feature_config = load_json(ROOT / required_files['phase22_feature_config'])
phase16_label_manifest = load_json(ROOT / required_files['phase16_label_manifest'])
phase22_label_manifest = load_json(ROOT / required_files['phase22_label_manifest'])
phase17_embedding_manifest = load_json(ROOT / required_files['phase17_embedding_manifest'])

jobfit_config = notebook_configs.get('jobfit_v2', {})
ats_config = notebook_configs.get('ats_quality_v1', {})
shared_config = notebook_configs.get('shared_features_schema_v1', {})
SPLIT_SEED = int(feature_config.get('split_seed') or jobfit_config.get('split_seed') or shared_config.get('split_seed'))

CONFIG_VERSIONS = {
    'setupSchemaVersion': SETUP_SCHEMA_VERSION,
    'modelVersion': MODEL_VERSION,
    'featureConfigVersion': FEATURE_CONFIG_VERSION,
    'datasetManifestVersion': DATASET_MANIFEST_VERSION,
    'labelManifestVersion': LABEL_MANIFEST_VERSION,
    'phase12JobfitSchemaVersion': jobfit_config.get('schema_version'),
    'phase12SharedSchemaVersion': shared_config.get('schema_version'),
    'phase22FeatureSchemaVersion': feature_config.get('schema_version'),
    'phase17EmbeddingSchemaVersion': phase17_embedding_manifest.get('schema_version'),
}

LABEL_VERSIONS = {
    'jobfitWeakLabel': jobfit_config.get('label_version'),
    'atsQualityLabel': ats_config.get('label_version'),
    'sharedBoundaryLabel': shared_config.get('label_version'),
    'humanValidationLabel': phase16_label_manifest.get('label_version'),
    'exportLabel': phase22_label_manifest.get('label_version'),
}

ARTIFACT_PATHS = {
    'phase25ArtifactDir': str(PHASE25_ARTIFACT_DIR.relative_to(ROOT)),
    'modelDir': str(PHASE25_MODEL_DIR.relative_to(ROOT)),
    'tensorboardDir': str(PHASE25_TENSORBOARD_DIR.relative_to(ROOT)),
    'fixtureDir': str(PHASE25_FIXTURE_DIR.relative_to(ROOT)),
    'exportDir': str(PHASE25_EXPORT_DIR.relative_to(ROOT)),
    'setupReport': 'reports/phase_25_reproducibility_setup.json',
    'setupArtifact': 'artifacts/phase_25_tensorflow_training_delivery/reproducibility_setup.json',
}

embedding_contract = {
    'embeddingModel': feature_config.get('embedding_model') or jobfit_config.get('embedding_contract', {}).get('embedding_model'),
    'profileCvPrefix': feature_config.get('embedding_prefixes', {}).get('profile_cv') or jobfit_config.get('embedding_contract', {}).get('profile_cv_prefix'),
    'jobPrefix': feature_config.get('embedding_prefixes', {}).get('job') or jobfit_config.get('embedding_contract', {}).get('job_prefix'),
    'normalizedEmbeddings': phase17_embedding_manifest.get('normalized_embeddings'),
    'productionEligibleE5': phase17_embedding_manifest.get('production_eligible_e5'),
    'embeddingDimension': phase17_embedding_manifest.get('embedding_dimension'),
}

git_porcelain = git_output(['status', '--porcelain'])
git_dirty_paths = [line[3:] if len(line) > 3 else line for line in git_porcelain.splitlines() if line]
git_head = git_output(['rev-parse', '--short', 'HEAD'])

summary_rows = [
    {'item': 'repo_root', 'value': str(ROOT), 'status': 'ok'},
    {'item': 'python', 'value': sys.version.split()[0], 'status': 'ok'},
    {'item': 'platform', 'value': f'{platform.system()} {platform.machine()}', 'status': 'ok'},
    {'item': 'seeds', 'value': f'python={PYTHON_SEED}, numpy={NUMPY_SEED}, tensorflow={TENSORFLOW_SEED}', 'status': 'ok'},
    {'item': 'split_seed', 'value': SPLIT_SEED, 'status': 'ok'},
    {'item': 'model_version', 'value': MODEL_VERSION, 'status': 'ok'},
    {'item': 'jobfit_label', 'value': LABEL_VERSIONS['jobfitWeakLabel'], 'status': 'ok'},
    {'item': 'human_label', 'value': LABEL_VERSIONS['humanValidationLabel'], 'status': 'ok'},
    {'item': 'ats_label', 'value': LABEL_VERSIONS['atsQualityLabel'], 'status': 'ok'},
    {'item': 'embedding', 'value': embedding_contract['embeddingModel'], 'status': 'ok' if embedding_contract['productionEligibleE5'] else 'blocked'},
    {'item': 'artifact_dir', 'value': ARTIFACT_PATHS['phase25ArtifactDir'], 'status': 'ok'},
    {'item': 'tensorboard_dir', 'value': ARTIFACT_PATHS['tensorboardDir'], 'status': 'ok'},
    {'item': 'git_dirty', 'value': len(git_dirty_paths), 'status': 'warn' if git_dirty_paths else 'ok'},
]

dependency_rows = [
    {
        'dependency': dep['importName'],
        'version': dep['version'] or 'missing',
        'required_now': dep['requiredNow'],
        'status': 'ok' if dep['installed'] else ('blocked' if dep['requiredNow'] else 'warn'),
    }
    for dep in dependencies
]

dataset_rows = [
    {
        'role': row['role'],
        'path': row['path'],
        'sha256': (row['sha256'] or 'missing')[:12],
        'status': 'ok' if row['exists'] and row['sha256'] else 'blocked',
    }
    for row in file_manifest
]

checks = {
    'required_files_exist': all(row['exists'] for row in file_manifest),
    'required_files_have_hashes': all(row['sha256'] for row in file_manifest),
    'required_setup_dependencies_installed': all(dep['installed'] for dep in dependencies if dep['requiredNow']),
    'split_seed_is_integer': isinstance(SPLIT_SEED, int),
    'config_versions_present': all(CONFIG_VERSIONS.values()),
    'label_versions_present': all(LABEL_VERSIONS.values()),
    'artifact_paths_defined': all(ARTIFACT_PATHS.values()),
    'e5_contract_present': embedding_contract['embeddingModel'] == 'intfloat/e5-base-v2' and embedding_contract['profileCvPrefix'] == 'query:' and embedding_contract['jobPrefix'] == 'passage:',
    'e5_contract_production_eligible': bool(embedding_contract['productionEligibleE5']),
}
failed_checks = [name for name, passed in checks.items() if not passed]
if failed_checks:
    raise AssertionError(f'Phase 25.2 reproducibility setup failed: {failed_checks}')

setup_report = {
    'schemaVersion': SETUP_SCHEMA_VERSION,
    'phaseId': PHASE_ID,
    'generatedAt': GENERATED_AT,
    'runtime': {
        'python': sys.version,
        'platform': platform.platform(),
        'gitCommit': git_head,
        'gitDirtyAtSetup': bool(git_dirty_paths),
        'gitDirtyPathCount': len(git_dirty_paths),
        'gitDirtyPaths': git_dirty_paths[:50],
    },
    'seeds': {
        'python': PYTHON_SEED,
        'numpy': NUMPY_SEED,
        'tensorflow': TENSORFLOW_SEED,
        'splitSeed': SPLIT_SEED,
    },
    'dependencies': dependencies,
    'configVersions': CONFIG_VERSIONS,
    'labelVersions': LABEL_VERSIONS,
    'embeddingContract': embedding_contract,
    'artifactPaths': ARTIFACT_PATHS,
    'datasetAndContractHashes': file_manifest,
    'checks': checks,
    'warnings': {
        'missingFutureTrainingDependencies': [dep['importName'] for dep in dependencies if not dep['installed'] and not dep['requiredNow']],
        'gitDirtyAtSetup': bool(git_dirty_paths),
        'productionReadyBlockedWhenDirty': bool(git_dirty_paths),
    },
    'status': 'passed_with_warnings' if git_dirty_paths or any(not dep['installed'] and not dep['requiredNow'] for dep in dependencies) else 'passed',
}

for rel_path in [ARTIFACT_PATHS['setupReport'], ARTIFACT_PATHS['setupArtifact']]:
    path = ROOT / rel_path
    path.write_text(json.dumps(setup_report, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print('Setup summary')
print_table(summary_rows, ['item', 'value', 'status'])
print('\nDependency checks')
print_table(dependency_rows, ['dependency', 'version', 'required_now', 'status'])
print('\nRequired dataset/contract hashes')
print_table(dataset_rows, ['role', 'path', 'sha256', 'status'])
print('\nValidation: passed')
for rel_path in [ARTIFACT_PATHS['setupReport'], ARTIFACT_PATHS['setupArtifact']]:
    print(f'- {rel_path}')


Setup summary
item            | value                                                       | status
----------------+-------------------------------------------------------------+-------
repo_root       | /Users/macbookpro/Development/bisakerja-model               | ok    
python          | 3.13.11                                                     | ok    
platform        | Darwin arm64                                                | ok    
seeds           | python=202625, numpy=202625, tensorflow=202625              | ok    
split_seed      | 202621                                                      | ok    
model_version   | jobfit-tensorflow-phase25-v1                                | ok    
jobfit_label    | weak-label-balanced-v2                                      | ok    
human_label     | human-validation-v1                                         | ok    
ats_label       | ats-quality-label-v1                                        | ok    
embedding       | intfloat/e5

In [48]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'GAP_MODEL_TRAINING.md').exists():
    ROOT = Path.cwd().parent.parent

REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
OPENAPI_PATH = ROOT / 'references/docs/generated/openapi.json'
PHASE23_CONTRACT_PATH = ROOT / 'artifacts/phase_23_model_api_contract_validation/model_core_contract.json'

REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-requirement-contract-matrix-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()


def sha256(path: Path) -> str | None:
    if not path.exists():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    sep = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))

openapi = load_json(OPENAPI_PATH)
schemas = openapi.get('components', {}).get('schemas', {})
phase23_contract = load_json(PHASE23_CONTRACT_PATH) if PHASE23_CONTRACT_PATH.exists() else {}

request_schema = schemas.get('AnalyzeCvMultipartRequest', {})
cv_analysis_schema = schemas.get('CvAnalysis', {})
analysis_result_schema = cv_analysis_schema.get('properties', {}).get('analysisResult', {})
analysis_result_properties = analysis_result_schema.get('properties', {})

requirement_matrix = [
    {
        'requirement': '1.1 TensorFlow deep learning architecture',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.5 builds Functional API or Model Subclassing job-fit scorer.',
        'evidence': 'Architecture summary, input tensor contract, selected model artifact.',
    },
    {
        'requirement': '1.2 Custom component',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.6 implements serializable custom layer/loss/callback.',
        'evidence': 'Registered custom object with get_config() and reload check.',
    },
    {
        'requirement': '1.3 Custom training loop',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.7 trains/evaluates with tf.GradientTape; model.fit() not used as main path.',
        'evidence': 'Loop metrics for train, validation, test, slices, and gates.',
    },
    {
        'requirement': '1.4 TensorBoard monitoring',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.8 writes bounded TensorBoard logs under artifacts/tensorboard/.',
        'evidence': 'Log path, hash references, scalar metrics, selected histograms.',
    },
    {
        'requirement': '1.5 Regression target MAE <= 0.02',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.9 applies strict selection gate on normalized 0-1 target.',
        'evidence': 'MAE <= 0.02 or readiness capped below production.',
    },
    {
        'requirement': '2.1 Export model',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.11 exports .keras or SavedModel.',
        'evidence': 'Artifact path, SHA-256, reload without hidden kernel state.',
    },
    {
        'requirement': '2.2 Inference',
        'owner': 'Notebook',
        'section_or_deliverable': 'Step 25.11 runs clean inference smoke tests; Step 25.12 exports JSON fixtures.',
        'evidence': 'Bounded JSON-compatible model-core outputs.',
    },
    {
        'requirement': '3.1 FastAPI/Flask REST API',
        'owner': 'Separate Model API deliverable',
        'section_or_deliverable': 'Notebook does not run a server; it exports schemas, model artifact, and fixtures for API code.',
        'evidence': 'Handoff fixtures and model-core contract.',
    },
    {
        'requirement': '3.2 API loads model, accepts input, runs inference, returns JSON',
        'owner': 'Separate Model API deliverable',
        'section_or_deliverable': 'API code must consume Step 25.11 artifact and Step 25.12 fixtures.',
        'evidence': 'Validated fixture request/response examples aligned with OpenAPI.',
    },
    {
        'requirement': '4.1 Generative AI secondary feature',
        'owner': 'Backend/API wrapper',
        'section_or_deliverable': 'Step 25.13 documents GenAI boundary; no external GenAI call from training cells.',
        'evidence': 'Deterministic summary signals and wrapper handoff contract.',
    },
    {
        'requirement': '5 Deliverables',
        'owner': 'Notebook plus separate API source',
        'section_or_deliverable': 'Steps 25.1-25.15 produce training source, custom component, GradientTape loop, TensorBoard logs, TensorFlow artifact, inference fixtures, reports, and README/deployment references.',
        'evidence': 'Final report and artifact manifest.',
    },
]

openapi_boundary = [
    {
        'contract_area': 'AnalyzeCvMultipartRequest',
        'public_fields': 'jobRoles, language, inputMode, compareSource, persistResult, cvFileId, cvFile',
        'training_boundary': 'Input contract only; backend validates multipart, auth, ownership, file limits, and persistence before Model API call.',
        'owner': 'Backend API',
    },
    {
        'contract_area': 'CvAnalysis.analysisResult',
        'public_fields': 'id, schemaVersion, jobFitAlignment, atsFriendliness, overallImpression, topActionables, sectionReviews, jobRecommendations, generatedCv, model, analyzedAt',
        'training_boundary': 'Notebook exports model-core fields and fixtures; backend maps wrapper-owned public response fields.',
        'owner': 'Backend API + Model core',
    },
    {
        'contract_area': 'jobFitAlignment',
        'public_fields': 'score 0-100, summary',
        'training_boundary': 'Model owns score and grounded alignment signals; wrapper may render product-safe summary copy.',
        'owner': 'Model core for score/signals; wrapper for final copy.',
    },
    {
        'contract_area': 'atsFriendliness',
        'public_fields': 'score 0-100, summary',
        'training_boundary': 'Model owns score and detected ATS issues; wrapper may render concise summary copy.',
        'owner': 'Model core for score/issues; wrapper for final copy.',
    },
    {
        'contract_area': 'overallImpression',
        'public_fields': 'string',
        'training_boundary': 'Model/training exports deterministic, evidence-grounded impression signal; wrapper may localize or polish without changing facts.',
        'owner': 'Model core for grounded signal; wrapper for final UX wording.',
    },
    {
        'contract_area': 'jobRecommendations',
        'public_fields': 'jobId, title, companyName, matchScore, reason, nextStep; max 5',
        'training_boundary': 'Backend supplies candidate set and hydrates job details. Model only ranks provided candidate IDs and emits bounded match scores/signals.',
        'owner': 'Backend API for retrieval/hydration/copy; model core for scoring/ranking.',
    },
]

checks = {
    'openapi_exists': OPENAPI_PATH.exists(),
    'phase23_contract_exists': PHASE23_CONTRACT_PATH.exists(),
    'has_AnalyzeCvMultipartRequest': 'AnalyzeCvMultipartRequest' in schemas,
    'has_CvAnalysis': 'CvAnalysis' in schemas,
    'request_required_fields_present': set(request_schema.get('required', [])) >= {'jobRoles', 'language', 'inputMode'},
    'request_optional_fields_present': {'compareSource', 'persistResult', 'cvFileId', 'cvFile'} <= set(request_schema.get('properties', {})),
    'request_language_enum_id_en': set(request_schema.get('properties', {}).get('language', {}).get('enum', [])) == {'id', 'en'},
    'request_job_roles_1_to_10': request_schema.get('properties', {}).get('jobRoles', {}).get('minItems') == 1 and request_schema.get('properties', {}).get('jobRoles', {}).get('maxItems') == 10,
    'analysis_schema_version_cv_analysis_v2': analysis_result_properties.get('schemaVersion', {}).get('const') == 'cv-analysis-v2',
    'analysis_required_fields_present': set(analysis_result_schema.get('required', [])) >= {'jobFitAlignment', 'atsFriendliness', 'overallImpression', 'jobRecommendations'},
    'jobfit_score_bounds_0_100': analysis_result_properties.get('jobFitAlignment', {}).get('properties', {}).get('score', {}).get('minimum') == 0 and analysis_result_properties.get('jobFitAlignment', {}).get('properties', {}).get('score', {}).get('maximum') == 100,
    'ats_score_bounds_0_100': analysis_result_properties.get('atsFriendliness', {}).get('properties', {}).get('score', {}).get('minimum') == 0 and analysis_result_properties.get('atsFriendliness', {}).get('properties', {}).get('score', {}).get('maximum') == 100,
    'overall_impression_is_string': analysis_result_properties.get('overallImpression', {}).get('type') == 'string',
    'job_recommendations_max_5': analysis_result_properties.get('jobRecommendations', {}).get('maxItems') == 5,
    'boundary_rows_cover_required_areas': {row['contract_area'] for row in openapi_boundary} >= {'AnalyzeCvMultipartRequest', 'CvAnalysis.analysisResult', 'jobFitAlignment', 'atsFriendliness', 'overallImpression', 'jobRecommendations'},
    'requirement_rows_cover_assignment_sections': len(requirement_matrix) == 11,
}

failed_checks = [name for name, passed in checks.items() if not passed]
if failed_checks:
    raise AssertionError(f'Requirement/contract matrix validation failed: {failed_checks}')

report = {
    'schemaVersion': SCHEMA_VERSION,
    'phaseId': PHASE_ID,
    'generatedAt': GENERATED_AT,
    'sources': {
        'requirement': {'path': 'REQUIREMENT.md', 'sha256': sha256(ROOT / 'REQUIREMENT.md')},
        'gapModelTraining': {'path': 'GAP_MODEL_TRAINING.md', 'sha256': sha256(ROOT / 'GAP_MODEL_TRAINING.md')},
        'openapi': {'path': 'references/docs/generated/openapi.json', 'sha256': sha256(OPENAPI_PATH)},
        'phase23Contract': {'path': 'artifacts/phase_23_model_api_contract_validation/model_core_contract.json', 'sha256': sha256(PHASE23_CONTRACT_PATH)},
    },
    'requirementMatrix': requirement_matrix,
    'openapiBoundary': openapi_boundary,
    'phase23MappingPolicy': phase23_contract.get('mappingPolicy', {}),
    'checks': checks,
    'status': 'passed',
}

report_paths = [
    REPORTS / 'phase_25_requirement_contract_matrix.json',
    ARTIFACT_DIR / 'requirement_contract_matrix.json',
]
for path in report_paths:
    path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print('Requirement matrix')
print_table(requirement_matrix, ['requirement', 'owner', 'section_or_deliverable'])
print('\nOpenAPI boundary')
print_table(openapi_boundary, ['contract_area', 'owner', 'training_boundary'])
print('\nValidation: passed')
for path in report_paths:
    print(f'- {path.relative_to(ROOT)}')


Requirement matrix
requirement                                                      | owner                             | section_or_deliverable                                                                                                                                                              
-----------------------------------------------------------------+-----------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1.1 TensorFlow deep learning architecture                        | Notebook                          | Step 25.5 builds Functional API or Model Subclassing job-fit scorer.                                                                                                                
1.2 Custom component                                             | Notebook                          | Step 25.6 implements seria

## Step 25.3 — Data and feature reuse from current notebooks

### Purpose
Load frozen Phase 13-16 data evidence and Phase 19/23 handoff fixtures before TensorFlow training uses any feature rows.

### Required input
- Phase 13 snapshot and model-core schema reports.
- `artifacts/pairs_v2.parquet` from Phase 15.
- Phase 14 feature-quality and normalization reports.
- Phase 16 frozen human validation labels.
- Phase 19 ATS benchmark labels.
- Phase 23 candidate-set fixtures.

### Action
Load the current artifacts, verify hashes against source reports, enforce required columns, reuse Phase 14 normalized language policy, enforce Phase 15 profile-level split isolation, and validate non-empty evidence fields.

### Expected output
- Compact data readiness tables.
- `reports/phase_25_data_feature_reuse.json`.
- `artifacts/phase_25_tensorflow_training_delivery/data_feature_reuse.json`.

### Verification
The executable cell fails on stale hashes, missing required columns, profile leakage, unsupported normalized languages, invalid scores, missing labels, empty required evidence, or candidate fixture violations.


In [49]:
from __future__ import annotations

import hashlib
import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from the repository or a child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_ROOT = ROOT / 'artifacts'
PHASE25_ARTIFACT_DIR = ARTIFACT_ROOT / 'phase_25_tensorflow_training_delivery'
PHASE25_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-data-feature-reuse-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()

PATHS = {
    'phase13_snapshot_manifests': REPORTS / 'phase_13_snapshot_manifests.json',
    'phase13_model_core_schema_contracts': REPORTS / 'phase_13_model_core_schema_contracts.json',
    'phase14_feature_quality_report': REPORTS / 'phase_14_feature_quality_report.json',
    'phase14_normalization_report': REPORTS / 'phase_14_normalization_feature_builder.json',
    'pairs_v2': ARTIFACT_ROOT / 'pairs_v2.parquet',
    'phase15_leakage_report': REPORTS / 'phase_15_leakage_report.json',
    'phase15_pair_distribution_diagnostics': REPORTS / 'phase_15_pair_distribution_diagnostics.json',
    'phase16_label_manifest': REPORTS / 'phase_16_label_manifest.json',
    'phase16_human_labels_frozen': ARTIFACT_ROOT / 'manual_validation/phase_16_human_labels_frozen.csv',
    'phase19_ats_issue_labels': REPORTS / 'phase_19_ats_issue_labels.json',
    'phase19_ats_benchmark': ARTIFACT_ROOT / 'ats_benchmark/phase_19_cv_benchmark.csv',
    'phase23_contract_fixtures': ARTIFACT_ROOT / 'phase_23_model_api_contract_validation/contract_fixtures.json',
}

PAIR_REQUIRED_COLUMNS = {
    'pair_id',
    'profile_id',
    'job_id',
    'pair_type',
    'split',
    'score_band',
    'job_fit_score',
    'skill_overlap',
    'requirement_coverage',
    'role_match',
    'experience_match',
    'experience_gap_years',
    'language',
    'role_family',
    'profile_role_family',
    'job_experience_band',
    'profile_experience_band',
    'experience_band',
    'matched_skill_count',
    'missing_skill_count',
    'matched_skills',
    'missing_skills',
    'label_version',
    'schema_version',
}

HUMAN_LABEL_REQUIRED_COLUMNS = {
    'review_item_id',
    'pair_id',
    'reviewer_id',
    'reviewed_at',
    'label_version',
    'reviewer_job_fit_score',
    'reviewer_job_fit_band',
    'recommendation_relevance',
    'unsupported_claim_flag',
    'disagreement_flag',
    'evidence_notes',
}

ATS_REQUIRED_COLUMNS = {
    'fixture_id',
    'case_family',
    'file_family',
    'supported_file_type',
    'extracted_text',
    'word_count',
    'section_count',
    'parser_error',
    'unsupported_file_type',
    'label_issue_keys',
    'label_ats_score',
    'label_ats_bucket',
    'label_version',
}

REQUIRED_PAIR_TYPES = {
    'high_fit_positive',
    'medium_fit',
    'hard_negative',
    'random_negative',
    'same_role_different_seniority',
    'cross_role_confusing',
}
SUPPORTED_NORMALIZED_LANGUAGES = {'ID', 'EN', 'MIXED', 'UNKNOWN'}
REQUIRED_SPLITS = {'train', 'validation', 'test'}
REQUIRED_SCORE_BANDS = {'low', 'medium', 'high'}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def require(condition: bool, message: str, failures: list[str]) -> None:
    if not condition:
        failures.append(message)


def parse_json_list(value: Any) -> list[Any]:
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        parsed = json.loads(text)
        if not isinstance(parsed, list):
            raise ValueError(f'Expected JSON list, got {type(parsed).__name__}')
        return parsed
    raise ValueError(f'Expected list-like value, got {type(value).__name__}')


def compact_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print('(no rows)')
        return
    try:
        display(pd.DataFrame(rows, columns=columns))  # type: ignore[name-defined]
    except NameError:
        print(pd.DataFrame(rows, columns=columns).to_string(index=False))


failures: list[str] = []
for name, path in PATHS.items():
    require(path.exists(), f'Missing required artifact: {name} -> {path.relative_to(ROOT)}', failures)

if failures:
    raise RuntimeError('Phase 25.3 preflight failed:\n- ' + '\n- '.join(failures))

phase13_snapshots = load_json(PATHS['phase13_snapshot_manifests'])
phase13_schema = load_json(PATHS['phase13_model_core_schema_contracts'])
phase14_quality = load_json(PATHS['phase14_feature_quality_report'])
phase14_normalization = load_json(PATHS['phase14_normalization_report'])
phase15_leakage = load_json(PATHS['phase15_leakage_report'])
phase15_distribution = load_json(PATHS['phase15_pair_distribution_diagnostics'])
phase16_manifest = load_json(PATHS['phase16_label_manifest'])
phase19_ats_labels = load_json(PATHS['phase19_ats_issue_labels'])
phase23_fixtures = load_json(PATHS['phase23_contract_fixtures'])

pairs = pd.read_parquet(PATHS['pairs_v2'])
human_labels = pd.read_csv(PATHS['phase16_human_labels_frozen'])
ats_benchmark = pd.read_csv(PATHS['phase19_ats_benchmark'])

pairs_hash = sha256(PATHS['pairs_v2'])
human_labels_hash = sha256(PATHS['phase16_human_labels_frozen'])
ats_benchmark_hash = sha256(PATHS['phase19_ats_benchmark'])
contract_fixtures_hash = sha256(PATHS['phase23_contract_fixtures'])

# Hash freshness checks against frozen upstream reports.
expected_pair_hashes = {
    phase15_leakage.get('artifact', {}).get('sha256'),
    phase15_distribution.get('artifact', {}).get('sha256'),
}
expected_pair_hashes.discard(None)
require(bool(expected_pair_hashes), 'Phase 15 reports do not expose pairs_v2 hash.', failures)
require(pairs_hash in expected_pair_hashes, 'pairs_v2.parquet hash does not match Phase 15 leakage/distribution reports.', failures)
require(
    human_labels_hash == phase16_manifest.get('human_labels', {}).get('sha256'),
    'Frozen human-label hash does not match Phase 16 label manifest.',
    failures,
)

# Pair schema, value, split, leakage, and evidence checks.
missing_pair_columns = sorted(PAIR_REQUIRED_COLUMNS - set(pairs.columns))
require(not missing_pair_columns, f'pairs_v2 missing required columns: {missing_pair_columns}', failures)
if not missing_pair_columns:
    require(not pairs[list(PAIR_REQUIRED_COLUMNS)].isna().any().any(), 'pairs_v2 has null values in required columns.', failures)
    require(pairs['pair_id'].is_unique, 'pairs_v2 pair_id values are not unique.', failures)
    require(set(pairs['split'].unique()) >= REQUIRED_SPLITS, 'pairs_v2 does not contain train/validation/test splits.', failures)
    require(set(pairs['pair_type'].unique()) >= REQUIRED_PAIR_TYPES, 'pairs_v2 does not contain every required pair_type.', failures)
    require(set(pairs['score_band'].unique()) >= REQUIRED_SCORE_BANDS, 'pairs_v2 does not contain low/medium/high score bands.', failures)
    require(pairs['job_fit_score'].between(0, 1).all(), 'pairs_v2 job_fit_score must stay within 0-1.', failures)
    unsupported_languages = sorted(set(pairs['language'].dropna().astype(str)) - SUPPORTED_NORMALIZED_LANGUAGES)
    require(not unsupported_languages, f'pairs_v2 contains unsupported normalized languages: {unsupported_languages}', failures)

    split_counts_by_profile = pairs.groupby('profile_id')['split'].nunique()
    leaking_profiles = split_counts_by_profile[split_counts_by_profile > 1]
    require(leaking_profiles.empty, f'Profile-level split leakage detected for {len(leaking_profiles)} profile_id values.', failures)
    require(phase15_leakage.get('passed') is True and phase15_leakage.get('leaking_profile_count') == 0, 'Phase 15 leakage report is not passing.', failures)

    parsed_matched = pairs['matched_skills'].map(parse_json_list)
    parsed_missing = pairs['missing_skills'].map(parse_json_list)
    empty_skill_evidence = (parsed_matched.map(len) + parsed_missing.map(len) == 0)
    require(not empty_skill_evidence.any(), f'pairs_v2 has empty matched/missing skill evidence for {int(empty_skill_evidence.sum())} rows.', failures)

    required_numeric_features = ['skill_overlap', 'requirement_coverage', 'role_match', 'experience_match', 'experience_gap_years']
    finite_numeric = pairs[required_numeric_features].apply(pd.to_numeric, errors='coerce').notna().all().all()
    require(bool(finite_numeric), 'pairs_v2 required numeric feature columns contain non-numeric values.', failures)

    pair_type_split = pairs.groupby(['split', 'pair_type']).size().unstack(fill_value=0)
    for split_name in ['validation', 'test']:
        missing_types_in_split = sorted(REQUIRED_PAIR_TYPES - set(pair_type_split.columns[pair_type_split.loc[split_name] > 0]))
        require(not missing_types_in_split, f'{split_name} split missing pair types: {missing_types_in_split}', failures)

# Phase 14 normalization and feature-quality gates.
require(phase14_normalization.get('passed') is True, 'Phase 14 normalization report is not passing.', failures)
phase14_checks = phase14_normalization.get('checks', {})
require(phase14_checks.get('experience_no_observed_fallthrough') is True, 'Phase 14 experience normalization has observed fallthrough.', failures)
require(phase14_checks.get('language_values_supported') is True, 'Phase 14 language normalization does not support current values.', failures)
require(phase14_checks.get('text_builder_fixtures_non_empty') is True, 'Phase 14 text-builder fixtures are empty.', failures)
require(phase14_quality.get('unknown_experience_observed_value_count') == 0, 'Feature-quality report has unknown experience values.', failures)

# Human label validation: evaluation-only, joined to pairs, complete band coverage, non-empty notes.
missing_human_columns = sorted(HUMAN_LABEL_REQUIRED_COLUMNS - set(human_labels.columns))
require(not missing_human_columns, f'Human labels missing required columns: {missing_human_columns}', failures)
if not missing_human_columns:
    require(not human_labels[list(HUMAN_LABEL_REQUIRED_COLUMNS)].isna().any().any(), 'Human labels have null values in required columns.', failures)
    require(set(human_labels['pair_id']).issubset(set(pairs['pair_id'])), 'Human labels reference pair_id values absent from pairs_v2.', failures)
    require(set(human_labels['reviewer_job_fit_band'].unique()) >= REQUIRED_SCORE_BANDS, 'Human labels do not cover low/medium/high bands.', failures)
    require(human_labels['evidence_notes'].fillna('').str.strip().ne('').all(), 'Human labels contain empty evidence_notes.', failures)
    require(phase16_manifest.get('evaluation_only_policy', {}).get('manual_labels_are_model_inputs') is False, 'Phase 16 manual labels are incorrectly allowed as model inputs.', failures)
    require(not phase16_manifest.get('blockers'), 'Phase 16 label manifest contains blockers.', failures)

# ATS benchmark labels: taxonomy complete, score bounds, supported parse evidence present.
missing_ats_columns = sorted(ATS_REQUIRED_COLUMNS - set(ats_benchmark.columns))
require(not missing_ats_columns, f'ATS benchmark missing required columns: {missing_ats_columns}', failures)
if not missing_ats_columns:
    require(phase19_ats_labels.get('missing_required_keys') == [], 'ATS issue taxonomy is missing required keys.', failures)
    require(ats_benchmark['fixture_id'].is_unique, 'ATS benchmark fixture_id values are not unique.', failures)
    require(ats_benchmark['label_ats_score'].between(0, 100).all(), 'ATS benchmark label_ats_score must stay within 0-100.', failures)
    require(set(ats_benchmark['label_ats_bucket'].unique()) >= REQUIRED_SCORE_BANDS, 'ATS benchmark does not cover low/medium/high buckets.', failures)
    supported_non_error = (
        ats_benchmark['supported_file_type'].astype(bool)
        & ~ats_benchmark['parser_error'].astype(bool)
        & ~ats_benchmark['unsupported_file_type'].astype(bool)
    )
    empty_supported_text = ats_benchmark.loc[supported_non_error, 'extracted_text'].fillna('').str.strip().eq('')
    require(not empty_supported_text.any(), f'ATS benchmark has empty text for {int(empty_supported_text.sum())} supported non-error fixtures.', failures)
    parsed_issue_keys = ats_benchmark['label_issue_keys'].map(parse_json_list)
    required_issue_keys = set(phase19_ats_labels.get('required_issue_keys', []))
    unknown_issue_keys = sorted({key for keys in parsed_issue_keys for key in keys} - required_issue_keys - {'unsupported_file_type'})
    require(not unknown_issue_keys, f'ATS benchmark contains unknown issue keys: {unknown_issue_keys}', failures)

# Candidate-set fixtures: backend provides candidates; model output must score only provided job IDs.
positive_fixtures = phase23_fixtures.get('positive', {})
candidate_request = positive_fixtures.get('candidateRerankingRequest', {})
candidate_response = positive_fixtures.get('candidateRerankingResponse', {})
job_candidates = candidate_request.get('jobCandidates', [])
recommendations = candidate_response.get('recommendations', [])
input_job_ids = [job.get('jobId') for job in job_candidates]
output_job_ids = [item.get('jobId') for item in recommendations]
require(isinstance(job_candidates, list) and len(job_candidates) > 0, 'Candidate reranking request has no jobCandidates.', failures)
require(isinstance(recommendations, list) and len(recommendations) > 0, 'Candidate reranking response has no recommendations.', failures)
require(len(input_job_ids) == len(set(input_job_ids)), 'Candidate reranking request has duplicate jobId values.', failures)
require(set(output_job_ids).issubset(set(input_job_ids)), 'Candidate reranking response emits jobId outside backend candidate set.', failures)
for candidate in job_candidates:
    require(bool(candidate.get('jobId')), 'Candidate fixture has empty jobId.', failures)
    require(bool(candidate.get('skills') or candidate.get('requirements')), f"Candidate fixture {candidate.get('jobId')} has empty skills and requirements.", failures)
for item in recommendations:
    require(0 <= float(item.get('matchScore', -1)) <= 100, f"Recommendation {item.get('jobId')} has out-of-range matchScore.", failures)
    require('matchedSkills' in item and 'missingSkills' in item, f"Recommendation {item.get('jobId')} missing skill evidence fields.", failures)

# Phase 13 schema/data snapshot presence checks used by later model-card export.
require(set(phase13_snapshots.keys()) >= {'jobs_v1', 'profiles_v1', 'legacy_pairs_v1'}, 'Phase 13 snapshots missing expected dataset manifests.', failures)
model_core_schemas = phase13_schema.get('model_core_output_schemas', {})
require(set(model_core_schemas.keys()) >= {'jobFitAlignment', 'atsFriendliness', 'overallImpression', 'recommendationRanking'}, 'Phase 13 model-core schemas missing required outputs.', failures)

if failures:
    raise RuntimeError('Phase 25.3 validation failed:\n- ' + '\n- '.join(failures))

pair_summary_rows = [
    {'metric': 'pairs_rows', 'value': int(len(pairs))},
    {'metric': 'profiles', 'value': int(pairs['profile_id'].nunique())},
    {'metric': 'jobs', 'value': int(pairs['job_id'].nunique())},
    {'metric': 'splits', 'value': dict(Counter(pairs['split']))},
    {'metric': 'languages', 'value': dict(Counter(pairs['language']))},
    {'metric': 'score_bands', 'value': dict(Counter(pairs['score_band']))},
    {'metric': 'pair_types', 'value': dict(Counter(pairs['pair_type']))},
]

validation_rows = [
    {'check': 'artifact_files_present', 'status': 'PASS'},
    {'check': 'pairs_v2_hash_matches_phase15', 'status': 'PASS'},
    {'check': 'phase14_normalization_reused', 'status': 'PASS'},
    {'check': 'profile_split_isolation', 'status': 'PASS'},
    {'check': 'normalized_languages_supported', 'status': 'PASS'},
    {'check': 'pair_skill_evidence_non_empty', 'status': 'PASS'},
    {'check': 'human_labels_hash_and_join', 'status': 'PASS'},
    {'check': 'ats_benchmark_labels_valid', 'status': 'PASS'},
    {'check': 'candidate_fixture_membership', 'status': 'PASS'},
]

readiness_report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete',
    'passed': True,
    'source_artifacts': {
        name: {
            'path': str(path.relative_to(ROOT)),
            'sha256': sha256(path),
            'bytes': path.stat().st_size,
        }
        for name, path in PATHS.items()
    },
    'pair_dataset': {
        'path': str(PATHS['pairs_v2'].relative_to(ROOT)),
        'sha256': pairs_hash,
        'row_count': int(len(pairs)),
        'profile_count': int(pairs['profile_id'].nunique()),
        'job_count': int(pairs['job_id'].nunique()),
        'split_counts': {str(k): int(v) for k, v in pairs['split'].value_counts().sort_index().items()},
        'language_counts': {str(k): int(v) for k, v in pairs['language'].value_counts().sort_index().items()},
        'score_band_counts': {str(k): int(v) for k, v in pairs['score_band'].value_counts().sort_index().items()},
        'pair_type_counts': {str(k): int(v) for k, v in pairs['pair_type'].value_counts().sort_index().items()},
        'score_range': [float(pairs['job_fit_score'].min()), float(pairs['job_fit_score'].max())],
        'label_versions': sorted(map(str, pairs['label_version'].unique())),
        'schema_versions': sorted(map(str, pairs['schema_version'].unique())),
    },
    'feature_quality': {
        'report_path': str(PATHS['phase14_feature_quality_report'].relative_to(ROOT)),
        'normalization_report_path': str(PATHS['phase14_normalization_report'].relative_to(ROOT)),
        'unknown_language_rate': phase14_quality.get('unknown_language_rate'),
        'unknown_experience_observed_value_count': phase14_quality.get('unknown_experience_observed_value_count'),
        'checks': phase14_checks,
        'supported_normalized_languages': sorted(SUPPORTED_NORMALIZED_LANGUAGES),
    },
    'split_isolation': {
        'phase15_report_path': str(PATHS['phase15_leakage_report'].relative_to(ROOT)),
        'leaking_profile_count': 0,
        'split_profile_counts': phase15_leakage.get('split_profile_counts'),
    },
    'human_validation': {
        'path': str(PATHS['phase16_human_labels_frozen'].relative_to(ROOT)),
        'sha256': human_labels_hash,
        'row_count': int(len(human_labels)),
        'review_item_count': int(human_labels['review_item_id'].nunique()),
        'reviewer_count': int(human_labels['reviewer_id'].nunique()),
        'band_counts': {str(k): int(v) for k, v in human_labels['reviewer_job_fit_band'].value_counts().sort_index().items()},
        'label_versions': sorted(map(str, human_labels['label_version'].unique())),
        'evaluation_only': True,
    },
    'ats_benchmark': {
        'path': str(PATHS['phase19_ats_benchmark'].relative_to(ROOT)),
        'sha256': ats_benchmark_hash,
        'row_count': int(len(ats_benchmark)),
        'bucket_counts': {str(k): int(v) for k, v in ats_benchmark['label_ats_bucket'].value_counts().sort_index().items()},
        'label_versions': sorted(map(str, ats_benchmark['label_version'].unique())),
        'required_issue_keys': phase19_ats_labels.get('required_issue_keys', []),
    },
    'candidate_set_fixtures': {
        'path': str(PATHS['phase23_contract_fixtures'].relative_to(ROOT)),
        'sha256': contract_fixtures_hash,
        'request_job_count': len(job_candidates),
        'response_recommendation_count': len(recommendations),
        'candidate_set_id': candidate_request.get('candidateSetId'),
        'output_jobs_subset_of_input': True,
    },
    'validation_checks': validation_rows,
    'failures': [],
}

for output_path in [
    REPORTS / 'phase_25_data_feature_reuse.json',
    PHASE25_ARTIFACT_DIR / 'data_feature_reuse.json',
]:
    output_path.write_text(json.dumps(readiness_report, indent=2, sort_keys=True), encoding='utf-8')

print('Phase 25.3 data + feature reuse gate: PASS')
compact_table(validation_rows, ['check', 'status'])
compact_table(pair_summary_rows, ['metric', 'value'])
compact_table(
    [
        {'artifact': 'human_labels_frozen', 'rows': len(human_labels), 'hash': human_labels_hash[:12]},
        {'artifact': 'ats_benchmark', 'rows': len(ats_benchmark), 'hash': ats_benchmark_hash[:12]},
        {'artifact': 'candidate_fixture_jobs', 'rows': len(job_candidates), 'hash': contract_fixtures_hash[:12]},
    ],
    ['artifact', 'rows', 'hash'],
)
print(f"Wrote {str((REPORTS / 'phase_25_data_feature_reuse.json').relative_to(ROOT))}")
print(f"Wrote {str((PHASE25_ARTIFACT_DIR / 'data_feature_reuse.json').relative_to(ROOT))}")


Phase 25.3 data + feature reuse gate: PASS


,check,status
0,artifact_files_present,PASS
1,pairs_v2_hash_matches_phase15,PASS
2,phase14_normalization_reused,PASS
3,profile_split_isolation,PASS
4,normalized_languages_supported,PASS
5,pair_skill_evidence_non_empty,PASS
6,human_labels_hash_and_join,PASS
7,ats_benchmark_labels_valid,PASS
8,candidate_fixture_membership,PASS


,metric,value
0,pairs_rows,3600
1,profiles,920
2,jobs,813
3,splits,"{'train': 2520, 'validation': 540, 'test': 540}"
4,languages,"{'UNKNOWN': 1304, 'EN': 2260, 'ID': 22, 'MIXED..."
5,score_bands,"{'high': 600, 'medium': 602, 'low': 2398}"
6,pair_types,"{'high_fit_positive': 600, 'medium_fit': 600, ..."


,artifact,rows,hash
0,human_labels_frozen,240,a5587a58447c
1,ats_benchmark,21,e28618309571
2,candidate_fixture_jobs,3,d4cf2d04af54


Wrote reports/phase_25_data_feature_reuse.json
Wrote artifacts/phase_25_tensorflow_training_delivery/data_feature_reuse.json


## Step 25.4 — E5 embedding contract

### Purpose
Validate the production E5 embedding cache before TensorFlow feature construction.

### Required input
- `artifacts/pairs_v2.parquet`.
- `artifacts/phase_25_tensorflow_training_delivery/embeddings/e5_pair_embeddings_v1.npz`.
- Upstream E5 manifests from Phases 17, 18, and 19.5.

### Action
Validate `intfloat/e5-base-v2`, `query:`/`passage:` prefixes, normalized finite vectors, fixed shape, cache metadata hashes, and explicit no-fallback policy. Export compact E5 pair features.

### Expected output
- `reports/phase_25_e5_embedding_contract.json`.
- `artifacts/phase_25_tensorflow_training_delivery/e5_embedding_contract.json`.
- `artifacts/phase_25_tensorflow_training_delivery/e5_pair_features.parquet`.

### Verification
The executable cell fails if the cache is missing/stale, embeddings are non-finite or unnormalized, prefixes/model metadata do not match, source hashes differ from upstream E5 evidence, or fallback evidence is detected.


In [50]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import importlib.metadata as importlib_metadata


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
EMBEDDING_DIR = ARTIFACT_DIR / 'embeddings'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-e5-embedding-contract-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
EMBEDDING_MODEL = 'intfloat/e5-base-v2'
PROFILE_PREFIX = 'query:'
JOB_PREFIX = 'passage:'
EXPECTED_DIM = 768

PATHS = {
    'pairs': ROOT / 'artifacts/pairs_v2.parquet',
    'profiles': ROOT / 'legacy/dataset/techtalent_profile_cleaned.csv',
    'jobs': ROOT / 'legacy/dataset/indotech_job_cleaned.csv',
    'cache': EMBEDDING_DIR / 'e5_pair_embeddings_v1.npz',
    'features': ARTIFACT_DIR / 'e5_pair_features.parquet',
    'phase17_embedding_manifest': REPORTS / 'phase_17_embedding_manifest.json',
    'phase18_embedding_manifest': REPORTS / 'phase_18_embedding_manifest.json',
    'phase19_5_e5_runtime_manifest': REPORTS / 'phase_19_5_e5_runtime_manifest.json',
}
REPORT_PATH = REPORTS / 'phase_25_e5_embedding_contract.json'
ARTIFACT_PATH = ARTIFACT_DIR / 'e5_embedding_contract.json'


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    print(' | '.join(col.ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


missing = [name for name, path in PATHS.items() if name not in {'features'} and not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing E5 contract inputs: {missing}')

pairs = pd.read_parquet(PATHS['pairs']).copy()
required_pair_columns = {'pair_id', 'split'}
missing_columns = sorted(required_pair_columns - set(pairs.columns))
if missing_columns:
    raise RuntimeError(f'pairs_v2 missing required columns: {missing_columns}')

cache = np.load(PATHS['cache'], allow_pickle=True)
cache_keys = set(cache.files)
required_cache_keys = {'query_embeddings', 'passage_embeddings', 'pair_id', 'metadata_json'}
missing_cache_keys = sorted(required_cache_keys - cache_keys)
if missing_cache_keys:
    raise RuntimeError(f'E5 cache missing keys: {missing_cache_keys}')

query_embeddings = cache['query_embeddings'].astype('float32')
passage_embeddings = cache['passage_embeddings'].astype('float32')
cache_pair_ids = cache['pair_id'].astype(str)
metadata = json.loads(str(cache['metadata_json'].item()))

failures: list[dict[str, Any]] = []
def check(name: str, passed: bool, detail: Any = None) -> None:
    if not passed:
        failures.append({'check': name, 'detail': detail})

check('row_count_matches_pairs', len(cache_pair_ids) == len(pairs), {'cache': int(len(cache_pair_ids)), 'pairs': int(len(pairs))})
check('pair_id_order_matches_pairs', cache_pair_ids.tolist() == pairs['pair_id'].astype(str).tolist())
check('fixed_embedding_shape_3600x768', query_embeddings.shape == (len(pairs), EXPECTED_DIM) and passage_embeddings.shape == (len(pairs), EXPECTED_DIM), {'query_shape': query_embeddings.shape, 'passage_shape': passage_embeddings.shape})
check('finite_embeddings', bool(np.isfinite(query_embeddings).all() and np.isfinite(passage_embeddings).all()))
query_norms = np.linalg.norm(query_embeddings, axis=1)
passage_norms = np.linalg.norm(passage_embeddings, axis=1)
check('normalized_embeddings', bool(np.allclose(query_norms, 1.0, atol=1e-3) and np.allclose(passage_norms, 1.0, atol=1e-3)))
check('embedding_model_metadata', metadata.get('embedding_model') == EMBEDDING_MODEL, metadata.get('embedding_model'))
check('query_passage_prefixes', metadata.get('profile_prefix') == PROFILE_PREFIX and metadata.get('job_prefix') == JOB_PREFIX, metadata)
check('fallback_backend_forbidden', metadata.get('backend') == 'sentence-transformers', metadata.get('backend'))
check('pairs_hash_matches_cache', metadata.get('pairs_sha256') == sha256(PATHS['pairs']), {'cache': metadata.get('pairs_sha256'), 'actual': sha256(PATHS['pairs'])})

upstream_evidence: dict[str, Any] = {}
for key in ['phase17_embedding_manifest', 'phase18_embedding_manifest', 'phase19_5_e5_runtime_manifest']:
    payload = load_json(PATHS[key])
    upstream_evidence[key] = {
        'path': str(PATHS[key].relative_to(ROOT)),
        'sha256': sha256(PATHS[key]),
        'production_eligible_e5': bool(payload.get('production_eligible_e5') or payload.get('embedding_manifest', {}).get('production_eligible_e5')),
        'source_text_hash': payload.get('source_text_hash') or payload.get('embedding_manifest', {}).get('source_text_hash'),
    }

source_hashes = {value.get('source_text_hash') for value in upstream_evidence.values() if value.get('source_text_hash')}
check('source_text_hash_matches_upstream', not source_hashes or metadata.get('source_text_hash') in source_hashes, {'cache': metadata.get('source_text_hash'), 'upstream': sorted(source_hashes)})
check('upstream_e5_production_eligible', all(value.get('production_eligible_e5') for value in upstream_evidence.values()))

cosine = np.sum(query_embeddings * passage_embeddings, axis=1).astype('float32')
feature_frame = pd.DataFrame({
    'pair_id': cache_pair_ids,
    'split': pairs['split'].astype(str).to_numpy(),
    'e5_cosine': cosine,
    'embedding_model': EMBEDDING_MODEL,
    'source_text_hash': metadata.get('source_text_hash'),
    'schema_version': SCHEMA_VERSION,
})
feature_frame.to_parquet(PATHS['features'], index=False)

try:
    st_version = importlib_metadata.version('sentence-transformers')
except importlib_metadata.PackageNotFoundError:
    st_version = None

validation_checks = [
    {'check': 'real_sentence_transformers_e5_backend', 'status': 'PASS' if metadata.get('backend') == 'sentence-transformers' else 'FAIL'},
    {'check': 'query_passage_prefixes', 'status': 'PASS' if metadata.get('profile_prefix') == PROFILE_PREFIX and metadata.get('job_prefix') == JOB_PREFIX else 'FAIL'},
    {'check': 'source_text_hash_matches_phase17_phase18', 'status': 'PASS' if not source_hashes or metadata.get('source_text_hash') in source_hashes else 'FAIL'},
    {'check': 'cache_metadata_matches_pairs_and_text_hash', 'status': 'PASS' if metadata.get('pairs_sha256') == sha256(PATHS['pairs']) else 'FAIL'},
    {'check': 'fixed_embedding_shape_3600x768', 'status': 'PASS' if query_embeddings.shape == (len(pairs), EXPECTED_DIM) and passage_embeddings.shape == (len(pairs), EXPECTED_DIM) else 'FAIL'},
    {'check': 'finite_normalized_embeddings', 'status': 'PASS' if np.isfinite(query_embeddings).all() and np.isfinite(passage_embeddings).all() and np.allclose(query_norms, 1.0, atol=1e-3) and np.allclose(passage_norms, 1.0, atol=1e-3) else 'FAIL'},
    {'check': 'fallback_backend_forbidden', 'status': 'PASS' if metadata.get('backend') == 'sentence-transformers' else 'FAIL'},
]

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if not failures else 'failed',
    'passed': not failures,
    'embedding_contract': {
        'embedding_model': EMBEDDING_MODEL,
        'profile_prefix': PROFILE_PREFIX,
        'job_prefix': JOB_PREFIX,
        'normalized_embeddings': True,
        'embedding_dimension': EXPECTED_DIM,
        'backend': 'sentence-transformers',
        'sentence_transformers_version': st_version,
        'batch_size': int(metadata.get('batch_size', 64)),
        'production_eligible_e5': True,
        'fallback_policy': 'TF-IDF, local-hash, and any fallback backend are forbidden for staging or production evidence.',
    },
    'cache': {'path': str(PATHS['cache'].relative_to(ROOT)), 'sha256': sha256(PATHS['cache']), 'status': 'validated_existing', 'metadata': metadata},
    'source_text_contract': {
        'pairs_path': str(PATHS['pairs'].relative_to(ROOT)),
        'pairs_sha256': sha256(PATHS['pairs']),
        'profiles_path': str(PATHS['profiles'].relative_to(ROOT)),
        'jobs_path': str(PATHS['jobs'].relative_to(ROOT)),
        'pair_count': int(len(pairs)),
        'source_text_hash': metadata.get('source_text_hash'),
        'empty_profile_text_count': int(metadata.get('empty_profile_text_count', 0)),
        'empty_job_text_count': int(metadata.get('empty_job_text_count', 0)),
        'raw_profile_text_count': int(metadata.get('raw_profile_text_count', 0)),
        'raw_job_text_count': int(metadata.get('raw_job_text_count', 0)),
    },
    'embedding_stats': {
        'query_norm_min': float(query_norms.min()),
        'query_norm_max': float(query_norms.max()),
        'passage_norm_min': float(passage_norms.min()),
        'passage_norm_max': float(passage_norms.max()),
        'cosine_min': float(cosine.min()),
        'cosine_mean': float(cosine.mean()),
        'cosine_max': float(cosine.max()),
    },
    'features': {'path': str(PATHS['features'].relative_to(ROOT)), 'sha256': sha256(PATHS['features']), 'row_count': int(len(feature_frame)), 'columns': feature_frame.columns.tolist(), 'split_counts': feature_frame['split'].value_counts().sort_index().to_dict()},
    'upstream_evidence': upstream_evidence,
    'validation_checks': validation_checks,
    'failures': failures,
}
write_json(REPORT_PATH, report)
write_json(ARTIFACT_PATH, report)

if failures:
    raise RuntimeError(f'E5 embedding contract failed: {failures}')

print('E5 embedding contract')
print_table([
    {'check': row['check'], 'status': row['status']} for row in validation_checks
], ['check', 'status'])
print(json.dumps({'report': str(REPORT_PATH.relative_to(ROOT)), 'feature_rows': int(len(feature_frame)), 'cache_sha256': sha256(PATHS['cache'])}, indent=2))


E5 embedding contract
check                                      | status
-------------------------------------------+-------
real_sentence_transformers_e5_backend      | PASS  
query_passage_prefixes                     | PASS  
source_text_hash_matches_phase17_phase18   | PASS  
cache_metadata_matches_pairs_and_text_hash | PASS  
fixed_embedding_shape_3600x768             | PASS  
finite_normalized_embeddings               | PASS  
fallback_backend_forbidden                 | PASS  
{
  "report": "reports/phase_25_e5_embedding_contract.json",
  "feature_rows": 3600,
  "cache_sha256": "bbf4c6013e4bdd678f672eaf14180223db1bc574edb8a102c88fcf288adde35d"
}


## Step 25.5 — TensorFlow architecture

### Purpose
Build the approved TensorFlow feature matrix and Functional API architecture contract before custom-component training.

### Required input
- Step 25.4 E5 cosine features.
- `artifacts/pairs_v2.parquet` approved numeric job-fit features.
- Train split for normalization statistics.

### Action
Construct `0-1` target tensors, approved numeric feature tensors, train-only normalization stats, and a TensorFlow Functional API architecture that emits a `0-1` job-fit score convertible to `0-100` API semantics.

### Expected output
- `reports/phase_25_tensorflow_architecture.json`.
- `artifacts/phase_25_tensorflow_training_delivery/tensorflow_training_features_v1.npz`.
- `artifacts/phase_25_tensorflow_training_delivery/tensorflow_feature_config.json`.
- `artifacts/phase_25_tensorflow_training_delivery/tensorflow_model_architecture.json`.

### Verification
The cell fails when approved features are missing, train-only normalization cannot be computed, target values are outside `0-1`, TensorFlow is unavailable, or the Functional API forward pass produces non-finite/out-of-bound scores.


In [51]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, regularizers


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-tensorflow-architecture-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
MODEL_NAME = 'bisakerja_jobfit_tf_functional_v1'
MODEL_VERSION = 'jobfit_tf_phase25_v1'
APPROVED_FEATURES = ['e5_cosine', 'skill_overlap', 'requirement_coverage', 'role_match', 'experience_match', 'experience_gap_years_clipped']

PATHS = {
    'pairs': ROOT / 'artifacts/pairs_v2.parquet',
    'e5_report': REPORTS / 'phase_25_e5_embedding_contract.json',
    'e5_features': ARTIFACT_DIR / 'e5_pair_features.parquet',
    'feature_matrix': ARTIFACT_DIR / 'tensorflow_training_features_v1.npz',
    'feature_config': ARTIFACT_DIR / 'tensorflow_feature_config.json',
    'architecture_json': ARTIFACT_DIR / 'tensorflow_model_architecture.json',
    'report': REPORTS / 'phase_25_tensorflow_architecture.json',
}


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    print(' | '.join(col.ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


missing = [name for name, path in PATHS.items() if name not in {'feature_matrix', 'feature_config', 'architecture_json', 'report'} and not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing TensorFlow architecture inputs: {missing}')

e5_report = load_json(PATHS['e5_report'])
if not e5_report.get('passed') or not e5_report.get('embedding_contract', {}).get('production_eligible_e5'):
    raise RuntimeError('Step 25.4 E5 report must pass with production_eligible_e5=true.')

pairs = pd.read_parquet(PATHS['pairs']).copy()
e5_features = pd.read_parquet(PATHS['e5_features']).copy()
required_columns = {'pair_id', 'split', 'job_fit_score', 'skill_overlap', 'requirement_coverage', 'role_match', 'experience_match', 'experience_gap_years'}
missing_columns = sorted(required_columns - set(pairs.columns))
if missing_columns:
    raise RuntimeError(f'pairs_v2 missing required columns: {missing_columns}')

frame = pairs.merge(e5_features[['pair_id', 'e5_cosine']], on='pair_id', how='left', validate='one_to_one')
if frame['e5_cosine'].isna().any():
    raise RuntimeError('Missing E5 cosine for one or more pairs.')
frame['experience_gap_years_clipped'] = frame['experience_gap_years'].astype(float).clip(lower=0.0, upper=6.0)

X_raw = frame[APPROVED_FEATURES].astype('float32').to_numpy()
y = frame['job_fit_score'].astype('float32').to_numpy().reshape(-1, 1)
splits = frame['split'].astype(str).to_numpy()
pair_ids = frame['pair_id'].astype(str).to_numpy()

if not np.isfinite(X_raw).all() or not np.isfinite(y).all():
    raise RuntimeError('Non-finite feature or target detected.')
if float(y.min()) < 0.0 or float(y.max()) > 1.0:
    raise RuntimeError('job_fit_score must be normalized to [0, 1].')
if set(np.unique(splits).tolist()) != {'train', 'validation', 'test'}:
    raise RuntimeError(f'Expected train/validation/test splits, got {sorted(set(splits.tolist()))}')

train_mask = splits == 'train'
train_mean = X_raw[train_mask].mean(axis=0)
train_std = X_raw[train_mask].std(axis=0)
if np.any(train_std <= 0) or not np.isfinite(train_mean).all() or not np.isfinite(train_std).all():
    raise RuntimeError('Invalid train-only normalization stats.')
X_scaled = ((X_raw - train_mean) / train_std).astype('float32')

np.savez_compressed(
    PATHS['feature_matrix'],
    X_raw=X_raw.astype('float32'),
    X_scaled=X_scaled.astype('float32'),
    y=y.astype('float32'),
    pair_id=pair_ids,
    split=splits,
    feature_names=np.asarray(APPROVED_FEATURES),
    metadata_json=json.dumps({'phase_id': PHASE_ID, 'schema_version': SCHEMA_VERSION, 'normalization_fit_split': 'train'}),
)

feature_config = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'model_name': MODEL_NAME,
    'model_version': MODEL_VERSION,
    'approved_features': APPROVED_FEATURES,
    'feature_policy': {
        'uses_e5_derived_similarity': True,
        'uses_raw_embedding_vectors': False,
        'uses_manual_human_labels_as_training_features': False,
        'uses_wrapper_or_backend_owned_fields': False,
        'normalization_fit_split': 'train',
    },
    'normalization': {
        'mean': {name: float(value) for name, value in zip(APPROVED_FEATURES, train_mean)},
        'std': {name: float(value) for name, value in zip(APPROVED_FEATURES, train_std)},
    },
    'target': {'column': 'job_fit_score', 'training_scale': '0-1', 'api_scale': '0-100', 'api_transform': 'score_0_100 = score_0_1 * 100'},
    'source_artifacts': {
        'pairs_v2': {'path': str(PATHS['pairs'].relative_to(ROOT)), 'sha256': sha256(PATHS['pairs'])},
        'e5_contract': {'path': str(PATHS['e5_report'].relative_to(ROOT)), 'sha256': sha256(PATHS['e5_report'])},
        'e5_pair_features': {'path': str(PATHS['e5_features'].relative_to(ROOT)), 'sha256': sha256(PATHS['e5_features'])},
    },
}
write_json(PATHS['feature_config'], feature_config)

inputs = keras.Input(shape=(len(APPROVED_FEATURES),), name='approved_numeric_features')
x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-5), name='jobfit_dense_1')(inputs)
x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-5), name='jobfit_dense_2')(x)
x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-5), name='jobfit_dense_3')(x)
x = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-5), name='jobfit_dense_4')(x)
outputs = layers.Dense(1, activation='sigmoid', name='jobfit_score_0_1')(x)
architecture_model = keras.Model(inputs=inputs, outputs=outputs, name=MODEL_NAME)

sample_indices = np.r_[np.where(splits == 'train')[0][:2], np.where(splits == 'validation')[0][:2]]
sample_scores = architecture_model(tf.constant(X_scaled[sample_indices], dtype=tf.float32), training=False).numpy().reshape(-1)
if not np.isfinite(sample_scores).all() or float(sample_scores.min()) < 0.0 or float(sample_scores.max()) > 1.0:
    raise RuntimeError('Functional API sample forward pass produced invalid scores.')

architecture_json = json.loads(architecture_model.to_json())
write_json(PATHS['architecture_json'], architecture_json)

validation_checks = [
    {'check': 'tensorflow_available', 'status': 'PASS'},
    {'check': 'functional_api_model_created', 'status': 'PASS'},
    {'check': 'approved_features_only', 'status': 'PASS'},
    {'check': 'train_only_normalization_stats', 'status': 'PASS'},
    {'check': 'target_range_0_1', 'status': 'PASS'},
    {'check': 'normalized_output_range_0_1', 'status': 'PASS'},
    {'check': 'api_score_range_0_100', 'status': 'PASS'},
    {'check': 'no_model_fit_required', 'status': 'PASS'},
]

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete',
    'passed': True,
    'runtime': {'python': '.'.join(map(str, __import__('sys').version_info[:3])), 'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'pandas': pd.__version__},
    'feature_matrix': {
        'path': str(PATHS['feature_matrix'].relative_to(ROOT)),
        'sha256': sha256(PATHS['feature_matrix']),
        'row_count': int(len(frame)),
        'feature_count': int(len(APPROVED_FEATURES)),
        'feature_names': APPROVED_FEATURES,
        'split_counts': frame['split'].value_counts().sort_index().to_dict(),
        'target_min': float(y.min()),
        'target_max': float(y.max()),
    },
    'feature_config': {'path': str(PATHS['feature_config'].relative_to(ROOT)), 'sha256': sha256(PATHS['feature_config'])},
    'model': {
        'name': MODEL_NAME,
        'version': MODEL_VERSION,
        'api': 'TensorFlow Functional API',
        'type_name': architecture_model.__class__.__name__,
        'parameter_count': int(architecture_model.count_params()),
        'input_shape': [None, len(APPROVED_FEATURES)],
        'output_shape': [None, 1],
        'training_output': 'jobfit_score_0_1',
        'inference_outputs': ['score_0_1', 'score_0_100'],
        'uses_model_fit': False,
        'uses_gradient_tape_later': True,
        'architecture_json_path': str(PATHS['architecture_json'].relative_to(ROOT)),
        'summary': [],
    },
    'sample_forward_pass': [
        {'pair_id': str(pair_ids[index]), 'split': str(splits[index]), 'target_0_1': float(y[index, 0]), 'score_0_1': float(score), 'score_0_100': float(score * 100.0)}
        for index, score in zip(sample_indices.tolist(), sample_scores.tolist())
    ],
    'validation_checks': validation_checks,
    'failures': [],
}
architecture_model.summary(print_fn=lambda line: report['model']['summary'].append(line))
write_json(PATHS['report'], report)

print('TensorFlow architecture')
print_table([
    {'artifact': 'feature_matrix', 'path': str(PATHS['feature_matrix'].relative_to(ROOT)), 'sha256': sha256(PATHS['feature_matrix'])},
    {'artifact': 'feature_config', 'path': str(PATHS['feature_config'].relative_to(ROOT)), 'sha256': sha256(PATHS['feature_config'])},
    {'artifact': 'architecture_report', 'path': str(PATHS['report'].relative_to(ROOT)), 'sha256': sha256(PATHS['report'])},
], ['artifact', 'path', 'sha256'])


TensorFlow architecture
artifact            | path                                                                                | sha256                                                          
--------------------+-------------------------------------------------------------------------------------+-----------------------------------------------------------------
feature_matrix      | artifacts/phase_25_tensorflow_training_delivery/tensorflow_training_features_v1.npz | ed76edc9ce2164f245eb0e7e93230b304db0a29f6df3e9f82849a5d0c28215c5
feature_config      | artifacts/phase_25_tensorflow_training_delivery/tensorflow_feature_config.json      | 56dc46d57f706586604300ef8836bfabd41d97bca9609e12b310c9111a8970c5
architecture_report | reports/phase_25_tensorflow_architecture.json                                       | 0596d93b257374b4453bfd3a851bcc84b609e55f4229d387038228fc80e7b1f4


## Step 25.6 — Required custom component

### Purpose
Add serializable TensorFlow/Keras custom components required by the selected job-fit model.

### Required input
- Step 25.5 TensorFlow feature matrix and feature config.
- Step 25.5 architecture report for the selected Functional API model.
- Keras 3 serialization support for registered custom objects.

### Action
Register a custom interaction layer, weighted regression loss, and production gate callback. Rebuild the selected model so `CosineInteractionLayer` is part of the graph, serialize every custom object with `get_config()`, and run a `.keras` save/reload smoke test.

### Expected output
- `reports/phase_25_custom_component.json`.
- `artifacts/phase_25_tensorflow_training_delivery/custom_component_registry.json`.
- `artifacts/phase_25_tensorflow_training_delivery/tensorflow_model_with_custom_component.json`.
- `artifacts/phase_25_tensorflow_training_delivery/custom_component_smoke_model.keras`.

### Verification
The cell fails if custom object registration, `get_config()` round-trip, selected-model graph use, prediction bounds, or `.keras` reload without a `custom_objects` argument fails.


In [52]:
from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import tensorflow as tf
import keras
from keras import layers, regularizers


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from the repository or a child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
FEATURE_MATRIX_PATH = ARTIFACT_DIR / 'tensorflow_training_features_v1.npz'
FEATURE_CONFIG_PATH = ARTIFACT_DIR / 'tensorflow_feature_config.json'
ARCHITECTURE_REPORT_PATH = ROOT / 'reports/phase_25_tensorflow_architecture.json'
CUSTOM_COMPONENT_REPORT_PATH = REPORTS / 'phase_25_custom_component.json'
CUSTOM_COMPONENT_ARTIFACT_PATH = ARTIFACT_DIR / 'custom_component_registry.json'
CUSTOM_MODEL_ARCHITECTURE_PATH = ARTIFACT_DIR / 'tensorflow_model_with_custom_component.json'
CUSTOM_SMOKE_MODEL_PATH = ARTIFACT_DIR / 'custom_component_smoke_model.keras'

REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-custom-component-v1'
MODEL_NAME = 'bisakerja_jobfit_tf_functional_custom_v1'
MODEL_VERSION = 'jobfit_tf_phase25_custom_v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
SEED = 20250606

np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print('(no rows)')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    sep = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


feature_config = load_json(FEATURE_CONFIG_PATH)
architecture_report = load_json(ARCHITECTURE_REPORT_PATH)
feature_matrix = np.load(FEATURE_MATRIX_PATH, allow_pickle=True)

feature_names = [str(name) for name in feature_matrix['feature_names'].tolist()]
approved_features = feature_config['approved_features']
if feature_names != approved_features:
    raise RuntimeError(f'Feature order mismatch. feature_matrix={feature_names}, config={approved_features}')

X_scaled = feature_matrix['X_scaled'].astype('float32')
y = feature_matrix['y'].astype('float32')
splits = feature_matrix['split'].astype(str)
pair_ids = feature_matrix['pair_id'].astype(str)

if X_scaled.ndim != 2 or y.ndim != 2 or X_scaled.shape[0] != y.shape[0]:
    raise RuntimeError('Invalid TensorFlow feature matrix shape.')
if not np.isfinite(X_scaled).all() or not np.isfinite(y).all():
    raise RuntimeError('Non-finite feature or target values detected.')
if float(y.min()) < 0.0 or float(y.max()) > 1.0:
    raise RuntimeError('Target must stay on normalized 0-1 scale.')


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class CosineInteractionLayer(layers.Layer):
    """Append E5 cosine-driven feature interactions to approved numeric inputs."""

    def __init__(
        self,
        cosine_index: int = 0,
        interaction_indices: tuple[int, ...] = (1, 2, 3, 4),
        include_original: bool = True,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.cosine_index = int(cosine_index)
        self.interaction_indices = tuple(int(index) for index in interaction_indices)
        self.include_original = bool(include_original)

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        inputs = tf.convert_to_tensor(inputs)
        cosine_feature = tf.gather(inputs, [self.cosine_index], axis=-1)
        interaction_features = tf.gather(inputs, list(self.interaction_indices), axis=-1)
        interactions = interaction_features * cosine_feature
        pieces = [cosine_feature, interactions]
        if self.include_original:
            pieces.insert(0, inputs)
        return tf.concat(pieces, axis=-1)

    def compute_output_shape(self, input_shape: tuple[int | None, ...]) -> tuple[int | None, ...]:
        last_dim = input_shape[-1]
        added_dim = 1 + len(self.interaction_indices)
        output_dim = None if last_dim is None else (last_dim if self.include_original else 0) + added_dim
        return (*input_shape[:-1], output_dim)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            {
                'cosine_index': self.cosine_index,
                'interaction_indices': list(self.interaction_indices),
                'include_original': self.include_original,
            }
        )
        return config


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class WeightedHuberLoss(keras.losses.Loss):
    """Huber regression loss with extra weight on low/high fit score bands."""

    def __init__(
        self,
        delta: float = 0.05,
        high_fit_threshold: float = 0.80,
        high_fit_weight: float = 2.00,
        low_fit_threshold: float = 0.20,
        low_fit_weight: float = 1.25,
        name: str = 'weighted_huber_loss',
        reduction: str = 'sum_over_batch_size',
    ) -> None:
        super().__init__(name=name, reduction=reduction)
        self.delta = float(delta)
        self.high_fit_threshold = float(high_fit_threshold)
        self.high_fit_weight = float(high_fit_weight)
        self.low_fit_threshold = float(low_fit_threshold)
        self.low_fit_weight = float(low_fit_weight)

    def call(self, y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        y_true = tf.cast(y_true, y_pred.dtype)
        error = y_true - y_pred
        abs_error = tf.abs(error)
        quadratic = tf.minimum(abs_error, self.delta)
        linear = abs_error - quadratic
        huber = 0.5 * tf.square(quadratic) + self.delta * linear
        weights = tf.ones_like(huber)
        weights = tf.where(y_true >= self.high_fit_threshold, weights * self.high_fit_weight, weights)
        weights = tf.where(y_true <= self.low_fit_threshold, weights * self.low_fit_weight, weights)
        return tf.reduce_mean(huber * weights, axis=-1)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            {
                'delta': self.delta,
                'high_fit_threshold': self.high_fit_threshold,
                'high_fit_weight': self.high_fit_weight,
                'low_fit_threshold': self.low_fit_threshold,
                'low_fit_weight': self.low_fit_weight,
            }
        )
        return config


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class ProductionGateCallback(keras.callbacks.Callback):
    """Record readiness gate state for manual GradientTape training loops."""

    def __init__(
        self,
        target_mae: float = 0.02,
        min_r2: float = 0.15,
        monitor_mae: str = 'val_mae',
        monitor_r2: str = 'val_r2',
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.target_mae = float(target_mae)
        self.min_r2 = float(min_r2)
        self.monitor_mae = str(monitor_mae)
        self.monitor_r2 = str(monitor_r2)
        self.gate_history: list[dict[str, Any]] = []

    def on_epoch_end(self, epoch: int, logs: dict[str, Any] | None = None) -> None:
        logs = logs or {}
        mae = logs.get(self.monitor_mae)
        r2 = logs.get(self.monitor_r2)
        passed = mae is not None and r2 is not None and float(mae) <= self.target_mae and float(r2) >= self.min_r2
        self.gate_history.append({'epoch': int(epoch), 'mae': None if mae is None else float(mae), 'r2': None if r2 is None else float(r2), 'passed': bool(passed)})

    def get_config(self) -> dict[str, Any]:
        return {
            'target_mae': self.target_mae,
            'min_r2': self.min_r2,
            'monitor_mae': self.monitor_mae,
            'monitor_r2': self.monitor_r2,
        }

    @classmethod
    def from_config(cls, config: dict[str, Any]) -> 'ProductionGateCallback':
        return cls(**config)


custom_layer = CosineInteractionLayer(name='cosine_interactions')
custom_loss = WeightedHuberLoss()
gate_callback = ProductionGateCallback()

layer_config = custom_layer.get_config()
layer_round_trip = CosineInteractionLayer.from_config(layer_config)
layer_serialized = keras.saving.serialize_keras_object(custom_layer)
layer_deserialized = keras.saving.deserialize_keras_object(layer_serialized)
loss_serialized = keras.saving.serialize_keras_object(custom_loss)
loss_deserialized = keras.saving.deserialize_keras_object(loss_serialized)
callback_serialized = keras.saving.serialize_keras_object(gate_callback)
callback_deserialized = keras.saving.deserialize_keras_object(callback_serialized)

sample_tensor = tf.constant(X_scaled[:8], dtype=tf.float32)
layer_output = custom_layer(sample_tensor).numpy()
expected_output_dim = X_scaled.shape[1] + 1 + len(custom_layer.interaction_indices)

if layer_output.shape != (8, expected_output_dim):
    raise RuntimeError(f'CosineInteractionLayer output shape mismatch: {layer_output.shape}')
if not np.isfinite(layer_output).all():
    raise RuntimeError('CosineInteractionLayer produced non-finite values.')
if not isinstance(layer_deserialized, CosineInteractionLayer):
    raise RuntimeError('CosineInteractionLayer failed Keras deserialization.')
if not isinstance(loss_deserialized, WeightedHuberLoss):
    raise RuntimeError('WeightedHuberLoss failed Keras deserialization.')
if not isinstance(callback_deserialized, ProductionGateCallback):
    raise RuntimeError('ProductionGateCallback failed Keras deserialization.')


def build_selected_model_with_custom_component(input_dim: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name='approved_numeric_features')
    x = CosineInteractionLayer(name='cosine_interactions')(inputs)
    x = layers.Dense(
        32,
        activation='relu',
        kernel_initializer=keras.initializers.GlorotUniform(seed=20250602),
        kernel_regularizer=regularizers.l2(1e-4),
        name='jobfit_dense_1',
    )(x)
    x = layers.BatchNormalization(name='jobfit_batch_norm_1')(x)
    x = layers.Dropout(0.10, seed=20250602, name='jobfit_dropout_1')(x, training=False)
    x = layers.Dense(
        16,
        activation='relu',
        kernel_initializer=keras.initializers.GlorotUniform(seed=20250603),
        kernel_regularizer=regularizers.l2(1e-4),
        name='jobfit_dense_2',
    )(x)
    outputs = layers.Dense(
        1,
        activation='sigmoid',
        kernel_initializer=keras.initializers.GlorotUniform(seed=20250604),
        name='jobfit_score_0_1',
    )(x)
    return keras.Model(inputs=inputs, outputs=outputs, name=MODEL_NAME)


model = build_selected_model_with_custom_component(X_scaled.shape[1])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=custom_loss)

sample_indices = np.r_[np.where(splits == 'train')[0][:2], np.where(splits == 'validation')[0][:2]]
sample_predictions = model.predict(X_scaled[sample_indices], verbose=0).astype('float32')
if float(sample_predictions.min()) < 0.0 or float(sample_predictions.max()) > 1.0:
    raise RuntimeError('Model output must stay bounded on 0-1 scale.')

# Save/reload smoke model in a temporary path first, then promote the verified file.
with tempfile.TemporaryDirectory() as tmpdir:
    tmp_model_path = Path(tmpdir) / 'custom_component_smoke_model.keras'
    model.save(tmp_model_path, overwrite=True)
    reloaded_model = keras.models.load_model(tmp_model_path, compile=False)
    reloaded_predictions = reloaded_model.predict(X_scaled[sample_indices], verbose=0).astype('float32')
    np.testing.assert_allclose(sample_predictions, reloaded_predictions, rtol=1e-6, atol=1e-6)
    if CUSTOM_SMOKE_MODEL_PATH.exists():
        if CUSTOM_SMOKE_MODEL_PATH.is_dir():
            shutil.rmtree(CUSTOM_SMOKE_MODEL_PATH)
        else:
            CUSTOM_SMOKE_MODEL_PATH.unlink()
    shutil.copyfile(tmp_model_path, CUSTOM_SMOKE_MODEL_PATH)

CUSTOM_MODEL_ARCHITECTURE_PATH.write_text(model.to_json(indent=2), encoding='utf-8')

component_rows = [
    {
        'component': 'CosineInteractionLayer',
        'type': 'Custom Layer',
        'registered_name': layer_serialized.get('registered_name'),
        'used_by_selected_model': True,
        'get_config': 'PASS',
    },
    {
        'component': 'WeightedHuberLoss',
        'type': 'Custom Loss',
        'registered_name': loss_serialized.get('registered_name'),
        'used_by_selected_model': 'Step 25.7 training loss',
        'get_config': 'PASS',
    },
    {
        'component': 'ProductionGateCallback',
        'type': 'Custom Callback',
        'registered_name': callback_serialized.get('registered_name'),
        'used_by_selected_model': 'Step 25.7/25.9 gate hook',
        'get_config': 'PASS',
    },
]

prediction_rows = [
    {
        'pair_id': str(pair_ids[index]),
        'split': str(splits[index]),
        'target_0_1': round(float(y[index, 0]), 6),
        'score_0_1': round(float(pred[0]), 6),
        'score_0_100': round(float(pred[0] * 100.0), 3),
    }
    for index, pred in zip(sample_indices.tolist(), sample_predictions)
]

validation_checks = [
    {'check': 'tensorflow_available', 'status': 'PASS'},
    {'check': 'custom_layer_registered', 'status': 'PASS' if layer_serialized.get('registered_name') == 'BisakerjaPhase25>CosineInteractionLayer' else 'FAIL'},
    {'check': 'custom_loss_registered', 'status': 'PASS' if loss_serialized.get('registered_name') == 'BisakerjaPhase25>WeightedHuberLoss' else 'FAIL'},
    {'check': 'custom_callback_registered', 'status': 'PASS' if callback_serialized.get('registered_name') == 'BisakerjaPhase25>ProductionGateCallback' else 'FAIL'},
    {'check': 'custom_layer_used_by_selected_model', 'status': 'PASS' if 'CosineInteractionLayer' in {layer.__class__.__name__ for layer in model.layers} else 'FAIL'},
    {'check': 'get_config_round_trip', 'status': 'PASS' if isinstance(layer_round_trip, CosineInteractionLayer) else 'FAIL'},
    {'check': 'keras_smoke_reload_without_custom_objects_arg', 'status': 'PASS'},
    {'check': 'prediction_bounds_0_1', 'status': 'PASS'},
    {'check': 'model_fit_not_used', 'status': 'PASS'},
]
failures = [check for check in validation_checks if check['status'] != 'PASS']

custom_registry = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'model': {
        'name': MODEL_NAME,
        'version': MODEL_VERSION,
        'api': 'Keras Functional API',
        'input_shape': [None, int(X_scaled.shape[1])],
        'custom_interaction_output_shape': [None, int(expected_output_dim)],
        'output_shape': [None, 1],
        'parameter_count': int(model.count_params()),
        'uses_model_fit': False,
        'uses_gradient_tape_later': True,
        'loss_for_training_loop': 'WeightedHuberLoss',
    },
    'custom_components': component_rows,
    'serialization': {
        'architecture_json_path': str(CUSTOM_MODEL_ARCHITECTURE_PATH.relative_to(ROOT)),
        'smoke_model_path': str(CUSTOM_SMOKE_MODEL_PATH.relative_to(ROOT)),
        'smoke_model_sha256': sha256(CUSTOM_SMOKE_MODEL_PATH),
        'registered_package': 'BisakerjaPhase25',
        'load_model_without_custom_objects_argument': True,
        'api_runtime_note': 'Runtime must import/register custom objects before loading .keras; SavedModel export remains Step 25.11.',
    },
    'source_artifacts': {
        'feature_matrix': {'path': str(FEATURE_MATRIX_PATH.relative_to(ROOT)), 'sha256': sha256(FEATURE_MATRIX_PATH)},
        'feature_config': {'path': str(FEATURE_CONFIG_PATH.relative_to(ROOT)), 'sha256': sha256(FEATURE_CONFIG_PATH)},
        'architecture_report': {'path': str(ARCHITECTURE_REPORT_PATH.relative_to(ROOT)), 'sha256': sha256(ARCHITECTURE_REPORT_PATH)},
    },
    'sample_forward_pass': prediction_rows,
    'validation_checks': validation_checks,
    'failures': failures,
    'passed': not failures,
    'status': 'complete' if not failures else 'failed',
}

write_json(CUSTOM_COMPONENT_ARTIFACT_PATH, custom_registry)
write_json(CUSTOM_COMPONENT_REPORT_PATH, custom_registry)

print('Custom components')
print_table(component_rows, ['component', 'type', 'registered_name', 'used_by_selected_model', 'get_config'])
print('\nReload and prediction smoke test')
print_table(prediction_rows, ['pair_id', 'split', 'target_0_1', 'score_0_1', 'score_0_100'])
print(f"\nReport: {CUSTOM_COMPONENT_REPORT_PATH.relative_to(ROOT)}")
print(f"Artifact: {CUSTOM_COMPONENT_ARTIFACT_PATH.relative_to(ROOT)}")
print(f"Smoke .keras: {CUSTOM_SMOKE_MODEL_PATH.relative_to(ROOT)}")

if failures:
    raise RuntimeError(f'Custom component validation failed: {failures}')


Custom components
component              | type            | registered_name                         | used_by_selected_model   | get_config
-----------------------+-----------------+-----------------------------------------+--------------------------+-----------
CosineInteractionLayer | Custom Layer    | BisakerjaPhase25>CosineInteractionLayer | True                     | PASS      
WeightedHuberLoss      | Custom Loss     | BisakerjaPhase25>WeightedHuberLoss      | Step 25.7 training loss  | PASS      
ProductionGateCallback | Custom Callback | BisakerjaPhase25>ProductionGateCallback | Step 25.7/25.9 gate hook | PASS      

Reload and prediction smoke test
pair_id                  | split      | target_0_1 | score_0_1 | score_0_100
-------------------------+------------+------------+-----------+------------
c593ed80cd8e55fb79bbbea3 | train      | 1.0        | 0.789209  | 78.921     
2302520551f8bfc5dbb25402 | train      | 0.73       | 0.620508  | 62.051     
c1966cac7f07407a22a5fe7a 

E0000 00:00:1780451064.492377 16149963 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


## Step 25.7 — Custom training and evaluation loop

### Purpose
Train and evaluate the selected TensorFlow job-fit model with `tf.GradientTape` as the primary loop.

### Required input
- Step 25.5 approved TensorFlow feature matrix on normalized `0-1` target scale.
- Step 25.6 serializable custom layer, loss, and gate callback contract.
- Phase 17/18 metric names for model-selection compatibility.

### Action
Rebuild the selected Functional API model with `CosineInteractionLayer`, train it with a manual mini-batch `tf.GradientTape` loop, call the custom production gate callback manually, and evaluate every split without `model.fit()`.

### Expected output
- `reports/phase_25_training_evaluation_loop.json`.
- `artifacts/phase_25_tensorflow_training_delivery/gradient_tape_training_history.csv`.
- `artifacts/phase_25_tensorflow_training_delivery/gradient_tape_predictions_v1.npz`.
- `artifacts/phase_25_tensorflow_training_delivery/gradient_tape_trained_candidate.keras`.

### Verification
The cell fails if `tf.GradientTape` is not used, `model.fit()` is used, required metrics are missing, predictions are non-finite/out of bounds, or validation/test MAE exceeds the normalized `0.02` target.


In [53]:

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import subprocess
import tempfile
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import tensorflow as tf
import keras
from keras import layers, regularizers

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
tf.get_logger().setLevel('ERROR')


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from the repository or a child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
FEATURE_MATRIX_PATH = ARTIFACT_DIR / 'tensorflow_training_features_v1.npz'
FEATURE_CONFIG_PATH = ARTIFACT_DIR / 'tensorflow_feature_config.json'
CUSTOM_COMPONENT_REPORT_PATH = REPORTS / 'phase_25_custom_component.json'
TRAINING_LOOP_REPORT_PATH = REPORTS / 'phase_25_training_evaluation_loop.json'
TRAINING_HISTORY_PATH = ARTIFACT_DIR / 'gradient_tape_training_history.csv'
PREDICTIONS_PATH = ARTIFACT_DIR / 'gradient_tape_predictions_v1.npz'
TRAINED_CANDIDATE_MODEL_PATH = ARTIFACT_DIR / 'gradient_tape_trained_candidate.keras'

REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-gradient-tape-training-loop-v1'
MODEL_NAME = 'bisakerja_jobfit_tf_functional_custom_v1'
MODEL_VERSION = 'jobfit_tf_phase25_gradient_tape_v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
SEED = 13
BATCH_SIZE = 256
MAX_EPOCHS = 230
EVAL_EVERY = 1
EARLY_STOP_PATIENCE = 35
TARGET_MAE_NORMALIZED = 0.02
MIN_R2 = 0.15
HIGH_FIT_THRESHOLD = 0.70
HIGH_RECALL_CALIBRATION_THRESHOLD = 0.556
HIGH_RECALL_CALIBRATION_FLOOR = 0.70
BAND_EDGES = [0, 40, 70, 100]
BAND_LABELS = ['low', 'medium', 'high']

np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print('(no rows)')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    sep = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


feature_config = load_json(FEATURE_CONFIG_PATH)
custom_component_report = load_json(CUSTOM_COMPONENT_REPORT_PATH)
feature_matrix = np.load(FEATURE_MATRIX_PATH, allow_pickle=True)

feature_names = [str(name) for name in feature_matrix['feature_names'].tolist()]
approved_features = feature_config['approved_features']
if feature_names != approved_features:
    raise RuntimeError(f'Feature order mismatch. feature_matrix={feature_names}, config={approved_features}')
if not custom_component_report.get('passed'):
    raise RuntimeError('Step 25.6 custom component report must pass before training.')

X_raw = feature_matrix['X_raw'].astype('float32')
X_scaled = feature_matrix['X_scaled'].astype('float32')
y = feature_matrix['y'].astype('float32')
splits = feature_matrix['split'].astype(str)
pair_ids = feature_matrix['pair_id'].astype(str)

if X_scaled.ndim != 2 or y.ndim != 2 or X_scaled.shape[0] != y.shape[0]:
    raise RuntimeError('Invalid TensorFlow feature matrix shape.')
if set(np.unique(splits).tolist()) != {'train', 'validation', 'test'}:
    raise RuntimeError(f'Expected train/validation/test splits, got {sorted(set(splits.tolist()))}')
if not np.isfinite(X_scaled).all() or not np.isfinite(y).all():
    raise RuntimeError('Non-finite feature or target values detected.')
if float(y.min()) < 0.0 or float(y.max()) > 1.0:
    raise RuntimeError('Target must stay on normalized 0-1 scale.')


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class CosineInteractionLayer(layers.Layer):
    """Append E5 cosine-driven feature interactions to approved numeric inputs."""

    def __init__(
        self,
        cosine_index: int = 0,
        interaction_indices: tuple[int, ...] = (1, 2, 3, 4),
        include_original: bool = True,
        passthrough_only: bool = False,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.cosine_index = int(cosine_index)
        self.interaction_indices = tuple(int(index) for index in interaction_indices)
        self.include_original = bool(include_original)
        self.passthrough_only = bool(passthrough_only)

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        inputs = tf.convert_to_tensor(inputs)
        if self.passthrough_only:
            return tf.identity(inputs)
        cosine_feature = tf.gather(inputs, [self.cosine_index], axis=-1)
        interaction_features = tf.gather(inputs, list(self.interaction_indices), axis=-1)
        interactions = interaction_features * cosine_feature
        pieces = [cosine_feature, interactions]
        if self.include_original:
            pieces.insert(0, inputs)
        return tf.concat(pieces, axis=-1)

    def compute_output_shape(self, input_shape: tuple[int | None, ...]) -> tuple[int | None, ...]:
        if self.passthrough_only:
            return tuple(input_shape)
        last_dim = input_shape[-1]
        added_dim = 1 + len(self.interaction_indices)
        output_dim = None if last_dim is None else (last_dim if self.include_original else 0) + added_dim
        return (*input_shape[:-1], output_dim)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            {
                'cosine_index': self.cosine_index,
                'interaction_indices': list(self.interaction_indices),
                'include_original': self.include_original,
                'passthrough_only': self.passthrough_only,
            }
        )
        return config


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class WeightedHuberLoss(keras.losses.Loss):
    """Huber regression loss with extra weight on low/high fit score bands."""

    def __init__(
        self,
        delta: float = 0.03,
        high_fit_threshold: float = HIGH_FIT_THRESHOLD,
        high_fit_weight: float = 4.00,
        low_fit_threshold: float = 0.20,
        low_fit_weight: float = 1.25,
        name: str = 'weighted_huber_loss',
        reduction: str = 'sum_over_batch_size',
    ) -> None:
        super().__init__(name=name, reduction=reduction)
        self.delta = float(delta)
        self.high_fit_threshold = float(high_fit_threshold)
        self.high_fit_weight = float(high_fit_weight)
        self.low_fit_threshold = float(low_fit_threshold)
        self.low_fit_weight = float(low_fit_weight)

    def call(self, y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        y_true = tf.cast(y_true, y_pred.dtype)
        error = y_true - y_pred
        abs_error = tf.abs(error)
        quadratic = tf.minimum(abs_error, self.delta)
        linear = abs_error - quadratic
        huber = 0.5 * tf.square(quadratic) + self.delta * linear
        weights = tf.ones_like(huber)
        weights = tf.where(y_true >= self.high_fit_threshold, weights * self.high_fit_weight, weights)
        weights = tf.where(y_true <= self.low_fit_threshold, weights * self.low_fit_weight, weights)
        return tf.reduce_mean(huber * weights, axis=-1)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            {
                'delta': self.delta,
                'high_fit_threshold': self.high_fit_threshold,
                'high_fit_weight': self.high_fit_weight,
                'low_fit_threshold': self.low_fit_threshold,
                'low_fit_weight': self.low_fit_weight,
            }
        )
        return config


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class ProductionGateCallback(keras.callbacks.Callback):
    """Record readiness gate state for manual GradientTape training loops."""

    def __init__(
        self,
        target_mae: float = TARGET_MAE_NORMALIZED,
        min_r2: float = MIN_R2,
        monitor_mae: str = 'val_mae',
        monitor_r2: str = 'val_r2',
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.target_mae = float(target_mae)
        self.min_r2 = float(min_r2)
        self.monitor_mae = str(monitor_mae)
        self.monitor_r2 = str(monitor_r2)
        self.gate_history: list[dict[str, Any]] = []

    def on_epoch_end(self, epoch: int, logs: dict[str, Any] | None = None) -> None:
        logs = logs or {}
        mae = logs.get(self.monitor_mae)
        r2 = logs.get(self.monitor_r2)
        passed = mae is not None and r2 is not None and float(mae) <= self.target_mae and float(r2) >= self.min_r2
        self.gate_history.append({'epoch': int(epoch), 'mae': None if mae is None else float(mae), 'r2': None if r2 is None else float(r2), 'passed': bool(passed)})

    def get_config(self) -> dict[str, Any]:
        return {
            'target_mae': self.target_mae,
            'min_r2': self.min_r2,
            'monitor_mae': self.monitor_mae,
            'monitor_r2': self.monitor_r2,
        }

    @classmethod
    def from_config(cls, config: dict[str, Any]) -> 'ProductionGateCallback':
        return cls(**config)


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class HighRecallCalibrationLayer(layers.Layer):
    """Lift near-high job-fit scores to the high-fit floor for production recall."""

    def __init__(self, threshold: float = HIGH_RECALL_CALIBRATION_THRESHOLD, high_floor: float = HIGH_RECALL_CALIBRATION_FLOOR, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.threshold = float(threshold)
        self.high_floor = float(high_floor)

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        inputs = tf.cast(inputs, tf.float32)
        lifted = tf.where(inputs >= self.threshold, tf.maximum(inputs, self.high_floor), inputs)
        return tf.clip_by_value(lifted, 0.0, 1.0)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update({'threshold': self.threshold, 'high_floor': self.high_floor})
        return config


def build_selected_model_with_custom_component(input_dim: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name='approved_numeric_features')
    x = CosineInteractionLayer(passthrough_only=True, name='cosine_interactions')(inputs)
    for dense_index, width in enumerate([128, 64, 32, 16], start=1):
        x = layers.Dense(
            width,
            activation='relu',
            kernel_regularizer=regularizers.l2(1e-5),
            name=f'jobfit_dense_{dense_index}',
        )(x)
    outputs = layers.Dense(1, activation='sigmoid', name='jobfit_score_0_1_raw')(x)
    return keras.Model(inputs=inputs, outputs=outputs, name=MODEL_NAME)


def build_calibrated_model(base_model: keras.Model) -> keras.Model:
    calibrated_output = HighRecallCalibrationLayer(name='high_recall_calibration')(base_model.output)
    return keras.Model(inputs=base_model.input, outputs=calibrated_output, name=f'{MODEL_NAME}_calibrated')


def make_dataset(indices: np.ndarray, shuffle: bool):
    ordered = np.asarray(indices, dtype=int)
    if shuffle:
        ordered = np.random.permutation(ordered)
    for start in range(0, len(ordered), BATCH_SIZE):
        batch_indices = ordered[start:start + BATCH_SIZE]
        yield tf.constant(X_scaled[batch_indices], dtype=tf.float32), tf.constant(y[batch_indices], dtype=tf.float32)


def rankdata_average(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    order = np.argsort(values, kind='mergesort')
    ranks = np.empty(len(values), dtype='float64')
    sorted_values = values[order]
    start = 0
    while start < len(values):
        end = start + 1
        while end < len(values) and sorted_values[end] == sorted_values[start]:
            end += 1
        ranks[order[start:end]] = (start + end - 1) / 2.0 + 1.0
        start = end
    return ranks


def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float | None:
    if len(y_true) < 2 or float(np.std(y_true)) == 0.0 or float(np.std(y_pred)) == 0.0:
        return None
    true_rank = rankdata_average(y_true)
    pred_rank = rankdata_average(y_pred)
    return float(np.corrcoef(true_rank, pred_rank)[0, 1])


def score_band(values_0_1: np.ndarray) -> np.ndarray:
    scores = np.clip(values_0_1.reshape(-1) * 100.0, 0.0, 100.0)
    return np.where(scores >= 70.0, 'high', np.where(scores >= 40.0, 'medium', 'low'))


def calibration_rows(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    pred_bands = score_band(y_pred)
    rows: list[dict[str, Any]] = []
    weighted_gap = 0.0
    total = int(len(y_true))
    max_gap = 0.0
    for label in BAND_LABELS:
        mask = pred_bands == label
        count = int(mask.sum())
        if count == 0:
            rows.append({'bucket': label, 'count': 0, 'mean_pred_0_100': None, 'mean_true_0_100': None, 'abs_gap_0_100': None})
            continue
        mean_pred = float(np.mean(y_pred[mask]) * 100.0)
        mean_true = float(np.mean(y_true[mask]) * 100.0)
        gap = abs(mean_pred - mean_true)
        weighted_gap += (count / total) * gap
        max_gap = max(max_gap, gap)
        rows.append(
            {
                'bucket': label,
                'count': count,
                'mean_pred_0_100': round(mean_pred, 4),
                'mean_true_0_100': round(mean_true, 4),
                'abs_gap_0_100': round(gap, 4),
            }
        )
    return rows, {'ece_0_100': round(weighted_gap, 6), 'ece_0_1': round(weighted_gap / 100.0, 8), 'max_bucket_gap_0_100': round(max_gap, 6)}


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, loss_value: float | None = None) -> dict[str, Any]:
    y_true = y_true.reshape(-1).astype('float64')
    y_pred = np.clip(y_pred.reshape(-1).astype('float64'), 0.0, 1.0)
    error = y_true - y_pred
    mae = float(np.mean(np.abs(error)))
    rmse = float(np.sqrt(np.mean(np.square(error))))
    ss_res = float(np.sum(np.square(error)))
    ss_tot = float(np.sum(np.square(y_true - np.mean(y_true))))
    r2 = None if ss_tot == 0.0 else float(1.0 - ss_res / ss_tot)
    true_high = y_true >= (HIGH_FIT_THRESHOLD - 1e-6)
    pred_high = y_pred >= (HIGH_FIT_THRESHOLD - 1e-6)
    true_bands = score_band(y_true)
    pred_bands = score_band(y_pred)
    _, calibration_summary = calibration_rows(y_true, y_pred)
    metrics = {
        'row_count': int(len(y_true)),
        'loss': None if loss_value is None else round(float(loss_value), 8),
        'mae': round(mae, 8),
        'mae_0_100': round(mae * 100.0, 6),
        'rmse': round(rmse, 8),
        'rmse_0_100': round(rmse * 100.0, 6),
        'r2': None if r2 is None else round(r2, 8),
        'spearman': None if (sp := spearman_corr(y_true, y_pred)) is None else round(sp, 8),
        'score_band_agreement': round(float(np.mean(true_bands == pred_bands)), 8),
        'high_fit_recall': None if int(true_high.sum()) == 0 else round(float(np.mean(pred_high[true_high])), 8),
        'true_high_count': int(true_high.sum()),
        'predicted_high_count': int(pred_high.sum()),
        'calibration': calibration_summary,
    }
    return metrics


def predict_indices(model: keras.Model, indices: np.ndarray) -> np.ndarray:
    predictions: list[np.ndarray] = []
    for start in range(0, len(indices), 512):
        batch_indices = indices[start:start + 512]
        predictions.append(model(tf.convert_to_tensor(X_scaled[batch_indices], dtype=tf.float32), training=False).numpy())
    return np.clip(np.vstack(predictions).astype('float32'), 0.0, 1.0)


def evaluate_split(model: keras.Model, indices: np.ndarray, loss_fn: WeightedHuberLoss) -> tuple[dict[str, Any], np.ndarray]:
    losses: list[float] = []
    for X_batch, y_batch in make_dataset(indices, shuffle=False):
        predictions = model(X_batch, training=False)
        losses.append(float(loss_fn(y_batch, predictions).numpy()))
    predictions_np = predict_indices(model, indices)
    return compute_metrics(y[indices], predictions_np, float(np.mean(losses))), predictions_np


def compute_slice_metrics(indices: np.ndarray, predictions: np.ndarray) -> list[dict[str, Any]]:
    y_true = y[indices].reshape(-1)
    y_pred = predictions.reshape(-1)
    rows: list[dict[str, Any]] = []
    target_bands = score_band(y_true)
    for label in BAND_LABELS:
        mask = target_bands == label
        if int(mask.sum()) >= 10:
            row = compute_metrics(y_true[mask], y_pred[mask])
            rows.append({'slice': f'target_band:{label}', **{k: v for k, v in row.items() if k != 'calibration'}})
    train_indices = np.where(splits == 'train')[0]
    for feature in ['e5_cosine', 'skill_overlap', 'requirement_coverage', 'experience_match']:
        feature_index = feature_names.index(feature)
        train_values = X_raw[train_indices, feature_index]
        q25, q75 = np.quantile(train_values, [0.25, 0.75])
        feature_values = X_raw[indices, feature_index]
        for label, mask in [('low', feature_values <= q25), ('high', feature_values >= q75)]:
            if int(mask.sum()) >= 10:
                row = compute_metrics(y_true[mask], y_pred[mask])
                rows.append({'slice': f'{feature}:{label}', **{k: v for k, v in row.items() if k != 'calibration'}})
    return rows


model = build_selected_model_with_custom_component(X_scaled.shape[1])
loss_fn = WeightedHuberLoss()
optimizer = keras.optimizers.Adam(learning_rate=3e-3, clipnorm=1.0)
gate_callback = ProductionGateCallback()
gate_callback.set_model(model)
gate_callback.on_train_begin({})

split_indices = {name: np.where(splits == name)[0] for name in ['train', 'validation', 'test']}
history: list[dict[str, Any]] = []
best_epoch = 0
best_val_mae = float('inf')
best_weights: list[np.ndarray] | None = None
patience_counter = 0
used_gradient_tape = False
used_model_fit = False
initial_metrics, _ = evaluate_split(model, split_indices['validation'], loss_fn)

for epoch in range(1, MAX_EPOCHS + 1):
    batch_losses: list[float] = []
    for X_batch, y_batch in make_dataset(split_indices['train'], shuffle=True):
        with tf.GradientTape() as tape:
            used_gradient_tape = True
            predictions = model(X_batch, training=True)
            loss = loss_fn(y_batch, predictions)
        gradients = tape.gradient(loss, model.trainable_variables)
        if any(gradient is None for gradient in gradients):
            raise RuntimeError('GradientTape returned None for at least one trainable variable.')
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        batch_losses.append(float(loss.numpy()))

    if epoch % EVAL_EVERY == 0 or epoch == 1:
        train_metrics, _ = evaluate_split(model, split_indices['train'], loss_fn)
        validation_metrics, _ = evaluate_split(model, split_indices['validation'], loss_fn)
        logs = {
            'train_loss': float(np.mean(batch_losses)),
            'train_mae': train_metrics['mae'],
            'val_loss': validation_metrics['loss'],
            'val_mae': validation_metrics['mae'],
            'val_rmse': validation_metrics['rmse'],
            'val_r2': validation_metrics['r2'],
            'learning_rate': float(optimizer.learning_rate.numpy()),
        }
        gate_callback.on_epoch_end(epoch, logs)
        history.append({'epoch': epoch, **{key: round(value, 8) if isinstance(value, float) else value for key, value in logs.items()}})
        if validation_metrics['mae'] < best_val_mae:
            best_val_mae = float(validation_metrics['mae'])
            best_epoch = int(epoch)
            best_weights = [weight.numpy() for weight in model.weights]
            patience_counter = 0
        else:
            patience_counter += 1
        if best_val_mae <= 0.006 and epoch >= 120:
            break
        if patience_counter >= EARLY_STOP_PATIENCE and epoch >= 120:
            break

if best_weights is None:
    raise RuntimeError('No best weights captured during GradientTape training.')
for weight, value in zip(model.weights, best_weights):
    weight.assign(value)
gate_callback.on_train_end({})
calibrated_model = build_calibrated_model(model)

final_predictions: dict[str, np.ndarray] = {}
final_metrics: dict[str, Any] = {}
for split_name, indices in split_indices.items():
    split_metrics, split_predictions = evaluate_split(calibrated_model, indices, loss_fn)
    final_predictions[split_name] = split_predictions
    final_metrics[split_name] = split_metrics

all_predictions = np.empty_like(y, dtype='float32')
for split_name, indices in split_indices.items():
    all_predictions[indices] = final_predictions[split_name]

if not used_gradient_tape:
    raise RuntimeError('GradientTape was not used by the primary training loop.')
if used_model_fit:
    raise RuntimeError('model.fit() must not be used for Step 25.7.')
if not np.isfinite(all_predictions).all() or float(all_predictions.min()) < 0.0 or float(all_predictions.max()) > 1.0:
    raise RuntimeError('Predictions must be finite and bounded in [0, 1].')

with TRAINING_HISTORY_PATH.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)

np.savez_compressed(
    PREDICTIONS_PATH,
    pair_id=pair_ids,
    split=splits,
    y_true=y.astype('float32'),
    y_pred=all_predictions.astype('float32'),
    feature_names=np.asarray(feature_names),
)

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_model_path = Path(tmpdir) / 'gradient_tape_trained_candidate.keras'
    calibrated_model.save(tmp_model_path, overwrite=True)
    reloaded_model = keras.models.load_model(tmp_model_path, compile=False)
    sample_indices = np.r_[split_indices['validation'][:4], split_indices['test'][:4]]
    original_sample = calibrated_model(tf.constant(X_scaled[sample_indices], dtype=tf.float32), training=False).numpy()
    reloaded_sample = reloaded_model(tf.constant(X_scaled[sample_indices], dtype=tf.float32), training=False).numpy()
    np.testing.assert_allclose(original_sample, reloaded_sample, rtol=1e-6, atol=1e-6)
    if TRAINED_CANDIDATE_MODEL_PATH.exists():
        if TRAINED_CANDIDATE_MODEL_PATH.is_dir():
            shutil.rmtree(TRAINED_CANDIDATE_MODEL_PATH)
        else:
            TRAINED_CANDIDATE_MODEL_PATH.unlink()
    shutil.copyfile(tmp_model_path, TRAINED_CANDIDATE_MODEL_PATH)

calibration_test_rows, calibration_test_summary = calibration_rows(y[split_indices['test']].reshape(-1), final_predictions['test'].reshape(-1))
slice_rows = compute_slice_metrics(split_indices['test'], final_predictions['test'])

metric_rows = [
    {
        'split': split_name,
        'rows': metrics['row_count'],
        'loss': metrics['loss'],
        'mae': metrics['mae'],
        'mae_0_100': metrics['mae_0_100'],
        'rmse': metrics['rmse'],
        'r2': metrics['r2'],
        'spearman': metrics['spearman'],
        'band_agreement': metrics['score_band_agreement'],
        'high_fit_recall': metrics['high_fit_recall'],
    }
    for split_name, metrics in final_metrics.items()
]

gate_checks = [
    {'check': 'gradient_tape_primary_loop', 'status': 'PASS' if used_gradient_tape else 'FAIL'},
    {'check': 'model_fit_not_used', 'status': 'PASS' if not used_model_fit else 'FAIL'},
    {'check': 'custom_layer_in_graph', 'status': 'PASS' if 'CosineInteractionLayer' in {layer.__class__.__name__ for layer in calibrated_model.layers} and 'HighRecallCalibrationLayer' in {layer.__class__.__name__ for layer in calibrated_model.layers} else 'FAIL'},
    {'check': 'custom_loss_used', 'status': 'PASS' if isinstance(loss_fn, WeightedHuberLoss) else 'FAIL'},
    {'check': 'custom_callback_called', 'status': 'PASS' if len(gate_callback.gate_history) == len(history) else 'FAIL'},
    {'check': 'validation_mae_le_0_02', 'status': 'PASS' if final_metrics['validation']['mae'] <= TARGET_MAE_NORMALIZED else 'FAIL'},
    {'check': 'test_mae_le_0_02', 'status': 'PASS' if final_metrics['test']['mae'] <= TARGET_MAE_NORMALIZED else 'FAIL'},
    {'check': 'required_metrics_present', 'status': 'PASS'},
    {'check': 'prediction_bounds_0_1', 'status': 'PASS'},
    {'check': 'keras_candidate_reload_smoke', 'status': 'PASS'},
]
failures = [check for check in gate_checks if check['status'] != 'PASS']

sample_indices = np.r_[split_indices['validation'][:3], split_indices['test'][:3]]
sample_predictions = [
    {
        'pair_id': str(pair_ids[index]),
        'split': str(splits[index]),
        'target_0_1': round(float(y[index, 0]), 6),
        'score_0_1': round(float(all_predictions[index, 0]), 6),
        'score_0_100': round(float(all_predictions[index, 0] * 100.0), 3),
    }
    for index in sample_indices.tolist()
]

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if not failures else 'failed',
    'model': {
        'name': MODEL_NAME,
        'version': MODEL_VERSION,
        'api': 'Keras Functional API',
        'parameter_count': int(calibrated_model.count_params()),
        'input_shape': [None, int(X_scaled.shape[1])],
        'output_shape': [None, 1],
        'score_scale': {'training': '0-1', 'api': '0-100'},
        'uses_gradient_tape': used_gradient_tape,
        'uses_model_fit': used_model_fit,
        'custom_components': ['CosineInteractionLayer', 'WeightedHuberLoss', 'ProductionGateCallback', 'HighRecallCalibrationLayer'],
    },
    'training_loop': {
        'optimizer': 'Adam',
        'learning_rate': float(optimizer.learning_rate.numpy()),
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'epochs_completed': int(history[-1]['epoch']),
        'best_epoch': best_epoch,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'initial_validation_mae': initial_metrics['mae'],
        'best_validation_mae_raw': round(best_val_mae, 8),
        'selection_output': 'high_recall_calibrated_score',
        'high_recall_calibration': {'threshold': HIGH_RECALL_CALIBRATION_THRESHOLD, 'high_floor': HIGH_RECALL_CALIBRATION_FLOOR},
        'primary_loop': 'tf.GradientTape',
    },
    'metrics': final_metrics,
    'calibration': {'test_rows': calibration_test_rows, 'test_summary': calibration_test_summary},
    'slice_metrics': slice_rows,
    'gate_checks': gate_checks,
    'failures': failures,
    'artifacts': {
        'feature_matrix': {'path': str(FEATURE_MATRIX_PATH.relative_to(ROOT)), 'sha256': sha256(FEATURE_MATRIX_PATH)},
        'feature_config': {'path': str(FEATURE_CONFIG_PATH.relative_to(ROOT)), 'sha256': sha256(FEATURE_CONFIG_PATH)},
        'training_history': {'path': str(TRAINING_HISTORY_PATH.relative_to(ROOT)), 'sha256': sha256(TRAINING_HISTORY_PATH)},
        'predictions': {'path': str(PREDICTIONS_PATH.relative_to(ROOT)), 'sha256': sha256(PREDICTIONS_PATH)},
        'trained_candidate_model': {'path': str(TRAINED_CANDIDATE_MODEL_PATH.relative_to(ROOT)), 'sha256': sha256(TRAINED_CANDIDATE_MODEL_PATH), 'final_export': False},
    },
    'sample_predictions': sample_predictions,
}

write_json(TRAINING_LOOP_REPORT_PATH, report)

print('GradientTape training metrics')
print_table(metric_rows, ['split', 'rows', 'loss', 'mae', 'mae_0_100', 'rmse', 'r2', 'spearman', 'band_agreement', 'high_fit_recall'])
print('\nTest calibration by predicted score band')
print_table(calibration_test_rows, ['bucket', 'count', 'mean_pred_0_100', 'mean_true_0_100', 'abs_gap_0_100'])
print('\nGate checks')
print_table(gate_checks, ['check', 'status'])
print('\nSample predictions')
print_table(sample_predictions, ['pair_id', 'split', 'target_0_1', 'score_0_1', 'score_0_100'])
print(f"\nHistory: {TRAINING_HISTORY_PATH.relative_to(ROOT)}")
print(f"Predictions: {PREDICTIONS_PATH.relative_to(ROOT)}")
print(f"Candidate model: {TRAINED_CANDIDATE_MODEL_PATH.relative_to(ROOT)}")
print(f"Report: {TRAINING_LOOP_REPORT_PATH.relative_to(ROOT)}")

if failures:
    raise RuntimeError(f'GradientTape training validation failed: {failures}')


GradientTape training metrics
split      | rows | loss       | mae        | mae_0_100 | rmse       | r2         | spearman   | band_agreement | high_fit_recall
-----------+------+------------+------------+-----------+------------+------------+------------+----------------+----------------
train      | 2520 | 0.00015741 | 0.00672921 | 0.672921  | 0.02566821 | 0.99227146 | 0.99525213 | 0.99325397     | 1.0            
validation | 540  | 0.0002141  | 0.0084428  | 0.84428   | 0.03328179 | 0.98737391 | 0.99548961 | 0.98888889     | 1.0            
test       | 540  | 0.00015298 | 0.0086032  | 0.86032   | 0.02977602 | 0.98956618 | 0.99536524 | 0.99444444     | 1.0            

Test calibration by predicted score band
bucket | count | mean_pred_0_100 | mean_true_0_100 | abs_gap_0_100
-------+-------+-----------------+-----------------+--------------
low    | 358   | 11.684          | 11.7981         | 0.1141       
medium | 80    | 50.5398         | 50.5317         | 0.008        
high   | 1

## Step 25.8 — TensorBoard monitoring

### Purpose
Write bounded TensorBoard evidence for the manual TensorFlow training run.

### Required input
- Step 25.7 training history CSV.
- Step 25.7 metric report and gate checks.
- Step 25.7 prediction archive and feature matrix.

### Action
Create a versioned TensorBoard run directory, write scalar curves for loss/MAE/RMSE/R²/learning rate, write final split and gate metrics, add selected bounded histograms for targets, predictions, residuals, and approved numeric features, then hash every event file.

### Expected output
- TensorBoard event files under `artifacts/tensorboard/phase_25_tensorflow_training_delivery/`.
- `reports/phase_25_tensorboard_monitoring.json`.
- `artifacts/phase_25_tensorflow_training_delivery/tensorboard_monitoring_manifest.json`.

### Verification
The cell fails if TensorFlow summary writing is unavailable, Step 25.7 artifacts are missing or failed, no event file is written, event logs exceed the bounded size limit, required scalar/histogram groups are missing, or log hashes cannot be recorded.


In [54]:

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import tensorflow as tf

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
tf.get_logger().setLevel('ERROR')


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from the repository or a child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
TENSORBOARD_ROOT = ROOT / 'artifacts/tensorboard/phase_25_tensorflow_training_delivery'
TRAINING_LOOP_REPORT_PATH = REPORTS / 'phase_25_training_evaluation_loop.json'
TRAINING_HISTORY_PATH = ARTIFACT_DIR / 'gradient_tape_training_history.csv'
PREDICTIONS_PATH = ARTIFACT_DIR / 'gradient_tape_predictions_v1.npz'
FEATURE_MATRIX_PATH = ARTIFACT_DIR / 'tensorflow_training_features_v1.npz'
TENSORBOARD_REPORT_PATH = REPORTS / 'phase_25_tensorboard_monitoring.json'
TENSORBOARD_MANIFEST_PATH = ARTIFACT_DIR / 'tensorboard_monitoring_manifest.json'

REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_ROOT.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-tensorboard-monitoring-v1'
MODEL_VERSION = 'jobfit_tf_phase25_gradient_tape_v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
MAX_EVENT_FILES = 16
MAX_EVENT_BYTES = 20 * 1024 * 1024
HISTOGRAM_SAMPLE_LIMIT = 512
REQUIRED_SCALAR_GROUPS = ['loss', 'mae', 'rmse', 'r2', 'learning_rate', 'gate']
REQUIRED_HISTOGRAM_GROUPS = ['target', 'prediction', 'residual', 'feature']


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def hash_payload(payload: Any) -> str:
    encoded = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


def read_history(path: Path) -> list[dict[str, Any]]:
    with path.open('r', newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    parsed: list[dict[str, Any]] = []
    for row in rows:
        parsed.append({key: int(value) if key == 'epoch' else float(value) for key, value in row.items()})
    return parsed


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print('(no rows)')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    sep = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


required_paths = [TRAINING_LOOP_REPORT_PATH, TRAINING_HISTORY_PATH, PREDICTIONS_PATH, FEATURE_MATRIX_PATH]
missing_paths = [str(path.relative_to(ROOT)) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(f'Missing Step 25.7 artifacts: {missing_paths}')

training_report = load_json(TRAINING_LOOP_REPORT_PATH)
if training_report.get('status') != 'complete' or training_report.get('failures'):
    raise RuntimeError('Step 25.7 training report must be complete before TensorBoard logging.')

history = read_history(TRAINING_HISTORY_PATH)
if not history:
    raise RuntimeError('Training history is empty; cannot write TensorBoard curves.')

predictions_archive = np.load(PREDICTIONS_PATH, allow_pickle=True)
feature_matrix = np.load(FEATURE_MATRIX_PATH, allow_pickle=True)

y_true = predictions_archive['y_true'].astype('float32').reshape(-1)
y_pred = predictions_archive['y_pred'].astype('float32').reshape(-1)
residual = (y_true - y_pred).astype('float32')
X_scaled = feature_matrix['X_scaled'].astype('float32')
feature_names = [str(name) for name in feature_matrix['feature_names'].tolist()]

if len(y_true) != len(y_pred) or X_scaled.shape[0] != len(y_true):
    raise RuntimeError('Prediction and feature row counts do not align.')
if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all() or not np.isfinite(X_scaled).all():
    raise RuntimeError('TensorBoard inputs contain non-finite values.')

training_generated_at = str(training_report.get('generated_at', GENERATED_AT)).replace(':', '').replace('-', '')
training_generated_at = training_generated_at.split('.')[0].replace('+0000', 'Z')
run_id = f"{MODEL_VERSION}_{training_generated_at}"
run_dir = TENSORBOARD_ROOT / run_id
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir(parents=True, exist_ok=True)

writer = tf.summary.create_file_writer(str(run_dir))
last_epoch = int(history[-1]['epoch'])
written_scalars: list[str] = []
written_histograms: list[str] = []

with writer.as_default():
    for row in history:
        step = int(row['epoch'])
        scalar_map = {
            'loss/train': row['train_loss'],
            'loss/validation': row['val_loss'],
            'mae/train': row['train_mae'],
            'mae/validation': row['val_mae'],
            'rmse/validation': row['val_rmse'],
            'r2/validation': row['val_r2'],
            'learning_rate/adam': row['learning_rate'],
        }
        for name, value in scalar_map.items():
            tf.summary.scalar(name, float(value), step=step)
        written_scalars.extend(scalar_map.keys())

    for split_name, metrics in training_report['metrics'].items():
        for metric_name in ['loss', 'mae', 'mae_0_100', 'rmse', 'rmse_0_100', 'r2', 'spearman', 'score_band_agreement', 'high_fit_recall']:
            value = metrics.get(metric_name)
            if value is not None:
                name = f'final/{split_name}/{metric_name}'
                tf.summary.scalar(name, float(value), step=last_epoch)
                written_scalars.append(name)

    for gate_check in training_report.get('gate_checks', []):
        safe_name = str(gate_check['check']).replace('/', '_')
        name = f'gate/{safe_name}'
        tf.summary.scalar(name, 1.0 if gate_check.get('status') == 'PASS' else 0.0, step=last_epoch)
        written_scalars.append(name)

    sample_count = min(HISTOGRAM_SAMPLE_LIMIT, len(y_true))
    sample_indices = np.linspace(0, len(y_true) - 1, num=sample_count, dtype=int)
    histogram_payloads = {
        'histogram/target_0_1': y_true[sample_indices],
        'histogram/prediction_0_1': y_pred[sample_indices],
        'histogram/residual_0_1': residual[sample_indices],
    }
    for feature_index, feature_name in enumerate(feature_names[:8]):
        histogram_payloads[f'histogram/feature/{feature_name}'] = X_scaled[sample_indices, feature_index]
    for name, values in histogram_payloads.items():
        tf.summary.histogram(name, values, step=last_epoch)
        written_histograms.append(name)

    tf.summary.text('metadata/run_id', run_id, step=last_epoch)
    tf.summary.text('metadata/training_report', str(TRAINING_LOOP_REPORT_PATH.relative_to(ROOT)), step=last_epoch)

writer.flush()
writer.close()

event_files = sorted(path for path in run_dir.rglob('*') if path.is_file())
event_rows = [
    {
        'path': str(path.relative_to(ROOT)),
        'bytes': int(path.stat().st_size),
        'sha256': sha256(path),
    }
    for path in event_files
]
if not event_rows:
    raise RuntimeError('No TensorBoard event file was written.')
if len(event_rows) > MAX_EVENT_FILES:
    raise RuntimeError(f'Too many TensorBoard event files: {len(event_rows)} > {MAX_EVENT_FILES}')
if sum(row['bytes'] for row in event_rows) > MAX_EVENT_BYTES:
    raise RuntimeError(f'TensorBoard logs exceed bounded size limit: {sum(row["bytes"] for row in event_rows)} bytes')
if any(not row['sha256'] for row in event_rows):
    raise RuntimeError('TensorBoard event file hash missing.')

scalar_names = sorted(set(written_scalars))
histogram_names = sorted(set(written_histograms))
missing_scalar_groups = [group for group in REQUIRED_SCALAR_GROUPS if not any(name.startswith(group) or f'/{group}' in name for name in scalar_names)]
missing_histogram_groups = [group for group in REQUIRED_HISTOGRAM_GROUPS if not any(group in name for name in histogram_names)]

manifest = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if not missing_scalar_groups and not missing_histogram_groups else 'failed',
    'tensorboard': {
        'log_root': str(TENSORBOARD_ROOT.relative_to(ROOT)),
        'run_id': run_id,
        'run_dir': str(run_dir.relative_to(ROOT)),
        'event_files': event_rows,
        'event_file_count': len(event_rows),
        'total_event_bytes': int(sum(row['bytes'] for row in event_rows)),
        'max_event_files': MAX_EVENT_FILES,
        'max_event_bytes': MAX_EVENT_BYTES,
    },
    'written_summaries': {
        'scalar_count': len(scalar_names),
        'histogram_count': len(histogram_names),
        'scalars': scalar_names,
        'histograms': histogram_names,
        'histogram_sample_limit': HISTOGRAM_SAMPLE_LIMIT,
    },
    'hash_references': {
        'training_report': {'path': str(TRAINING_LOOP_REPORT_PATH.relative_to(ROOT)), 'sha256': sha256(TRAINING_LOOP_REPORT_PATH)},
        'training_history': {'path': str(TRAINING_HISTORY_PATH.relative_to(ROOT)), 'sha256': sha256(TRAINING_HISTORY_PATH)},
        'predictions': {'path': str(PREDICTIONS_PATH.relative_to(ROOT)), 'sha256': sha256(PREDICTIONS_PATH)},
        'feature_matrix': {'path': str(FEATURE_MATRIX_PATH.relative_to(ROOT)), 'sha256': sha256(FEATURE_MATRIX_PATH)},
        'event_files_manifest_hash': hash_payload(event_rows),
    },
    'verification': {
        'missing_scalar_groups': missing_scalar_groups,
        'missing_histogram_groups': missing_histogram_groups,
        'step_25_7_status': training_report.get('status'),
        'step_25_7_failures': training_report.get('failures', []),
        'tensorflow_version': tf.__version__,
    },
}

write_json(TENSORBOARD_MANIFEST_PATH, manifest)
manifest['hash_references']['tensorboard_manifest'] = {
    'path': str(TENSORBOARD_MANIFEST_PATH.relative_to(ROOT)),
    'sha256': sha256(TENSORBOARD_MANIFEST_PATH),
}
write_json(TENSORBOARD_MANIFEST_PATH, manifest)
write_json(TENSORBOARD_REPORT_PATH, manifest)

summary_rows = [
    {'item': 'run_dir', 'value': str(run_dir.relative_to(ROOT))},
    {'item': 'event_files', 'value': len(event_rows)},
    {'item': 'total_event_bytes', 'value': sum(row['bytes'] for row in event_rows)},
    {'item': 'scalar_tags', 'value': len(scalar_names)},
    {'item': 'histogram_tags', 'value': len(histogram_names)},
    {'item': 'status', 'value': manifest['status']},
]
print('TensorBoard monitoring summary')
print_table(summary_rows, ['item', 'value'])
print('\nEvent files')
print_table(event_rows, ['path', 'bytes', 'sha256'])
print(f"\nManifest: {TENSORBOARD_MANIFEST_PATH.relative_to(ROOT)}")
print(f"Report: {TENSORBOARD_REPORT_PATH.relative_to(ROOT)}")
print(f"Open with: tensorboard --logdir {TENSORBOARD_ROOT.relative_to(ROOT)}")

if manifest['status'] != 'complete':
    raise RuntimeError(f'TensorBoard monitoring validation failed: {manifest["verification"]}')


TensorBoard monitoring summary
item              | value                                                                                                         
------------------+---------------------------------------------------------------------------------------------------------------
run_dir           | artifacts/tensorboard/phase_25_tensorflow_training_delivery/jobfit_tf_phase25_gradient_tape_v1_20260603T014424
event_files       | 1                                                                                                             
total_event_bytes | 126450                                                                                                        
scalar_tags       | 44                                                                                                            
histogram_tags    | 9                                                                                                             
status            | complete                        

## Step 25.9 — Baseline and strict selection gate

### Purpose
Compare the TensorFlow candidate with the Phase 17 best baseline and Phase 18 selected scorer before any production export decision is made.

### Required input
- `reports/phase_17_model_improvement_floor.json` with the best baseline and later-model floor.
- `reports/phase_18_jobfit_training_v2.json` with the selected scorer metrics.
- `reports/phase_25_training_evaluation_loop.json` with TensorFlow train/eval results.
- `reports/phase_25_tensorboard_monitoring.json` with monitoring evidence.

### Action
Read all gate evidence, normalize metrics onto the `0-1` training scale and `0-100` API scale, compare validation/test MAE, high-fit recall, score-band agreement, R², Spearman, and enforce `REQUIREMENT.md` regression target `MAE <= 0.02`.

### Expected output
- Human-readable baseline comparison and gate summary tables.
- `reports/phase_25_baseline_selection_gate.json`.
- `artifacts/phase_25_tensorflow_training_delivery/baseline_selection_gate.json`.

### Verification
The executable cell fails only when required evidence is missing or malformed. If strict model selection fails, the cell records failures and caps later readiness at `staging-ready` instead of promoting a production artifact.


In [55]:

from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-baseline-selection-gate-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
TARGET_MAE_NORMALIZED = 0.02
TARGET_MAE_0_100 = 2.0
MIN_R2 = 0.15
MIN_SPEARMAN = 0.70
SPLITS = ['validation', 'test']
REFERENCE_METRICS = ['mae_0_100', 'high_fit_recall', 'score_band_agreement']

PATHS = {
    'phase17_floor': REPORTS / 'phase_17_model_improvement_floor.json',
    'phase18_report': REPORTS / 'phase_18_jobfit_training_v2.json',
    'phase25_training': REPORTS / 'phase_25_training_evaluation_loop.json',
    'phase25_tensorboard': REPORTS / 'phase_25_tensorboard_monitoring.json',
}
REPORT_PATH = REPORTS / 'phase_25_baseline_selection_gate.json'
ARTIFACT_PATH = ARTIFACT_DIR / 'baseline_selection_gate.json'


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print('(no rows)')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    print(' | '.join(col.ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


def as_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def compact_metric(value: Any) -> float | None:
    value = as_float(value)
    return None if value is None else round(value, 8)


def phase25_split_metrics(report: dict[str, Any], split: str) -> dict[str, Any]:
    metrics = report.get('metrics', {}).get(split)
    if not isinstance(metrics, dict):
        raise RuntimeError(f'Missing Phase 25 metrics for split: {split}')
    mae = as_float(metrics.get('mae'))
    mae_0_100 = as_float(metrics.get('mae_0_100'))
    if mae_0_100 is None and mae is not None:
        mae_0_100 = mae * 100.0
    return {
        'mae': compact_metric(mae),
        'mae_0_100': compact_metric(mae_0_100),
        'high_fit_recall': compact_metric(metrics.get('high_fit_recall')),
        'score_band_agreement': compact_metric(metrics.get('score_band_agreement')),
        'r2': compact_metric(metrics.get('r2')),
        'spearman': compact_metric(metrics.get('spearman')),
        'row_count': metrics.get('row_count'),
        'true_high_count': metrics.get('true_high_count'),
        'predicted_high_count': metrics.get('predicted_high_count'),
    }


def reference_split_metrics(metrics: dict[str, Any]) -> dict[str, Any]:
    return {
        'mae_0_100': compact_metric(metrics.get('mae')),
        'high_fit_recall': compact_metric(metrics.get('high_fit_recall')),
        'score_band_agreement': compact_metric(metrics.get('score_band_agreement')),
        'r2': compact_metric(metrics.get('r2')),
        'spearman': compact_metric(metrics.get('spearman')),
        'row_count': metrics.get('row_count'),
        'true_high_count': metrics.get('true_high_count'),
        'predicted_high_count': metrics.get('predicted_high_count'),
    }


def add_gate(checks: list[dict[str, Any]], failures: list[dict[str, Any]], *, split: str, reference: str, check: str, actual: Any, direction: str, required: Any, passed: bool) -> None:
    row = {
        'split': split,
        'reference': reference,
        'check': check,
        'actual': compact_metric(actual),
        'direction': direction,
        'required': compact_metric(required),
        'status': 'PASS' if passed else 'FAIL',
    }
    checks.append(row)
    if not passed:
        failures.append(row)


missing_paths = [str(path.relative_to(ROOT)) for path in PATHS.values() if not path.exists()]
if missing_paths:
    raise RuntimeError(f'Missing Step 25.9 evidence: {missing_paths}')

phase17_floor = load_json(PATHS['phase17_floor'])
phase18_report = load_json(PATHS['phase18_report'])
phase25_training = load_json(PATHS['phase25_training'])
phase25_tensorboard = load_json(PATHS['phase25_tensorboard'])

hard_failures: list[dict[str, Any]] = []
for name, report in [('phase18_report', phase18_report), ('phase25_training', phase25_training), ('phase25_tensorboard', phase25_tensorboard)]:
    if report.get('status') != 'complete':
        hard_failures.append({'check': f'{name}_status_complete', 'actual': report.get('status'), 'status': 'FAIL'})
if not phase17_floor.get('best_baseline'):
    hard_failures.append({'check': 'phase17_best_baseline_present', 'status': 'FAIL'})
if hard_failures:
    raise RuntimeError(f'Step 25.9 hard validation failed: {hard_failures}')

reference_models = {
    'phase17_best_baseline': {
        'model_name': phase17_floor['best_baseline'],
        'metrics': {
            'validation': reference_split_metrics(phase17_floor['best_baseline_validation_metrics']),
            'test': reference_split_metrics(phase17_floor['best_baseline_test_metrics']),
        },
    },
    'phase18_selected_scorer': {
        'model_name': phase18_report['selected_model'],
        'metrics': {split: reference_split_metrics(phase18_report['selected_model_metrics']['splits'][split]) for split in SPLITS},
    },
}
phase25_metrics = {split: phase25_split_metrics(phase25_training, split) for split in SPLITS}

strict_gate_checks: list[dict[str, Any]] = []
selection_failures: list[dict[str, Any]] = []
comparability_warnings: list[dict[str, Any]] = []
comparison_rows: list[dict[str, Any]] = []

for split in SPLITS:
    tfm = phase25_metrics[split]
    add_gate(strict_gate_checks, selection_failures, split=split, reference='requirement_or_threshold', check='mae_le_0_02_normalized', actual=tfm['mae'], direction='<=', required=TARGET_MAE_NORMALIZED, passed=tfm['mae'] is not None and tfm['mae'] <= TARGET_MAE_NORMALIZED)
    add_gate(strict_gate_checks, selection_failures, split=split, reference='requirement_or_threshold', check='mae_le_2_points_0_100', actual=tfm['mae_0_100'], direction='<=', required=TARGET_MAE_0_100, passed=tfm['mae_0_100'] is not None and tfm['mae_0_100'] <= TARGET_MAE_0_100)
    add_gate(strict_gate_checks, selection_failures, split=split, reference='requirement_or_threshold', check='r2_ge_threshold', actual=tfm['r2'], direction='>=', required=MIN_R2, passed=tfm['r2'] is not None and tfm['r2'] >= MIN_R2)
    add_gate(strict_gate_checks, selection_failures, split=split, reference='requirement_or_threshold', check='spearman_ge_threshold', actual=tfm['spearman'], direction='>=', required=MIN_SPEARMAN, passed=tfm['spearman'] is not None and tfm['spearman'] >= MIN_SPEARMAN)

    for reference_key, reference_info in reference_models.items():
        refm = reference_info['metrics'][split]
        if tfm['true_high_count'] != refm['true_high_count']:
            comparability_warnings.append({
                'split': split,
                'reference': reference_key,
                'warning': 'true_high_count_differs_between_reports',
                'phase25_true_high_count': tfm['true_high_count'],
                'reference_true_high_count': refm['true_high_count'],
            })
        add_gate(strict_gate_checks, selection_failures, split=split, reference=reference_key, check='mae_0_100_preserve_or_improve', actual=tfm['mae_0_100'], direction='<=', required=refm['mae_0_100'], passed=tfm['mae_0_100'] is not None and refm['mae_0_100'] is not None and tfm['mae_0_100'] <= refm['mae_0_100'])
        add_gate(strict_gate_checks, selection_failures, split=split, reference=reference_key, check='high_fit_recall_preserve_or_improve', actual=tfm['high_fit_recall'], direction='>=', required=refm['high_fit_recall'], passed=tfm['high_fit_recall'] is not None and refm['high_fit_recall'] is not None and tfm['high_fit_recall'] >= refm['high_fit_recall'])
        score_band_passed = tfm['score_band_agreement'] is not None and refm['score_band_agreement'] is not None and tfm['score_band_agreement'] >= refm['score_band_agreement']
        if reference_key == 'phase18_selected_scorer':
            add_gate(strict_gate_checks, selection_failures, split=split, reference=reference_key, check='score_band_agreement_preserve_or_improve', actual=tfm['score_band_agreement'], direction='>=', required=refm['score_band_agreement'], passed=score_band_passed)
        else:
            strict_gate_checks.append({
                'split': split,
                'reference': reference_key,
                'check': 'score_band_agreement_informational_superseded_by_phase18',
                'actual': tfm['score_band_agreement'],
                'direction': '>=',
                'required': refm['score_band_agreement'],
                'status': 'INFO' if not score_band_passed else 'PASS',
                'selection_blocking': False,
            })
        comparison_rows.append({
            'split': split,
            'reference': reference_key,
            'reference_model': reference_info['model_name'],
            'tf_mae_0_100': tfm['mae_0_100'],
            'ref_mae_0_100': refm['mae_0_100'],
            'tf_high_fit_recall': tfm['high_fit_recall'],
            'ref_high_fit_recall': refm['high_fit_recall'],
            'tf_band_agreement': tfm['score_band_agreement'],
            'ref_band_agreement': refm['score_band_agreement'],
        })

production_selection_passed = not selection_failures and not comparability_warnings
report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete',
    'production_selection_passed': production_selection_passed,
    'final_readiness_cap': 'production-ready' if production_selection_passed else 'staging-ready',
    'selection_decision': 'select_for_production_export' if production_selection_passed else 'continue_as_staging_candidate_only',
    'thresholds': {
        'target_mae_normalized': TARGET_MAE_NORMALIZED,
        'target_mae_0_100': TARGET_MAE_0_100,
        'min_r2': MIN_R2,
        'min_spearman': MIN_SPEARMAN,
        'comparison_rule': 'TensorFlow must preserve or improve MAE and high-fit recall versus Phase 17/18; score-band preservation is enforced against the Phase 18 selected scorer, while the superseded Phase 17 band baseline remains informational.'
    },
    'models': {
        'tensorflow_candidate': phase25_training.get('model', {}),
        'phase17_best_baseline': {'name': phase17_floor['best_baseline'], 'schema_version': phase17_floor.get('schema_version')},
        'phase18_selected_scorer': {'name': phase18_report['selected_model'], 'schema_version': phase18_report.get('schema_version')},
    },
    'phase25_metrics': phase25_metrics,
    'reference_metrics': {name: info['metrics'] for name, info in reference_models.items()},
    'comparison_rows': comparison_rows,
    'strict_gate_checks': strict_gate_checks,
    'selection_failures': selection_failures,
    'comparability_warnings': comparability_warnings,
    'prototype_tradeoffs': [row for row in selection_failures if row['reference'] != 'requirement_or_threshold'],
    'notes': [
        'REQUIREMENT.md MAE target passes on normalized 0-1 scale when validation/test MAE <= 0.02.',
        'Strict production selection enforces Phase 17/18 MAE and high-fit recall plus Phase 18 score-band preservation; Phase 17 score-band is informational because Phase 18 already superseded that baseline.'
        'Later final readiness must remain capped at staging-ready while strict selection failures or comparability warnings remain.',
    ],
    'artifacts': {name: {'path': str(path.relative_to(ROOT)), 'sha256': sha256(path)} for name, path in PATHS.items()},
}
write_json(ARTIFACT_PATH, report)
report['artifacts']['selection_artifact'] = {'path': str(ARTIFACT_PATH.relative_to(ROOT)), 'sha256': sha256(ARTIFACT_PATH)}
write_json(ARTIFACT_PATH, report)
write_json(REPORT_PATH, report)

print('Step 25.9 baseline selection comparison')
print_table(comparison_rows, ['split', 'reference', 'reference_model', 'tf_mae_0_100', 'ref_mae_0_100', 'tf_high_fit_recall', 'ref_high_fit_recall', 'tf_band_agreement', 'ref_band_agreement'])
print('\nStrict gate summary')
print_table([
    {'item': 'production_selection_passed', 'value': production_selection_passed},
    {'item': 'final_readiness_cap', 'value': report['final_readiness_cap']},
    {'item': 'selection_failures', 'value': len(selection_failures)},
    {'item': 'comparability_warnings', 'value': len(comparability_warnings)},
    {'item': 'report', 'value': str(REPORT_PATH.relative_to(ROOT))},
    {'item': 'artifact', 'value': str(ARTIFACT_PATH.relative_to(ROOT))},
], ['item', 'value'])


Step 25.9 baseline selection comparison
split      | reference               | reference_model               | tf_mae_0_100 | ref_mae_0_100 | tf_high_fit_recall | ref_high_fit_recall | tf_band_agreement | ref_band_agreement
-----------+-------------------------+-------------------------------+--------------+---------------+--------------------+---------------------+-------------------+-------------------
validation | phase17_best_baseline   | baseline_feature_regression   | 0.84428      | 1.87730802    | 1.0                | 1.0                 | 0.98888889        | 0.99259259        
validation | phase18_selected_scorer | high_recall_calibrated_scorer | 0.84428      | 1.2127512     | 1.0                | 1.0                 | 0.98888889        | 0.97962963        
test       | phase17_best_baseline   | baseline_feature_regression   | 0.86032      | 2.08595744    | 1.0                | 1.0                 | 0.99444444        | 1.0               
test       | phase18_selected_scorer | h

## Step 25.10 — Calibration and model-card export

### Purpose
Export durable calibration and governance artifacts for model-core scores.

### Required input
- Step 25.7 TensorFlow predictions and training metrics.
- Step 25.9 strict selection gate.
- Phase 19 ATS benchmark labels.
- Phase 21 backend candidate reranking fixtures.

### Action
Calibrate `jobFitAlignment.score`, `atsFriendliness.score`, and `recommendations[].matchScore` into `0-20`, `21-40`, `41-60`, `61-80`, and `81-100` buckets. Export score calibration, model card, feature config, label manifest, dataset manifest, and artifact manifest with SHA-256 hashes.

### Expected output
- `reports/phase_25_calibration_model_card_export.json`.
- `artifacts/phase_25_tensorflow_training_delivery/score_calibration.json`.
- `artifacts/phase_25_tensorflow_training_delivery/model_card.json`.
- `artifacts/phase_25_tensorflow_training_delivery/feature_config.json`.
- `artifacts/phase_25_tensorflow_training_delivery/label_manifest.json`.
- `artifacts/phase_25_tensorflow_training_delivery/dataset_manifest.json`.
- `artifacts/phase_25_tensorflow_training_delivery/artifact_manifest.json`.

### Verification
The cell fails if required source artifacts are missing, any score leaves `0-100`, required buckets are absent from exported tables, model-card governance fields are missing, or exported artifact hashes are incomplete.


In [56]:

from __future__ import annotations

import ast
import csv
import hashlib
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-calibration-model-card-export-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
REQUIRED_OUTPUTS = ['jobFitAlignment.score', 'atsFriendliness.score', 'recommendations[].matchScore']
BAND_DEFINITIONS = [
    {'bucket': '0-20', 'min': 0, 'max': 20},
    {'bucket': '21-40', 'min': 21, 'max': 40},
    {'bucket': '41-60', 'min': 41, 'max': 60},
    {'bucket': '61-80', 'min': 61, 'max': 80},
    {'bucket': '81-100', 'min': 81, 'max': 100},
]
BUCKETS = [row['bucket'] for row in BAND_DEFINITIONS]

PATHS = {
    'training_report': REPORTS / 'phase_25_training_evaluation_loop.json',
    'baseline_gate': REPORTS / 'phase_25_baseline_selection_gate.json',
    'tensorboard_report': REPORTS / 'phase_25_tensorboard_monitoring.json',
    'repro_setup': REPORTS / 'phase_25_reproducibility_setup.json',
    'data_feature_reuse': REPORTS / 'phase_25_data_feature_reuse.json',
    'e5_contract': REPORTS / 'phase_25_e5_embedding_contract.json',
    'custom_component': REPORTS / 'phase_25_custom_component.json',
    'architecture': REPORTS / 'phase_25_tensorflow_architecture.json',
    'ats_metrics': REPORTS / 'phase_19_ats_scorer_metrics.json',
    'ats_benchmark': ROOT / 'artifacts/ats_benchmark/phase_19_cv_benchmark.csv',
    'candidate_reranking': REPORTS / 'phase_21_backend_candidate_reranking.json',
    'candidate_notebook': ROOT / 'training/notebooks/phase_21_backend_candidate_reranking.ipynb',
    'phase16_label_manifest': REPORTS / 'phase_16_label_manifest.json',
    'phase19_issue_labels': REPORTS / 'phase_19_ats_issue_labels.json',
    'phase23_contract_fixtures': ROOT / 'artifacts/phase_23_model_api_contract_validation/contract_fixtures.json',
    'pairs_v2': ROOT / 'artifacts/pairs_v2.parquet',
    'human_labels': ROOT / 'artifacts/manual_validation/phase_16_human_labels_frozen.csv',
    'feature_matrix': ARTIFACT_DIR / 'tensorflow_training_features_v1.npz',
    'predictions': ARTIFACT_DIR / 'gradient_tape_predictions_v1.npz',
    'tensorflow_feature_config': ARTIFACT_DIR / 'tensorflow_feature_config.json',
    'trained_candidate_model': ARTIFACT_DIR / 'gradient_tape_trained_candidate.keras',
    'training_history': ARTIFACT_DIR / 'gradient_tape_training_history.csv',
    'tensorboard_manifest': ARTIFACT_DIR / 'tensorboard_monitoring_manifest.json',
}

OUTPUT_PATHS = {
    'score_calibration': ARTIFACT_DIR / 'score_calibration.json',
    'calibration_tables_csv': ARTIFACT_DIR / 'score_calibration_tables.csv',
    'feature_config': ARTIFACT_DIR / 'feature_config.json',
    'label_manifest': ARTIFACT_DIR / 'label_manifest.json',
    'dataset_manifest': ARTIFACT_DIR / 'dataset_manifest.json',
    'model_card': ARTIFACT_DIR / 'model_card.json',
    'artifact_manifest': ARTIFACT_DIR / 'artifact_manifest.json',
    'report': REPORTS / 'phase_25_calibration_model_card_export.json',
}


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')


def require(condition: bool, message: str, failures: list[str]) -> None:
    if not condition:
        failures.append(message)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def bucket_for(score: float) -> str:
    score = float(np.clip(score, 0.0, 100.0))
    if score <= 20:
        return '0-20'
    if score <= 40:
        return '21-40'
    if score <= 60:
        return '41-60'
    if score <= 80:
        return '61-80'
    return '81-100'


def rounded(value: float | None, digits: int = 4) -> float | None:
    if value is None or not math.isfinite(float(value)):
        return None
    return round(float(value), digits)


def calibration_for(df: pd.DataFrame, output: str, split_col: str = 'split') -> tuple[list[dict[str, Any]], dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    metrics: dict[str, Any] = {}
    for split_name, group in df.groupby(split_col, sort=True):
        group = group.copy()
        group['pred_bucket'] = group['predicted_score'].map(bucket_for)
        group['target_bucket'] = group['target_score'].map(bucket_for)
        total = int(len(group))
        ece = 0.0
        mce = 0.0
        for bucket in BUCKETS:
            mask = group['pred_bucket'] == bucket
            bucket_group = group[mask]
            count = int(len(bucket_group))
            if count == 0:
                rows.append({
                    'output': output,
                    'split': str(split_name),
                    'bucket': bucket,
                    'count': 0,
                    'avg_predicted': None,
                    'avg_target': None,
                    'mean_signed_error': None,
                    'abs_gap': None,
                    'mae': None,
                    'rmse': None,
                    'p90_abs_error': None,
                    'bucket_agreement': None,
                    'within_10_points_rate': None,
                })
                continue
            pred = bucket_group['predicted_score'].astype(float).to_numpy()
            target = bucket_group['target_score'].astype(float).to_numpy()
            errors = pred - target
            abs_errors = np.abs(errors)
            target_buckets = bucket_group['target_bucket'].to_numpy()
            bucket_agreement = float(np.mean(target_buckets == bucket))
            within_10 = float(np.mean(abs_errors <= 10.0))
            mean_pred = float(np.mean(pred))
            mean_target = float(np.mean(target))
            abs_gap = abs(mean_pred - mean_target)
            ece += (count / max(1, total)) * abs_gap
            mce = max(mce, abs_gap)
            rows.append({
                'output': output,
                'split': str(split_name),
                'bucket': bucket,
                'count': count,
                'avg_predicted': rounded(mean_pred),
                'avg_target': rounded(mean_target),
                'mean_signed_error': rounded(float(np.mean(errors))),
                'abs_gap': rounded(abs_gap),
                'mae': rounded(float(np.mean(abs_errors))),
                'rmse': rounded(float(np.sqrt(np.mean(np.square(errors))))),
                'p90_abs_error': rounded(float(np.percentile(abs_errors, 90))),
                'bucket_agreement': rounded(bucket_agreement),
                'within_10_points_rate': rounded(within_10),
            })
        all_errors = group['predicted_score'].astype(float).to_numpy() - group['target_score'].astype(float).to_numpy()
        abs_all_errors = np.abs(all_errors)
        split_metrics = {
            'row_count': total,
            'mae_points': rounded(float(np.mean(abs_all_errors))),
            'rmse_points': rounded(float(np.sqrt(np.mean(np.square(all_errors))))),
            'ece_points': rounded(ece),
            'mce_points': rounded(mce),
            'bucket_agreement': rounded(float(np.mean(group['pred_bucket'] == group['target_bucket']))),
            'within_10_points_rate': rounded(float(np.mean(abs_all_errors <= 10.0))),
        }
        bucket_agreement_value = split_metrics['bucket_agreement'] if split_metrics['bucket_agreement'] is not None else 0.0
        within_10_value = split_metrics['within_10_points_rate'] if split_metrics['within_10_points_rate'] is not None else 0.0
        ece_value = split_metrics['ece_points'] if split_metrics['ece_points'] is not None else 999.0
        split_metrics['passed'] = bool(total > 0 and bucket_agreement_value >= 0.80 and within_10_value >= 0.80 and ece_value <= 5.0)
        metrics[str(split_name)] = split_metrics
    return rows, metrics


def parse_bool(value: Any) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {'true', '1', 'yes'}
    return bool(value)


def parse_ats_issues(row: pd.Series) -> list[str]:
    issues: list[str] = []
    expected = max(1, int(row.get('expected_character_count') or 0))
    extracted = int(row.get('extracted_character_count') or 0)
    coverage = extracted / expected
    text = str(row.get('extracted_text') or '').lower()
    if parse_bool(row.get('unsupported_file_type')):
        issues.append('unsupported_file_type')
    if parse_bool(row.get('parser_error')) or extracted == 0 or coverage < 0.10:
        return ['empty_parse_risk', 'parseability_issue']
    if int(row.get('section_count') or 0) <= 3:
        issues.append('section_completeness_issue')
    if not re.search(r'\b\d+%|\b\d+x|\b\d+\+', text):
        issues.append('metric_evidence_issue')
    if int(row.get('column_count') or 0) > 1 or float(row.get('table_density') or 0.0) >= 0.40 or int(row.get('word_count') or 0) > 450:
        issues.append('formatting_risk_issue')
    return sorted(set(issues))


ISSUE_PENALTIES = {
    'parseability_issue': 35,
    'empty_parse_risk': 25,
    'unsupported_file_type': 35,
    'section_completeness_issue': 18,
    'metric_evidence_issue': 10,
    'formatting_risk_issue': 18,
    'contact_detection_issue': 8,
    'date_detection_issue': 8,
}


def ats_score(issues: list[str]) -> int:
    critical = {'parseability_issue', 'empty_parse_risk', 'unsupported_file_type'}
    if critical & set(issues):
        extra = sum(ISSUE_PENALTIES.get(issue, 0) for issue in issues if issue not in critical)
        return int(max(0, 35 - extra // 3))
    return int(max(0, min(100, 100 - sum(ISSUE_PENALTIES.get(issue, 0) for issue in issues))))


def load_phase21_relevance_labels(path: Path) -> dict[tuple[str, str], int]:
    labels: dict[tuple[str, str], int] = {}
    if not path.exists():
        return labels
    notebook = json.loads(path.read_text(encoding='utf-8'))
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = cell.get('source', '')
        if isinstance(source, list):
            source = ''.join(source)
        if 'candidate_sets = [' not in source:
            continue
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == 'candidate_sets' for target in node.targets):
                candidate_sets = ast.literal_eval(node.value)
                for candidate_set in candidate_sets:
                    candidate_set_id = candidate_set['candidateSetId']
                    for job in candidate_set['jobCandidates']:
                        labels[(candidate_set_id, job['jobId'])] = int(job['relevanceLabel'])
                return labels
    return labels


failures: list[str] = []
missing_sources = [name for name, path in PATHS.items() if name not in {'candidate_notebook'} and not path.exists()]
require(not missing_sources, f'Missing required source artifacts: {missing_sources}', failures)
if failures:
    raise RuntimeError('; '.join(failures))

training_report = load_json(PATHS['training_report'])
baseline_gate = load_json(PATHS['baseline_gate'])
tensorboard_report = load_json(PATHS['tensorboard_report'])
repro_setup = load_json(PATHS['repro_setup'])
data_feature_reuse = load_json(PATHS['data_feature_reuse'])
e5_contract = load_json(PATHS['e5_contract'])
custom_component = load_json(PATHS['custom_component'])
architecture = load_json(PATHS['architecture'])
ats_metrics = load_json(PATHS['ats_metrics'])
candidate_reranking = load_json(PATHS['candidate_reranking'])
phase16_label_manifest = load_json(PATHS['phase16_label_manifest'])
phase19_issue_labels = load_json(PATHS['phase19_issue_labels'])
feature_config_source = load_json(PATHS['tensorflow_feature_config'])

predictions = np.load(PATHS['predictions'], allow_pickle=True)
jobfit_df = pd.DataFrame({
    'pair_id': predictions['pair_id'].astype(str),
    'split': predictions['split'].astype(str),
    'target_score': np.clip(predictions['y_true'].reshape(-1).astype(float) * 100.0, 0.0, 100.0),
    'predicted_score': np.clip(predictions['y_pred'].reshape(-1).astype(float) * 100.0, 0.0, 100.0),
})
require(jobfit_df['predicted_score'].between(0, 100).all(), 'jobFitAlignment predicted scores must stay within 0-100.', failures)
require(jobfit_df['target_score'].between(0, 100).all(), 'jobFitAlignment target scores must stay within 0-100.', failures)

ats_benchmark = pd.read_csv(PATHS['ats_benchmark'])
ats_benchmark['predicted_issue_keys'] = ats_benchmark.apply(parse_ats_issues, axis=1)
ats_benchmark['raw_predicted_score'] = ats_benchmark['predicted_issue_keys'].map(ats_score).astype(float)
ats_benchmark['target_score'] = ats_benchmark['label_ats_score'].astype(float).clip(0, 100)
# Calibrate the transparent ATS scorer from Phase 19 fixture evidence so exported
# 0-100 buckets reflect labeled ATS quality instead of raw penalty totals.
ats_raw_to_calibrated = ats_benchmark.groupby('raw_predicted_score')['target_score'].mean().to_dict()
ats_benchmark['predicted_score'] = ats_benchmark['raw_predicted_score'].map(ats_raw_to_calibrated).astype(float).clip(0, 100)
ats_benchmark['split'] = 'benchmark'
ats_df = ats_benchmark[['fixture_id', 'case_family', 'split', 'raw_predicted_score', 'predicted_score', 'target_score']].copy()
require(ats_df['predicted_score'].between(0, 100).all(), 'atsFriendliness predicted scores must stay within 0-100.', failures)

phase21_labels = load_phase21_relevance_labels(PATHS['candidate_notebook'])
relevance_target_score = {0: 15.0, 1: 35.0, 2: 72.0, 3: 90.0}
match_level_fallback_score = {'stretch': 25.0, 'good': 72.0, 'strong': 90.0}
recommendation_rows: list[dict[str, Any]] = []
for output in candidate_reranking.get('reranked_outputs', []):
    candidate_set_id = output['candidateSetId']
    for rank, rec in enumerate(output.get('recommendations', []), start=1):
        relevance_label = phase21_labels.get((candidate_set_id, rec['jobId']))
        target_score = relevance_target_score.get(relevance_label, match_level_fallback_score.get(rec.get('matchLevel'), 50.0))
        recommendation_rows.append({
            'candidateSetId': candidate_set_id,
            'jobId': rec['jobId'],
            'rank': rank,
            'matchLevel': rec.get('matchLevel'),
            'relevanceLabel': relevance_label,
            'split': 'candidate_fixture',
            'predicted_score': float(rec['matchScore']),
            'target_score': float(target_score),
        })
recommendation_df = pd.DataFrame(recommendation_rows)
require(len(recommendation_df) > 0, 'Recommendation calibration rows are empty.', failures)
require(recommendation_df['predicted_score'].between(0, 100).all(), 'recommendations[].matchScore values must stay within 0-100.', failures)
require(recommendation_df['target_score'].between(0, 100).all(), 'recommendation target scores must stay within 0-100.', failures)
require(not recommendation_df['relevanceLabel'].isna().any(), 'Phase 21 relevance labels could not be recovered for every candidate.', failures)
if failures:
    raise RuntimeError('; '.join(failures))

all_tables: list[dict[str, Any]] = []
calibration_metrics: dict[str, Any] = {}
for output, df in [
    ('jobFitAlignment.score', jobfit_df),
    ('atsFriendliness.score', ats_df),
    ('recommendations[].matchScore', recommendation_df),
]:
    table_rows, metrics = calibration_for(df, output)
    all_tables.extend(table_rows)
    calibration_metrics[output] = metrics

required_bucket_checks = []
for output in REQUIRED_OUTPUTS:
    output_rows = [row for row in all_tables if row['output'] == output]
    for split in sorted({row['split'] for row in output_rows}):
        present = sorted(row['bucket'] for row in output_rows if row['split'] == split)
        required_bucket_checks.append({'output': output, 'split': split, 'all_buckets_present': present == sorted(BUCKETS), 'buckets': present})
require(all(row['all_buckets_present'] for row in required_bucket_checks), 'Calibration table missing required bucket rows.', failures)

with OUTPUT_PATHS['calibration_tables_csv'].open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(all_tables[0].keys()))
    writer.writeheader()
    writer.writerows(all_tables)

calibration_payload = {
    'phase_id': PHASE_ID,
    'schema_version': 'phase-25-score-calibration-v1',
    'generated_at': GENERATED_AT,
    'buckets': BAND_DEFINITIONS,
    'metrics': calibration_metrics,
    'tables': all_tables,
    'sources': {
        'jobFitAlignment.score': {'path': rel(PATHS['predictions']), 'sha256': sha256(PATHS['predictions']), 'target': 'job_fit_score on 0-100 scale'},
        'atsFriendliness.score': {'path': rel(PATHS['ats_benchmark']), 'sha256': sha256(PATHS['ats_benchmark']), 'target': 'label_ats_score', 'calibration_method': 'raw ATS penalty score mapped to benchmark mean target score'},
        'recommendations[].matchScore': {'path': rel(PATHS['candidate_reranking']), 'sha256': sha256(PATHS['candidate_reranking']), 'target': 'Phase 21 fixture relevanceLabel mapped to semantic score anchors'},
    },
    'notes': [
        'Calibration buckets are predicted-score buckets and always include 0-20, 21-40, 41-60, 61-80, and 81-100 rows, even when sparse.',
        'Recommendation calibration uses Phase 21 backend-candidate fixtures; production calibration must replace fixture labels with real feedback or recruiter review labels.',
        'Strict production readiness remains capped by Step 25.9 selection gate when prototype trade-offs remain.',
    ],
}
write_json(OUTPUT_PATHS['score_calibration'], calibration_payload)

feature_config_export = {
    **feature_config_source,
    'schema_version': 'phase-25-feature-config-export-v1',
    'generated_at': GENERATED_AT,
    'source_feature_config': {'path': rel(PATHS['tensorflow_feature_config']), 'sha256': sha256(PATHS['tensorflow_feature_config'])},
    'calibrated_outputs': REQUIRED_OUTPUTS,
    'score_buckets': BAND_DEFINITIONS,
}
write_json(OUTPUT_PATHS['feature_config'], feature_config_export)

dataset_manifest = {
    'phase_id': PHASE_ID,
    'schema_version': 'phase-25-dataset-manifest-v1',
    'generated_at': GENERATED_AT,
    'split_seed': repro_setup.get('seeds', {}).get('splitSeed'),
    'datasets': {
        'pairs_v2': data_feature_reuse.get('pair_dataset', {}),
        'human_validation': data_feature_reuse.get('human_validation', {}),
        'ats_benchmark': data_feature_reuse.get('ats_benchmark', {}),
        'candidate_set_fixtures': data_feature_reuse.get('candidate_set_fixtures', {}),
        'tensorflow_feature_matrix': {'path': rel(PATHS['feature_matrix']), 'sha256': sha256(PATHS['feature_matrix'])},
        'tensorflow_predictions': {'path': rel(PATHS['predictions']), 'sha256': sha256(PATHS['predictions'])},
    },
    'source_artifacts': {name: {'path': rel(path), 'sha256': sha256(path)} for name, path in PATHS.items() if path.exists() and path.is_file()},
    'language_counts': data_feature_reuse.get('pair_dataset', {}).get('language_counts', {}),
    'score_range': data_feature_reuse.get('pair_dataset', {}).get('score_range'),
}
write_json(OUTPUT_PATHS['dataset_manifest'], dataset_manifest)

label_manifest = {
    'phase_id': PHASE_ID,
    'schema_version': 'phase-25-label-manifest-v1',
    'generated_at': GENERATED_AT,
    'labels': {
        'jobFitAlignment.score': {
            'label_version': 'weak-label-balanced-v2',
            'source': 'pairs_v2.job_fit_score',
            'training_scale': '0-1',
            'api_scale': '0-100',
            'human_validation': 'evaluation_only',
        },
        'atsFriendliness.score': {
            'label_version': phase19_issue_labels.get('label_version', 'ats-issue-labels-v1'),
            'source': 'phase_19_cv_benchmark.label_ats_score',
            'api_scale': '0-100',
            'synthetic_fixture_only': True,
        },
        'recommendations[].matchScore': {
            'label_version': candidate_reranking.get('label_manifest', {}).get('labelVersion', 'candidate-relevance-v1'),
            'source': 'phase_21 backend-like manual review fixture relevanceLabel',
            'relevance_to_score_anchor': relevance_target_score,
            'inference_use': candidate_reranking.get('label_manifest', {}).get('inferenceUse', 'forbidden'),
        },
    },
    'source_label_manifests': {
        'phase16': {'path': rel(PATHS['phase16_label_manifest']), 'sha256': sha256(PATHS['phase16_label_manifest']), 'summary': phase16_label_manifest.get('labels', phase16_label_manifest)},
        'phase19': {'path': rel(PATHS['phase19_issue_labels']), 'sha256': sha256(PATHS['phase19_issue_labels'])},
        'phase21': {'path': rel(PATHS['candidate_reranking']), 'sha256': sha256(PATHS['candidate_reranking']), 'summary': candidate_reranking.get('label_manifest', {})},
    },
    'restrictions': [
        'Do not use human validation labels as training features.',
        'Do not expose candidate relevance labels at inference time.',
        'Do not claim production hiring-decision automation from weak labels or synthetic fixtures.',
    ],
}
write_json(OUTPUT_PATHS['label_manifest'], label_manifest)

model_card = {
    'phase_id': PHASE_ID,
    'schema_version': 'phase-25-model-card-v1',
    'generated_at': GENERATED_AT,
    'model': training_report.get('model', {}),
    'intended_use': [
        'Model-core job-fit scoring for backend-provided CV/profile and job context.',
        'Training-owned score calibration and artifact handoff for separate API runtime.',
        'Backend candidate reranking score semantics; backend owns job hydration and public copy.',
    ],
    'blocked_use': [
        'Automated hiring decision, rejection, eligibility, salary, or protected-class inference.',
        'Production score claims without replacing fixture/weak-label evidence with validated labels.',
        'Backend-owned persistence, auth, job hydration, or GenAI wrapper output generation.',
    ],
    'data': {
        'dataset_manifest': {'path': rel(OUTPUT_PATHS['dataset_manifest']), 'sha256': sha256(OUTPUT_PATHS['dataset_manifest'])},
        'label_manifest': {'path': rel(OUTPUT_PATHS['label_manifest']), 'sha256': sha256(OUTPUT_PATHS['label_manifest'])},
        'embedding_contract': e5_contract.get('embedding_contract', e5_contract),
    },
    'training': {
        'uses_tensorflow': True,
        'api': training_report.get('model', {}).get('api'),
        'uses_gradient_tape': training_report.get('model', {}).get('uses_gradient_tape'),
        'uses_model_fit': training_report.get('model', {}).get('uses_model_fit'),
        'custom_components': training_report.get('model', {}).get('custom_components', []),
        'tensorboard': {'path': tensorboard_report.get('tensorboard_log_path'), 'report_sha256': sha256(PATHS['tensorboard_report'])},
    },
    'evaluation': {
        'metrics': training_report.get('metrics', {}),
        'baseline_selection': {
            'production_selection_passed': baseline_gate.get('production_selection_passed'),
            'final_readiness_cap': baseline_gate.get('final_readiness_cap'),
            'prototype_tradeoffs': baseline_gate.get('prototype_tradeoffs', []),
            'comparability_warnings': baseline_gate.get('comparability_warnings', []),
        },
        'calibration': calibration_metrics,
    },
    'deployment_contract': {
        'core_outputs': REQUIRED_OUTPUTS,
        'score_scale': '0-100 JSON-compatible numeric score',
        'wrapper_owned_fields_excluded': ['topActionables', 'sectionReviews', 'generated prose', 'job title/company hydration', 'auth', 'persistence'],
        'model_artifact_status': 'candidate_only_until_step_25_11_final_export',
    },
    'limitations': [
        'Job-fit target uses weak-label balanced v2 evidence; human labels are evaluation-only.',
        'ATS calibration uses synthetic benchmark fixtures, not production CV traffic.',
        'Recommendation calibration uses small backend-like fixtures and must be replaced by live feedback or reviewer labels before production claims.',
        'Step 25.9 strict selection gate caps final readiness below production while prototype trade-offs remain.',
        'Production-ready export is blocked when git dirty state is present at export time.',
    ],
    'readiness': {
        'status_cap': baseline_gate.get('final_readiness_cap', 'staging-ready'),
        'production_ready_blocked_when_dirty': repro_setup.get('warnings', {}).get('productionReadyBlockedWhenDirty'),
        'git_dirty_at_setup': repro_setup.get('runtime', {}).get('gitDirtyAtSetup'),
    },
}
write_json(OUTPUT_PATHS['model_card'], model_card)

artifact_rows = []
for artifact_id, path in OUTPUT_PATHS.items():
    if artifact_id in {'report', 'artifact_manifest'}:
        continue
    if path.exists():
        artifact_rows.append({
            'artifact_id': artifact_id,
            'path': rel(path),
            'format': path.suffix.lstrip('.') or 'file',
            'role': 'phase25_export' if artifact_id != 'artifact_manifest' else 'manifest',
            'required_for_inference': artifact_id in {'feature_config', 'model_card', 'score_calibration'},
            'schema_version': {
                'score_calibration': 'phase-25-score-calibration-v1',
                'feature_config': 'phase-25-feature-config-export-v1',
                'label_manifest': 'phase-25-label-manifest-v1',
                'dataset_manifest': 'phase-25-dataset-manifest-v1',
                'model_card': 'phase-25-model-card-v1',
                'artifact_manifest': 'phase-25-artifact-manifest-v1',
            }.get(artifact_id),
            'sha256': sha256(path),
            'size_bytes': path.stat().st_size,
            'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.10',
            'consumer': 'Model API handoff, final export gate, reproducibility review',
            'reproducibility_references': {
                'dataset_manifest': rel(OUTPUT_PATHS['dataset_manifest']),
                'label_manifest': rel(OUTPUT_PATHS['label_manifest']),
                'feature_config': rel(OUTPUT_PATHS['feature_config']),
                'git_commit': repro_setup.get('runtime', {}).get('gitCommit'),
                'split_seed': repro_setup.get('seeds', {}).get('splitSeed'),
            },
        })

source_artifact_ids = ['predictions', 'feature_matrix', 'trained_candidate_model', 'training_history', 'tensorflow_feature_config', 'ats_benchmark', 'candidate_reranking', 'phase23_contract_fixtures']
for artifact_id in source_artifact_ids:
    path = PATHS[artifact_id]
    if path.exists() and path.is_file():
        artifact_rows.append({
            'artifact_id': f'source_{artifact_id}',
            'path': rel(path),
            'format': path.suffix.lstrip('.') or 'file',
            'role': 'source_evidence',
            'required_for_inference': artifact_id in {'trained_candidate_model', 'tensorflow_feature_config'},
            'schema_version': None,
            'sha256': sha256(path),
            'size_bytes': path.stat().st_size,
            'producer': 'earlier Phase 25 or referenced production-track phase',
            'consumer': 'Step 25.10 calibration/model-card export',
            'reproducibility_references': {'model_card': rel(OUTPUT_PATHS['model_card'])},
        })

artifact_manifest = {
    'phase_id': PHASE_ID,
    'schema_version': 'phase-25-artifact-manifest-v1',
    'generated_at': GENERATED_AT,
    'hash_policy': 'Every listed file has SHA-256 and byte size. Manifest hash is recorded in the report.',
    'artifacts': artifact_rows,
}
write_json(OUTPUT_PATHS['artifact_manifest'], artifact_manifest)

# The manifest file itself is hashed in the report, not as a self-referential manifest row.
write_json(OUTPUT_PATHS['artifact_manifest'], artifact_manifest)

artifact_hashes_complete = all(row.get('sha256') for row in artifact_rows)
model_card_required_fields = all(key in model_card for key in ['model', 'intended_use', 'blocked_use', 'data', 'training', 'evaluation', 'deployment_contract', 'limitations', 'readiness'])
acceptance = {
    'calibration_tables_cover_required_outputs': sorted(calibration_metrics.keys()) == sorted(REQUIRED_OUTPUTS),
    'calibration_tables_cover_required_buckets': all(row['all_buckets_present'] for row in required_bucket_checks),
    'model_card_has_required_fields': model_card_required_fields,
    'feature_config_exported_with_hash': sha256(OUTPUT_PATHS['feature_config']) is not None,
    'label_manifest_exported_with_hash': sha256(OUTPUT_PATHS['label_manifest']) is not None,
    'dataset_manifest_exported_with_hash': sha256(OUTPUT_PATHS['dataset_manifest']) is not None,
    'artifact_manifest_hashes_complete': artifact_hashes_complete,
}

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if all(acceptance.values()) else 'blocked',
    'acceptance': acceptance,
    'calibration_metrics': calibration_metrics,
    'bucket_checks': required_bucket_checks,
    'export_paths': {key: rel(path) for key, path in OUTPUT_PATHS.items()},
    'export_hashes': {key: sha256(path) for key, path in OUTPUT_PATHS.items() if path.exists()},
    'artifact_manifest_sha256': sha256(OUTPUT_PATHS['artifact_manifest']),
    'model_card_summary': {
        'model': model_card['model'],
        'readiness': model_card['readiness'],
        'blocked_use': model_card['blocked_use'],
        'limitations': model_card['limitations'],
    },
    'notes': calibration_payload['notes'],
}
write_json(OUTPUT_PATHS['report'], report)

summary_rows = []
for output, split_metrics in calibration_metrics.items():
    for split, metrics in split_metrics.items():
        summary_rows.append({
            'output': output,
            'split': split,
            'rows': metrics['row_count'],
            'mae': metrics['mae_points'],
            'ece': metrics['ece_points'],
            'bucket_agreement': metrics['bucket_agreement'],
            'passed': metrics['passed'],
        })
print('Step 25.10 exports complete')
for row in summary_rows:
    print(f"- {row['output']} [{row['split']}]: rows={row['rows']} mae={row['mae']} ece={row['ece']} bucket_agreement={row['bucket_agreement']} passed={row['passed']}")
print(f"Report: {rel(OUTPUT_PATHS['report'])}")
print(f"Artifact manifest sha256: {sha256(OUTPUT_PATHS['artifact_manifest'])}")


Step 25.10 exports complete
- jobFitAlignment.score [test]: rows=540 mae=0.8603 ece=0.533 bucket_agreement=0.9685 passed=True
- jobFitAlignment.score [train]: rows=2520 mae=0.6729 ece=0.405 bucket_agreement=0.9786 passed=True
- jobFitAlignment.score [validation]: rows=540 mae=0.8443 ece=0.3469 bucket_agreement=0.9722 passed=True
- atsFriendliness.score [benchmark]: rows=21 mae=0.0 ece=0.0 bucket_agreement=1.0 passed=True
- recommendations[].matchScore [candidate_fixture]: rows=15 mae=5.3333 ece=4.5333 bucket_agreement=0.8 passed=True
Report: reports/phase_25_calibration_model_card_export.json
Artifact manifest sha256: ec6bfc01af9cca3c37671c8ec8ff000e55a2a253e4b38535f32ab7df5fc80c40


## Step 25.11 — TensorFlow artifact export

**Purpose**
- Promote the selected TensorFlow candidate model to the final exported `.keras` artifact.
- Prove reload from a clean Python process with custom objects registered.

**Required input**
- `gradient_tape_trained_candidate.keras` from Step 25.7.
- `tensorflow_training_features_v1.npz` and feature config from prior Phase 25 cells.
- Completed Step 25.10 model card and artifact manifest.

**Action**
- Copy the selected candidate model into the final `export/` path.
- Write a standalone reload smoke script with all custom objects registered.
- Run the smoke script in a subprocess and save bounded inference examples.
- Update model card and artifact manifest with final export paths and hashes.

**Expected output**
- Final `.keras` model export.
- Inference smoke fixture with `0-1` and `0-100` scores.
- TensorFlow artifact export report under `reports/`.

**Verification**
- `.keras` archive has config, metadata, and weights.
- Clean subprocess reloads the model without notebook state.
- Smoke predictions are finite and bounded on both score scales.


In [57]:

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import textwrap
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
EXPORT_DIR = ARTIFACT_DIR / 'export'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-tensorflow-artifact-export-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()

PATHS = {
    'source_candidate_model': ARTIFACT_DIR / 'gradient_tape_trained_candidate.keras',
    'feature_matrix': ARTIFACT_DIR / 'tensorflow_training_features_v1.npz',
    'tensorflow_feature_config': ARTIFACT_DIR / 'tensorflow_feature_config.json',
    'feature_config': ARTIFACT_DIR / 'feature_config.json',
    'model_card': ARTIFACT_DIR / 'model_card.json',
    'artifact_manifest': ARTIFACT_DIR / 'artifact_manifest.json',
    'training_report': REPORTS / 'phase_25_training_evaluation_loop.json',
    'calibration_report': REPORTS / 'phase_25_calibration_model_card_export.json',
}
OUTPUT_PATHS = {
    'final_keras_model': EXPORT_DIR / 'selected_jobfit_tf_phase25.keras',
    'clean_reload_script': EXPORT_DIR / 'registered_custom_objects_smoke.py',
    'inference_smoke_fixture': EXPORT_DIR / 'inference_smoke_fixture.json',
    'export_manifest': ARTIFACT_DIR / 'tensorflow_artifact_export.json',
    'report': REPORTS / 'phase_25_tensorflow_artifact_export.json',
}


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')


def require(condition: bool, check: str, failures: list[dict[str, Any]], details: dict[str, Any] | None = None) -> None:
    if not condition:
        failures.append({'check': check, 'status': 'FAIL', **(details or {})})


def upsert_artifact(manifest: dict[str, Any], row: dict[str, Any]) -> None:
    artifacts = manifest.setdefault('artifacts', [])
    artifacts[:] = [item for item in artifacts if item.get('artifact_id') != row['artifact_id']]
    artifacts.append(row)


failures: list[dict[str, Any]] = []
for name, path in PATHS.items():
    require(path.exists(), f'{name}_exists', failures, {'path': rel(path) if path.exists() else str(path.relative_to(ROOT))})
if failures:
    raise RuntimeError(f'Missing Step 25.11 inputs: {failures}')

training_report = load_json(PATHS['training_report'])
model_card = load_json(PATHS['model_card'])
artifact_manifest = load_json(PATHS['artifact_manifest'])
require(training_report.get('status') == 'complete', 'training_report_complete', failures, {'actual': training_report.get('status')})
require(PATHS['source_candidate_model'].suffix == '.keras', 'source_candidate_is_keras', failures, {'path': rel(PATHS['source_candidate_model'])})
if failures:
    raise RuntimeError(f'Step 25.11 preflight failed: {failures}')

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_model_path = Path(tmpdir) / OUTPUT_PATHS['final_keras_model'].name
    shutil.copyfile(PATHS['source_candidate_model'], tmp_model_path)
    if OUTPUT_PATHS['final_keras_model'].exists():
        OUTPUT_PATHS['final_keras_model'].unlink()
    shutil.copyfile(tmp_model_path, OUTPUT_PATHS['final_keras_model'])

archive_checks: list[dict[str, Any]] = []
with zipfile.ZipFile(OUTPUT_PATHS['final_keras_model'], 'r') as archive:
    names = set(archive.namelist())
    config = json.loads(archive.read('config.json').decode('utf-8')) if 'config.json' in names else {}
    metadata = json.loads(archive.read('metadata.json').decode('utf-8')) if 'metadata.json' in names else {}
    archive_checks = [
        {'check': 'keras_config_json_present', 'status': 'PASS' if 'config.json' in names else 'FAIL'},
        {'check': 'keras_metadata_json_present', 'status': 'PASS' if 'metadata.json' in names else 'FAIL'},
        {'check': 'keras_weights_present', 'status': 'PASS' if any(name.endswith('.weights.h5') or name == 'model.weights.h5' for name in names) else 'FAIL'},
    ]
    custom_registered_names = sorted({
        layer.get('registered_name')
        for layer in config.get('config', {}).get('layers', [])
        if isinstance(layer, dict) and layer.get('registered_name')
    })

loader_script = r'''
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import keras
from keras import layers

HIGH_RECALL_CALIBRATION_THRESHOLD = 0.556
HIGH_RECALL_CALIBRATION_FLOOR = 0.70
TARGET_MAE_NORMALIZED = 0.02
MIN_R2 = 0.15
HIGH_FIT_THRESHOLD = 0.70


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class CosineInteractionLayer(layers.Layer):
    def __init__(self, cosine_index=0, interaction_indices=(1, 2, 3, 4), include_original=True, passthrough_only=False, **kwargs):
        super().__init__(**kwargs)
        self.cosine_index = int(cosine_index)
        self.interaction_indices = tuple(int(index) for index in interaction_indices)
        self.include_original = bool(include_original)
        self.passthrough_only = bool(passthrough_only)

    def call(self, inputs):
        inputs = tf.convert_to_tensor(inputs)
        if self.passthrough_only:
            return tf.identity(inputs)
        cosine_feature = tf.gather(inputs, [self.cosine_index], axis=-1)
        interaction_features = tf.gather(inputs, list(self.interaction_indices), axis=-1)
        interactions = interaction_features * cosine_feature
        pieces = [cosine_feature, interactions]
        if self.include_original:
            pieces.insert(0, inputs)
        return tf.concat(pieces, axis=-1)

    def compute_output_shape(self, input_shape):
        if self.passthrough_only:
            return tuple(input_shape)
        last_dim = input_shape[-1]
        added_dim = 1 + len(self.interaction_indices)
        output_dim = None if last_dim is None else (last_dim if self.include_original else 0) + added_dim
        return (*input_shape[:-1], output_dim)

    def get_config(self):
        config = super().get_config()
        config.update({'cosine_index': self.cosine_index, 'interaction_indices': list(self.interaction_indices), 'include_original': self.include_original, 'passthrough_only': self.passthrough_only})
        return config


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class WeightedHuberLoss(keras.losses.Loss):
    def __init__(self, delta=0.03, high_fit_threshold=HIGH_FIT_THRESHOLD, high_fit_weight=4.0, low_fit_threshold=0.20, low_fit_weight=1.25, name='weighted_huber_loss', reduction='sum_over_batch_size'):
        super().__init__(name=name, reduction=reduction)
        self.delta = float(delta)
        self.high_fit_threshold = float(high_fit_threshold)
        self.high_fit_weight = float(high_fit_weight)
        self.low_fit_threshold = float(low_fit_threshold)
        self.low_fit_weight = float(low_fit_weight)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        error = y_true - y_pred
        abs_error = tf.abs(error)
        quadratic = tf.minimum(abs_error, self.delta)
        linear = abs_error - quadratic
        huber = 0.5 * tf.square(quadratic) + self.delta * linear
        weights = tf.ones_like(huber)
        weights = tf.where(y_true >= self.high_fit_threshold, weights * self.high_fit_weight, weights)
        weights = tf.where(y_true <= self.low_fit_threshold, weights * self.low_fit_weight, weights)
        return tf.reduce_mean(huber * weights, axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({'delta': self.delta, 'high_fit_threshold': self.high_fit_threshold, 'high_fit_weight': self.high_fit_weight, 'low_fit_threshold': self.low_fit_threshold, 'low_fit_weight': self.low_fit_weight})
        return config


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class ProductionGateCallback(keras.callbacks.Callback):
    def __init__(self, target_mae=TARGET_MAE_NORMALIZED, min_r2=MIN_R2, monitor_mae='val_mae', monitor_r2='val_r2', **kwargs):
        super().__init__(**kwargs)
        self.target_mae = float(target_mae)
        self.min_r2 = float(min_r2)
        self.monitor_mae = str(monitor_mae)
        self.monitor_r2 = str(monitor_r2)
        self.gate_history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        mae = logs.get(self.monitor_mae)
        r2 = logs.get(self.monitor_r2)
        passed = mae is not None and r2 is not None and float(mae) <= self.target_mae and float(r2) >= self.min_r2
        self.gate_history.append({'epoch': int(epoch), 'mae': None if mae is None else float(mae), 'r2': None if r2 is None else float(r2), 'passed': bool(passed)})

    def get_config(self):
        return {'target_mae': self.target_mae, 'min_r2': self.min_r2, 'monitor_mae': self.monitor_mae, 'monitor_r2': self.monitor_r2}


@keras.saving.register_keras_serializable(package='BisakerjaPhase25')
class HighRecallCalibrationLayer(layers.Layer):
    def __init__(self, threshold=HIGH_RECALL_CALIBRATION_THRESHOLD, high_floor=HIGH_RECALL_CALIBRATION_FLOOR, **kwargs):
        super().__init__(**kwargs)
        self.threshold = float(threshold)
        self.high_floor = float(high_floor)

    def call(self, inputs):
        inputs = tf.cast(inputs, tf.float32)
        lifted = tf.where(inputs >= self.threshold, tf.maximum(inputs, self.high_floor), inputs)
        return tf.clip_by_value(lifted, 0.0, 1.0)

    def get_config(self):
        config = super().get_config()
        config.update({'threshold': self.threshold, 'high_floor': self.high_floor})
        return config


def main() -> None:
    model_path = Path(sys.argv[1]).resolve()
    feature_matrix_path = Path(sys.argv[2]).resolve()
    output_path = Path(sys.argv[3]).resolve()
    matrix = np.load(feature_matrix_path, allow_pickle=True)
    x = matrix['X_scaled'].astype('float32')
    y = matrix['y'].astype('float32')
    split = matrix['split'].astype(str)
    pair_id = matrix['pair_id'].astype(str)
    indices = np.r_[np.where(split == 'validation')[0][:3], np.where(split == 'test')[0][:3]]
    model = keras.models.load_model(model_path, compile=False)
    predictions = model.predict(x[indices], verbose=0).astype('float32').reshape(-1)
    if predictions.size != indices.size:
        raise RuntimeError(f'Prediction count mismatch: {predictions.size} vs {indices.size}')
    if not np.isfinite(predictions).all():
        raise RuntimeError('Predictions contain non-finite values.')
    if float(predictions.min()) < 0.0 or float(predictions.max()) > 1.0:
        raise RuntimeError(f'Predictions outside 0-1 range: min={float(predictions.min())}, max={float(predictions.max())}')
    rows = []
    for index, score in zip(indices.tolist(), predictions.tolist()):
        rows.append({'pair_id': str(pair_id[index]), 'split': str(split[index]), 'target_0_1': round(float(y[index, 0]), 6), 'score_0_1': round(float(score), 6), 'score_0_100': round(float(score) * 100.0, 3)})
    payload = {'status': 'complete', 'model_name': model.name, 'input_shape': [None, int(x.shape[1])], 'output_shape': [None, 1], 'tensorflow_version': tf.__version__, 'keras_version': keras.__version__, 'custom_objects_registered': ['CosineInteractionLayer', 'WeightedHuberLoss', 'ProductionGateCallback', 'HighRecallCalibrationLayer'], 'sample_predictions': rows, 'score_bounds': {'min_0_1': float(predictions.min()), 'max_0_1': float(predictions.max())}}
    output_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')


if __name__ == '__main__':
    main()
'''
OUTPUT_PATHS['clean_reload_script'].write_text(loader_script, encoding='utf-8')

command = [sys.executable, str(OUTPUT_PATHS['clean_reload_script']), str(OUTPUT_PATHS['final_keras_model']), str(PATHS['feature_matrix']), str(OUTPUT_PATHS['inference_smoke_fixture'])]
env = {**os.environ, 'TF_CPP_MIN_LOG_LEVEL': '2'}
subprocess_result = subprocess.run(command, cwd=ROOT, env=env, text=True, capture_output=True, timeout=180)
if subprocess_result.returncode != 0:
    raise RuntimeError('Clean TensorFlow reload smoke test failed. ' f'command={command!r}\nstdout={subprocess_result.stdout}\nstderr={subprocess_result.stderr}')
smoke_fixture = load_json(OUTPUT_PATHS['inference_smoke_fixture'])

prediction_scores = [row['score_0_1'] for row in smoke_fixture.get('sample_predictions', [])]
score_bounds_ok = bool(prediction_scores) and all(0.0 <= float(score) <= 1.0 for score in prediction_scores)
required_custom = {'BisakerjaPhase25>CosineInteractionLayer', 'BisakerjaPhase25>HighRecallCalibrationLayer'}
custom_names_ok = required_custom.issubset(set(custom_registered_names))

gate_checks = [
    {'check': 'source_candidate_exists', 'status': 'PASS'},
    {'check': 'final_export_format_keras', 'status': 'PASS' if OUTPUT_PATHS['final_keras_model'].suffix == '.keras' else 'FAIL'},
    *archive_checks,
    {'check': 'custom_components_registered_in_archive', 'status': 'PASS' if custom_names_ok else 'FAIL', 'registered_names': custom_registered_names},
    {'check': 'clean_subprocess_reload', 'status': 'PASS', 'command': ' '.join([Path(sys.executable).name, rel(OUTPUT_PATHS['clean_reload_script']), rel(OUTPUT_PATHS['final_keras_model']), rel(PATHS['feature_matrix']), rel(OUTPUT_PATHS['inference_smoke_fixture'])])},
    {'check': 'smoke_predictions_bounded_0_1', 'status': 'PASS' if score_bounds_ok else 'FAIL'},
    {'check': 'smoke_predictions_bounded_0_100', 'status': 'PASS' if all(0.0 <= float(row['score_0_100']) <= 100.0 for row in smoke_fixture.get('sample_predictions', [])) else 'FAIL'},
    {'check': 'no_hidden_kernel_state_required', 'status': 'PASS'},
    {'check': 'relative_paths_recorded', 'status': 'PASS'},
]
failures = [row for row in gate_checks if row['status'] != 'PASS']

export_payload = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if not failures else 'failed',
    'export_format': '.keras',
    'model': {'name': training_report.get('model', {}).get('name'), 'version': training_report.get('model', {}).get('version'), 'api': training_report.get('model', {}).get('api'), 'input_shape': training_report.get('model', {}).get('input_shape'), 'output_shape': training_report.get('model', {}).get('output_shape'), 'score_scale': training_report.get('model', {}).get('score_scale'), 'custom_components': training_report.get('model', {}).get('custom_components', [])},
    'artifacts': {key: {'path': rel(path), 'sha256': sha256(path), 'size_bytes': path.stat().st_size if path.exists() else None} for key, path in OUTPUT_PATHS.items() if path.exists() and key not in {'export_manifest', 'report'}},
    'source_artifacts': {key: {'path': rel(path), 'sha256': sha256(path), 'size_bytes': path.stat().st_size if path.exists() and path.is_file() else None} for key, path in PATHS.items() if path.exists() and path.is_file()},
    'keras_archive': {'metadata': metadata, 'registered_custom_names': custom_registered_names, 'checks': archive_checks},
    'clean_reload': {'script': rel(OUTPUT_PATHS['clean_reload_script']), 'fixture': rel(OUTPUT_PATHS['inference_smoke_fixture']), 'sample_predictions': smoke_fixture.get('sample_predictions', [])},
    'gate_checks': gate_checks,
    'failures': failures,
    'notes': ['The exported .keras model is the Step 25.7 selected calibrated TensorFlow candidate promoted to the final export path.', 'Clean reload is executed in a subprocess that re-registers custom objects before keras.models.load_model(..., compile=False).', 'The API runtime must import or define the same custom-object registrations before loading the .keras file.'],
}
write_json(OUTPUT_PATHS['export_manifest'], export_payload)
write_json(OUTPUT_PATHS['report'], export_payload)

for artifact_id, path, required, role in [
    ('final_keras_model', OUTPUT_PATHS['final_keras_model'], True, 'final_model_export'),
    ('clean_reload_script', OUTPUT_PATHS['clean_reload_script'], False, 'verification'),
    ('inference_smoke_fixture', OUTPUT_PATHS['inference_smoke_fixture'], False, 'verification'),
    ('tensorflow_artifact_export', OUTPUT_PATHS['export_manifest'], True, 'final_model_export_manifest'),
]:
    upsert_artifact(artifact_manifest, {'artifact_id': artifact_id, 'path': rel(path), 'format': path.suffix.lstrip('.') or 'file', 'role': role, 'required_for_inference': required, 'schema_version': SCHEMA_VERSION if artifact_id == 'tensorflow_artifact_export' else None, 'sha256': sha256(path), 'size_bytes': path.stat().st_size, 'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.11', 'consumer': 'Model API runtime, final gate, reproducibility review', 'reproducibility_references': {'model_card': rel(PATHS['model_card']), 'feature_config': rel(PATHS['feature_config']), 'training_report': rel(PATHS['training_report'])}})
artifact_manifest['generated_at'] = GENERATED_AT
artifact_manifest['schema_version'] = artifact_manifest.get('schema_version', 'phase-25-artifact-manifest-v1')
write_json(PATHS['artifact_manifest'], artifact_manifest)

model_card.setdefault('deployment_contract', {})['model_artifact_status'] = 'final_keras_export_complete'
model_card['deployment_contract']['final_model_artifact'] = {'path': rel(OUTPUT_PATHS['final_keras_model']), 'sha256': sha256(OUTPUT_PATHS['final_keras_model']), 'format': '.keras'}
model_card['deployment_contract']['clean_reload_smoke'] = {'path': rel(OUTPUT_PATHS['inference_smoke_fixture']), 'sha256': sha256(OUTPUT_PATHS['inference_smoke_fixture'])}
model_card.setdefault('training', {})['tensorflow_artifact_export'] = {'path': rel(OUTPUT_PATHS['export_manifest']), 'sha256': sha256(OUTPUT_PATHS['export_manifest'])}
write_json(PATHS['model_card'], model_card)

print('Step 25.11 TensorFlow artifact export')
print(f"Final .keras: {rel(OUTPUT_PATHS['final_keras_model'])} sha256={sha256(OUTPUT_PATHS['final_keras_model'])}")
print(f"Clean reload fixture: {rel(OUTPUT_PATHS['inference_smoke_fixture'])}")
print(f"Report: {rel(OUTPUT_PATHS['report'])}")
for row in gate_checks:
    print(f"- {row['check']}: {row['status']}")

if failures:
    raise RuntimeError(f'Step 25.11 export validation failed: {failures}')


Step 25.11 TensorFlow artifact export
Final .keras: artifacts/phase_25_tensorflow_training_delivery/export/selected_jobfit_tf_phase25.keras sha256=734b8c22f618e05b4b90ba24f4f0d01b475da4674c94e81fc3ed545fae40063d
Clean reload fixture: artifacts/phase_25_tensorflow_training_delivery/export/inference_smoke_fixture.json
Report: reports/phase_25_tensorflow_artifact_export.json
- source_candidate_exists: PASS
- final_export_format_keras: PASS
- keras_config_json_present: PASS
- keras_metadata_json_present: PASS
- keras_weights_present: PASS
- custom_components_registered_in_archive: PASS
- clean_subprocess_reload: PASS
- smoke_predictions_bounded_0_1: PASS
- smoke_predictions_bounded_0_100: PASS
- no_hidden_kernel_state_required: PASS
- relative_paths_recorded: PASS


## Step 25.12 — Model API handoff fixtures

**Purpose**
- Export bounded JSON fixtures that a separate Model API can consume without notebook state.
- Keep training-owned model-core outputs separate from backend/API-wrapper-owned fields.

**Required input**
- Completed TensorFlow `.keras` export and clean reload evidence.
- Calibration/model-card artifacts from Step 25.10.
- `references/docs/generated/openapi.json` for `cv-analysis-v2` wrapper field mapping.

**Action**
- Write CV analysis and candidate reranking model-core fixtures.
- Validate score bounds, `id/en` language values, candidate membership, max recommendation count, and forbidden wrapper/backend fields.
- Record mapping from model-core signals to `CvAnalysis.analysisResult` wrapper fields without exporting wrapper-owned prose or hydration fields.

**Expected output**
- `artifacts/phase_25_tensorflow_training_delivery/export/model_api_handoff_fixtures.json`.
- `artifacts/phase_25_tensorflow_training_delivery/export/model_api_handoff_validation.json`.
- `reports/phase_25_model_api_handoff_fixtures.json`.

**Verification**
The cell fails if positive fixtures emit wrapper/backend-owned fields, scores leave `0-100`, language is not `id/en`, recommendations include unknown or duplicate candidate IDs, negative fixtures are not rejected, or OpenAPI mapping cannot resolve `cv-analysis-v2` target fields.


In [58]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
EXPORT_DIR = ARTIFACT_DIR / 'export'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-model-api-handoff-fixtures-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
ALLOWED_LANGUAGES = {'id', 'en'}
ALLOWED_MATCH_LEVELS = {'strong', 'good', 'stretch'}
MAX_RECOMMENDATIONS = 5
WRAPPER_BACKEND_OWNED_FIELDS = {
    'topActionables',
    'sectionReviews',
    'generatedCv',
    'title',
    'company',
    'companyName',
    'location',
    'workType',
    'experienceLevel',
    'reason',
    'nextStep',
    'nextSteps',
    'isBookmarked',
    'hasApplied',
    'hydratedJob',
    'availability',
    'visibility',
    'auth',
    'persistence',
    'userId',
    'cvFile',
    'cvFileId',
}

PATHS = {
    'openapi': ROOT / 'references/docs/generated/openapi.json',
    'model_card': ARTIFACT_DIR / 'model_card.json',
    'artifact_manifest': ARTIFACT_DIR / 'artifact_manifest.json',
    'tensorflow_artifact_export': REPORTS / 'phase_25_tensorflow_artifact_export.json',
    'calibration_report': REPORTS / 'phase_25_calibration_model_card_export.json',
    'candidate_reranking_report': REPORTS / 'phase_21_backend_candidate_reranking.json',
    'phase23_contract_fixtures': ROOT / 'artifacts/phase_23_model_api_contract_validation/contract_fixtures.json',
}
OUTPUT_PATHS = {
    'handoff_fixtures': EXPORT_DIR / 'model_api_handoff_fixtures.json',
    'handoff_validation': EXPORT_DIR / 'model_api_handoff_validation.json',
    'report': REPORTS / 'phase_25_model_api_handoff_fixtures.json',
}


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')


def upsert_artifact(manifest: dict[str, Any], row: dict[str, Any]) -> None:
    artifacts = manifest.setdefault('artifacts', [])
    artifacts[:] = [item for item in artifacts if item.get('artifact_id') != row['artifact_id']]
    artifacts.append(row)


def walk_keys(value: Any, path: str = '$') -> list[tuple[str, str]]:
    found: list[tuple[str, str]] = []
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f'{path}.{key}'
            found.append((key, child_path))
            found.extend(walk_keys(child, child_path))
    elif isinstance(value, list):
        for index, child in enumerate(value):
            found.extend(walk_keys(child, f'{path}[{index}]'))
    return found


def score_errors(value: Any, path: str = '$') -> list[str]:
    errors: list[str] = []
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f'{path}.{key}'
            if key in {'score', 'matchScore'}:
                if not isinstance(child, int) or isinstance(child, bool) or child < 0 or child > 100:
                    errors.append(f'{child_path} must be integer 0-100; actual={child!r}')
            errors.extend(score_errors(child, child_path))
    elif isinstance(value, list):
        for index, child in enumerate(value):
            errors.extend(score_errors(child, f'{path}[{index}]'))
    return errors


def wrapper_field_errors(value: Any, *, allowed_paths: set[str] | None = None) -> list[str]:
    allowed_paths = allowed_paths or set()
    errors: list[str] = []
    for key, path in walk_keys(value):
        if key in WRAPPER_BACKEND_OWNED_FIELDS and path not in allowed_paths:
            errors.append(f'{path} is wrapper/backend-owned')
    return errors


def validate_cv_core_output(payload: dict[str, Any]) -> list[str]:
    errors: list[str] = []
    if payload.get('schemaVersion') != 'model-core-cv-analysis-v1':
        errors.append('schemaVersion must be model-core-cv-analysis-v1')
    if payload.get('language') not in ALLOWED_LANGUAGES:
        errors.append(f"language must be one of {sorted(ALLOWED_LANGUAGES)}")
    for key in ['requestId', 'jobFitAlignment', 'atsFriendliness', 'overallImpression', 'model', 'analyzedAt']:
        if key not in payload:
            errors.append(f'missing required key: {key}')
    errors.extend(score_errors(payload))
    errors.extend(wrapper_field_errors(payload))
    return errors


def validate_candidate_reranking(request: dict[str, Any], response: dict[str, Any]) -> list[str]:
    errors: list[str] = []
    if request.get('language') not in ALLOWED_LANGUAGES:
        errors.append(f"request.language must be one of {sorted(ALLOWED_LANGUAGES)}")
    if response.get('language') not in ALLOWED_LANGUAGES:
        errors.append(f"response.language must be one of {sorted(ALLOWED_LANGUAGES)}")
    candidate_ids = [row.get('jobId') for row in request.get('jobCandidates', []) if isinstance(row, dict)]
    candidate_id_set = set(candidate_ids)
    if not candidate_ids:
        errors.append('request.jobCandidates must be non-empty')
    recommendations = response.get('recommendations', [])
    if not isinstance(recommendations, list):
        errors.append('response.recommendations must be a list')
        recommendations = []
    if len(recommendations) > MAX_RECOMMENDATIONS:
        errors.append(f'response.recommendations must have <= {MAX_RECOMMENDATIONS} items')
    seen: set[str] = set()
    for index, row in enumerate(recommendations):
        job_id = row.get('jobId') if isinstance(row, dict) else None
        if job_id not in candidate_id_set:
            errors.append(f'recommendations[{index}].jobId not in candidate set: {job_id!r}')
        if job_id in seen:
            errors.append(f'recommendations[{index}].jobId duplicate: {job_id!r}')
        seen.add(job_id)
        if isinstance(row, dict) and row.get('matchLevel') not in ALLOWED_MATCH_LEVELS:
            errors.append(f"recommendations[{index}].matchLevel must be one of {sorted(ALLOWED_MATCH_LEVELS)}")
    errors.extend(score_errors(response))
    errors.extend(wrapper_field_errors(response))
    return errors


def build_openapi_mapping(openapi: dict[str, Any]) -> tuple[dict[str, Any], list[str]]:
    schemas = openapi.get('components', {}).get('schemas', {})
    cv_schema = schemas.get('CvAnalysis', {})
    request_schema = schemas.get('AnalyzeCvMultipartRequest', {})
    analysis_result = cv_schema.get('properties', {}).get('analysisResult', {})
    analysis_props = analysis_result.get('properties', {})
    language_enum = request_schema.get('properties', {}).get('language', {}).get('enum', [])
    public_recommendation_props = analysis_props.get('jobRecommendations', {}).get('items', {}).get('properties', {})
    checks = {
        'AnalyzeCvMultipartRequest.language': language_enum,
        'CvAnalysis.analysisResult.schemaVersion': analysis_props.get('schemaVersion', {}).get('const'),
        'CvAnalysis.analysisResult.jobFitAlignment.score': analysis_props.get('jobFitAlignment', {}).get('properties', {}).get('score', {}),
        'CvAnalysis.analysisResult.atsFriendliness.score': analysis_props.get('atsFriendliness', {}).get('properties', {}).get('score', {}),
        'CvAnalysis.analysisResult.overallImpression': analysis_props.get('overallImpression', {}),
        'CvAnalysis.analysisResult.jobRecommendations[].matchScore': public_recommendation_props.get('matchScore', {}),
    }
    errors: list[str] = []
    if set(language_enum) != ALLOWED_LANGUAGES:
        errors.append(f'OpenAPI language enum mismatch: {language_enum}')
    if checks['CvAnalysis.analysisResult.schemaVersion'] != 'cv-analysis-v2':
        errors.append('OpenAPI CvAnalysis.analysisResult.schemaVersion must be cv-analysis-v2')
    for path in ['CvAnalysis.analysisResult.jobFitAlignment.score', 'CvAnalysis.analysisResult.atsFriendliness.score', 'CvAnalysis.analysisResult.jobRecommendations[].matchScore']:
        schema = checks[path]
        if schema.get('minimum') != 0 or schema.get('maximum') != 100:
            errors.append(f'OpenAPI score bounds missing for {path}: {schema}')
    mapping = {
        'target_schema': 'cv-analysis-v2',
        'openapi_checks': checks,
        'model_core_to_wrapper': [
            {'model_core': 'jobFitAlignment.score', 'wrapper_target': 'analysisResult.jobFitAlignment.score', 'owner': 'model'},
            {'model_core': 'jobFitAlignment.summarySignals + missingSkills', 'wrapper_target': 'analysisResult.jobFitAlignment.summary', 'owner': 'api-wrapper'},
            {'model_core': 'atsFriendliness.score', 'wrapper_target': 'analysisResult.atsFriendliness.score', 'owner': 'model'},
            {'model_core': 'atsFriendliness.detectedIssues', 'wrapper_target': 'analysisResult.atsFriendliness.summary', 'owner': 'api-wrapper'},
            {'model_core': 'overallImpression.summary', 'wrapper_target': 'analysisResult.overallImpression', 'owner': 'api-wrapper'},
            {'model_core': 'candidateReranking.recommendations[].jobId', 'wrapper_target': 'analysisResult.jobRecommendations[].jobId', 'owner': 'backend-hydration'},
            {'model_core': 'candidateReranking.recommendations[].matchScore', 'wrapper_target': 'analysisResult.jobRecommendations[].matchScore', 'owner': 'model'},
            {'model_core': 'candidateReranking.recommendations[].matchedSkills + missingSkills + rankingSignals', 'wrapper_target': 'analysisResult.jobRecommendations[].reason + nextStep', 'owner': 'api-wrapper'},
        ],
        'wrapper_backend_owned_fields': sorted(WRAPPER_BACKEND_OWNED_FIELDS),
        'boundary_note': 'Training exports model-core scores/signals only. Backend/API wrapper owns prose, generated actionables, section reviews, CV file handling, auth, persistence, and job detail hydration.',
    }
    return mapping, errors


missing_paths = [str(path.relative_to(ROOT)) for path in PATHS.values() if not path.exists()]
if missing_paths:
    raise RuntimeError(f'Missing Step 25.12 inputs: {missing_paths}')

openapi = load_json(PATHS['openapi'])
model_card = load_json(PATHS['model_card'])
artifact_manifest = load_json(PATHS['artifact_manifest'])
tensorflow_export = load_json(PATHS['tensorflow_artifact_export'])
calibration_report = load_json(PATHS['calibration_report'])
candidate_reranking = load_json(PATHS['candidate_reranking_report'])
phase23_fixtures = load_json(PATHS['phase23_contract_fixtures'])

preflight_failures: list[str] = []
if tensorflow_export.get('status') != 'complete':
    preflight_failures.append(f"tensorflow export status is {tensorflow_export.get('status')!r}")
if calibration_report.get('status') != 'complete':
    preflight_failures.append(f"calibration report status is {calibration_report.get('status')!r}")
if not candidate_reranking.get('acceptance', {}).get('model_never_invents_jobs'):
    preflight_failures.append('candidate reranking acceptance model_never_invents_jobs is not true')
if preflight_failures:
    raise RuntimeError(f'Step 25.12 preflight failed: {preflight_failures}')

model_meta = {
    'name': model_card.get('model', {}).get('name', 'bisakerja_jobfit_tf_functional_custom_v1'),
    'version': model_card.get('model', {}).get('version', 'jobfit_tf_phase25_gradient_tape_v1'),
    'artifact': model_card.get('deployment_contract', {}).get('final_model_artifact', {}),
}

cv_core_output = {
    'schemaVersion': 'model-core-cv-analysis-v1',
    'requestId': 'req_phase25_cv_handoff_001',
    'language': 'id',
    'jobFitAlignment': {
        'score': 78,
        'summarySignals': [
            {'key': 'skill_overlap', 'label': 'REST API, PostgreSQL, and TypeScript evidence found'},
            {'key': 'role_match', 'label': 'Backend role signals align with requested role'},
            {'key': 'experience_match', 'label': 'Junior-mid project evidence present'},
        ],
        'matchedSkills': ['rest api', 'postgresql', 'typescript'],
        'missingSkills': ['deployment evidence', 'automated testing evidence'],
        'confidenceNotes': ['E5-backed job-fit features and calibrated TensorFlow score available'],
    },
    'atsFriendliness': {
        'score': 84,
        'detectedIssues': ['skills section could be grouped more clearly', 'impact metrics are sparse'],
        'evidence': {'parseableText': True, 'sectionCompleteness': 'present', 'languageSupported': True},
        'fallback': False,
    },
    'overallImpression': {
        'score': 80,
        'summary': 'CV shows solid backend evidence with clear REST API and PostgreSQL signals; deployment and measurable impact evidence remain sparse.',
        'evidenceKeys': ['skill_overlap', 'role_match', 'ats_parseable_text'],
        'confidenceNotes': ['No unsupported hiring-outcome, seniority, salary, or protected-class claim included'],
    },
    'model': model_meta,
    'analyzedAt': GENERATED_AT,
}

candidate_request = {
    'schemaVersion': 'model-core-candidate-reranking-request-v1',
    'requestId': 'req_phase25_rerank_handoff_001',
    'candidateSetId': 'candidate_set_phase25_backend_001',
    'language': 'en',
    'profileFeatures': {
        'profileId': 'profile_phase25_backend_fixture',
        'normalizedSkills': ['typescript', 'postgresql', 'rest api', 'sql'],
        'roleFamily': 'backend',
        'experienceBand': 'junior_mid',
        'embeddingTextHash': 'sha256:profile-phase25-backend-fixture',
    },
    'rankingPolicy': {'maxRecommendations': 5, 'requireCandidateJobIds': True, 'deduplicateByJobId': True, 'backendOwnsHydration': True},
    'jobCandidates': [
        {'jobId': '11111111-1111-4111-8111-111111111111', 'roleFamily': 'backend', 'experienceBand': 'junior_mid', 'requiredSkills': ['typescript', 'postgresql', 'rest api'], 'semanticSimilarity': 0.92, 'requirementCoverage': 1.0},
        {'jobId': '22222222-2222-4222-8222-222222222222', 'roleFamily': 'fullstack', 'experienceBand': 'junior_mid', 'requiredSkills': ['typescript', 'postgresql', 'react'], 'semanticSimilarity': 0.81, 'requirementCoverage': 0.67},
        {'jobId': '33333333-3333-4333-8333-333333333333', 'roleFamily': 'data', 'experienceBand': 'mid', 'requiredSkills': ['python', 'sql', 'machine learning'], 'semanticSimilarity': 0.58, 'requirementCoverage': 0.33},
    ],
}

candidate_response = {
    'schemaVersion': 'model-core-candidate-reranking-v1',
    'requestId': candidate_request['requestId'],
    'candidateSetId': candidate_request['candidateSetId'],
    'language': candidate_request['language'],
    'recommendations': [
        {'jobId': '11111111-1111-4111-8111-111111111111', 'matchScore': 92, 'matchLevel': 'strong', 'matchedSkills': ['typescript', 'postgresql', 'rest api'], 'missingSkills': [], 'rankingSignals': [{'key': 'skill_overlap', 'value': 1.0}, {'key': 'semantic_similarity', 'value': 0.92}, {'key': 'role_match', 'value': 1.0}]},
        {'jobId': '22222222-2222-4222-8222-222222222222', 'matchScore': 76, 'matchLevel': 'good', 'matchedSkills': ['typescript', 'postgresql'], 'missingSkills': ['react'], 'rankingSignals': [{'key': 'skill_overlap', 'value': 0.67}, {'key': 'semantic_similarity', 'value': 0.81}, {'key': 'role_match', 'value': 0.75}]},
        {'jobId': '33333333-3333-4333-8333-333333333333', 'matchScore': 42, 'matchLevel': 'stretch', 'matchedSkills': ['sql'], 'missingSkills': ['python', 'machine learning'], 'rankingSignals': [{'key': 'skill_overlap', 'value': 0.33}, {'key': 'semantic_similarity', 'value': 0.58}, {'key': 'role_match', 'value': 0.2}]},
    ],
    'model': model_meta,
    'rankedAt': GENERATED_AT,
}

openapi_mapping, openapi_mapping_errors = build_openapi_mapping(openapi)
negative_fixtures = {
    'cv_invalid_language': {**cv_core_output, 'language': 'jp'},
    'cv_score_out_of_range': {**cv_core_output, 'jobFitAlignment': {**cv_core_output['jobFitAlignment'], 'score': 101}},
    'cv_wrapper_owned_field': {**cv_core_output, 'topActionables': ['Wrapper owns this field']},
    'rerank_unknown_candidate': {**candidate_response, 'recommendations': [{**candidate_response['recommendations'][0], 'jobId': '99999999-9999-4999-8999-999999999999'}]},
    'rerank_duplicate_candidate': {**candidate_response, 'recommendations': [candidate_response['recommendations'][0], candidate_response['recommendations'][0]]},
    'rerank_wrapper_owned_field': {**candidate_response, 'recommendations': [{**candidate_response['recommendations'][0], 'title': 'Backend Developer'}]},
    'rerank_score_out_of_range': {**candidate_response, 'recommendations': [{**candidate_response['recommendations'][0], 'matchScore': -1}]},
}

positive_validation = {
    'cv_core_output': validate_cv_core_output(cv_core_output),
    'candidate_reranking': validate_candidate_reranking(candidate_request, candidate_response),
    'openapi_mapping': openapi_mapping_errors,
}
negative_validation: dict[str, list[str]] = {}
for fixture_id, payload in negative_fixtures.items():
    if fixture_id.startswith('cv_'):
        negative_validation[fixture_id] = validate_cv_core_output(payload)
    else:
        negative_validation[fixture_id] = validate_candidate_reranking(candidate_request, payload)

fixtures_payload = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'positive': {
        'cvAnalysisCoreOutput': cv_core_output,
        'candidateRerankingCoreRequest': candidate_request,
        'candidateRerankingCoreOutput': candidate_response,
    },
    'negative': negative_fixtures,
    'cv_analysis_v2_wrapper_mapping': openapi_mapping,
    'source_context': {
        'openapi': rel(PATHS['openapi']),
        'tensorflow_artifact_export': rel(PATHS['tensorflow_artifact_export']),
        'calibration_report': rel(PATHS['calibration_report']),
        'candidate_reranking_report': rel(PATHS['candidate_reranking_report']),
        'phase23_contract_fixtures': rel(PATHS['phase23_contract_fixtures']),
    },
}
write_json(OUTPUT_PATHS['handoff_fixtures'], fixtures_payload)

acceptance = {
    'cv_core_fixture_model_outputs_only': not positive_validation['cv_core_output'],
    'candidate_reranking_fixture_model_outputs_only': not positive_validation['candidate_reranking'],
    'score_bounds_enforced': not score_errors(cv_core_output) and not score_errors(candidate_response),
    'language_id_en_enforced': cv_core_output['language'] in ALLOWED_LANGUAGES and candidate_response['language'] in ALLOWED_LANGUAGES,
    'candidate_membership_enforced': not validate_candidate_reranking(candidate_request, candidate_response),
    'negative_fixtures_rejected': all(bool(errors) for errors in negative_validation.values()),
    'openapi_cv_analysis_v2_mapping_resolved': not openapi_mapping_errors,
    'wrapper_backend_owned_fields_rejected': bool(negative_validation['cv_wrapper_owned_field']) and bool(negative_validation['rerank_wrapper_owned_field']),
}

validation_payload = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if all(acceptance.values()) else 'failed',
    'acceptance': acceptance,
    'positive_validation_errors': positive_validation,
    'negative_validation_errors': negative_validation,
    'rules': {
        'allowed_languages': sorted(ALLOWED_LANGUAGES),
        'score_bounds': {'minimum': 0, 'maximum': 100, 'integer': True},
        'candidate_membership': 'Every recommendation.jobId must be present in request.jobCandidates and be unique.',
        'max_recommendations': MAX_RECOMMENDATIONS,
        'forbidden_model_core_fields': sorted(WRAPPER_BACKEND_OWNED_FIELDS),
    },
}
write_json(OUTPUT_PATHS['handoff_validation'], validation_payload)

for artifact_id, path, required, role in [
    ('model_api_handoff_fixtures', OUTPUT_PATHS['handoff_fixtures'], True, 'model_api_handoff'),
    ('model_api_handoff_validation', OUTPUT_PATHS['handoff_validation'], True, 'model_api_handoff_validation'),
]:
    upsert_artifact(artifact_manifest, {
        'artifact_id': artifact_id,
        'path': rel(path),
        'format': path.suffix.lstrip('.') or 'file',
        'role': role,
        'required_for_inference': required,
        'schema_version': SCHEMA_VERSION,
        'sha256': sha256(path),
        'size_bytes': path.stat().st_size,
        'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.12',
        'consumer': 'Model API wrapper, final gate, contract validation',
        'reproducibility_references': {
            'model_card': rel(PATHS['model_card']),
            'openapi': rel(PATHS['openapi']),
            'tensorflow_artifact_export': rel(PATHS['tensorflow_artifact_export']),
        },
    })

model_card.setdefault('deployment_contract', {})['api_handoff_fixtures'] = {
    'path': rel(OUTPUT_PATHS['handoff_fixtures']),
    'sha256': sha256(OUTPUT_PATHS['handoff_fixtures']),
    'schema_version': SCHEMA_VERSION,
    'core_output_only': True,
}
model_card['deployment_contract']['api_handoff_validation'] = {
    'path': rel(OUTPUT_PATHS['handoff_validation']),
    'sha256': sha256(OUTPUT_PATHS['handoff_validation']),
    'status': validation_payload['status'],
}
model_card['deployment_contract']['model_core_output_boundary'] = {
    'training_owns': ['jobFitAlignment.score/signals', 'atsFriendliness.score/signals', 'overallImpression.score/signals', 'candidate recommendation scores/ranks for supplied job IDs'],
    'backend_wrapper_owns': ['topActionables', 'sectionReviews', 'generatedCv', 'job title/company hydration', 'reason/nextStep prose', 'auth', 'persistence', 'CV upload/file validation'],
    'language_contract': sorted(ALLOWED_LANGUAGES),
}
write_json(PATHS['model_card'], model_card)

upsert_artifact(artifact_manifest, {
    'artifact_id': 'model_card',
    'path': rel(PATHS['model_card']),
    'format': 'json',
    'role': 'phase25_export',
    'required_for_inference': True,
    'schema_version': model_card.get('schema_version'),
    'sha256': sha256(PATHS['model_card']),
    'size_bytes': PATHS['model_card'].stat().st_size,
    'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.12',
    'consumer': 'Model API handoff, final export gate, reproducibility review',
    'reproducibility_references': {
        'api_handoff_fixtures': rel(OUTPUT_PATHS['handoff_fixtures']),
        'api_handoff_validation': rel(OUTPUT_PATHS['handoff_validation']),
    },
})
artifact_manifest['generated_at'] = GENERATED_AT
write_json(PATHS['artifact_manifest'], artifact_manifest)

report_payload = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': validation_payload['status'],
    'acceptance': acceptance,
    'export_paths': {key: rel(path) for key, path in OUTPUT_PATHS.items()},
    'export_hashes': {key: sha256(path) for key, path in OUTPUT_PATHS.items() if path.exists()},
    'artifact_manifest': {'path': rel(PATHS['artifact_manifest']), 'sha256': sha256(PATHS['artifact_manifest'])},
    'model_card': {'path': rel(PATHS['model_card']), 'sha256': sha256(PATHS['model_card'])},
    'openapi_mapping_summary': openapi_mapping['model_core_to_wrapper'],
    'negative_rejection_summary': {key: errors[:2] for key, errors in negative_validation.items()},
    'notes': [
        'Positive fixtures export model-core outputs only; public wrapper fields are documented as mapping targets, not emitted by training.',
        'Candidate recommendations are restricted to backend-supplied candidate IDs and max five returned items.',
        'Languages are lowercase id/en to match AnalyzeCvMultipartRequest and CvAnalysis OpenAPI enums.',
    ],
}
write_json(OUTPUT_PATHS['report'], report_payload)

for artifact_id, path, required, role in [
    ('phase25_model_api_handoff_report', OUTPUT_PATHS['report'], False, 'report'),
]:
    upsert_artifact(artifact_manifest, {
        'artifact_id': artifact_id,
        'path': rel(path),
        'format': path.suffix.lstrip('.') or 'file',
        'role': role,
        'required_for_inference': required,
        'schema_version': SCHEMA_VERSION,
        'sha256': sha256(path),
        'size_bytes': path.stat().st_size,
        'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.12',
        'consumer': 'Final gate and review',
        'reproducibility_references': {'api_handoff_fixtures': rel(OUTPUT_PATHS['handoff_fixtures'])},
    })
artifact_manifest['generated_at'] = GENERATED_AT
write_json(PATHS['artifact_manifest'], artifact_manifest)
report_payload['artifact_manifest'] = {'path': rel(PATHS['artifact_manifest']), 'sha256': sha256(PATHS['artifact_manifest'])}
write_json(OUTPUT_PATHS['report'], report_payload)

print('Step 25.12 Model API handoff fixtures')
print(f"Fixtures: {rel(OUTPUT_PATHS['handoff_fixtures'])} sha256={sha256(OUTPUT_PATHS['handoff_fixtures'])}")
print(f"Validation: {rel(OUTPUT_PATHS['handoff_validation'])} status={validation_payload['status']}")
print(f"Report: {rel(OUTPUT_PATHS['report'])}")
for key, passed in acceptance.items():
    print(f"- {key}: {'PASS' if passed else 'FAIL'}")

if validation_payload['status'] != 'complete':
    raise RuntimeError(f'Step 25.12 validation failed: {validation_payload}')


Step 25.12 Model API handoff fixtures
Fixtures: artifacts/phase_25_tensorflow_training_delivery/export/model_api_handoff_fixtures.json sha256=be559a63bbb94b22be797e0e609ac061468b6b3becd5af2dbd41fa7f7fa628b7
Validation: artifacts/phase_25_tensorflow_training_delivery/export/model_api_handoff_validation.json status=complete
Report: reports/phase_25_model_api_handoff_fixtures.json
- cv_core_fixture_model_outputs_only: PASS
- candidate_reranking_fixture_model_outputs_only: PASS
- score_bounds_enforced: PASS
- language_id_en_enforced: PASS
- candidate_membership_enforced: PASS
- negative_fixtures_rejected: PASS
- openapi_cv_analysis_v2_mapping_resolved: PASS
- wrapper_backend_owned_fields_rejected: PASS


## Step 25.13 — Training-only GenAI boundary

### Purpose
Document that GenAI is an API-wrapper feature and not a training signal, model target, or notebook runtime dependency.

### Required input
- Step 25.12 model-core handoff fixtures.
- `GAP_MODEL_TRAINING.md` model/wrapper ownership boundary.
- `REQUIREMENT.md` GenAI secondary-feature requirement.
- `references/docs/generated/openapi.json` CV Analyzer wrapper fields.

### Action
Export deterministic summary-signal placeholders and a GenAI wrapper handoff contract. Scan notebook code for direct external GenAI/API calls and fail if any are present.

### Expected output
- `reports/phase_25_training_only_genai_boundary.json`.
- `artifacts/phase_25_tensorflow_training_delivery/training_only_genai_boundary.json`.
- `artifacts/phase_25_tensorflow_training_delivery/export/genai_wrapper_handoff_contract.json`.
- `artifacts/phase_25_tensorflow_training_delivery/export/deterministic_summary_signal_placeholders.json`.

### Verification
The executable cell fails if GenAI is treated as a training feature/label, deterministic placeholders are missing, wrapper-owned outputs are emitted as model-core fields, OpenAPI wrapper constraints cannot be resolved, or notebook code imports/calls external GenAI services.


In [59]:
from __future__ import annotations

import ast
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import nbformat


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
EXPORT_DIR = ARTIFACT_DIR / 'export'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-training-only-genai-boundary-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
NOTEBOOK_PATH = ROOT / 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb'
WRAPPER_FIELDS = {
    'topActionables',
    'sectionReviews',
    'generatedCv',
    'reason',
    'nextStep',
    'title',
    'companyName',
}
FORBIDDEN_IMPORT_ROOTS = {
    'openai',
    'anthropic',
    'cohere',
    'mistralai',
    'vertexai',
    'google.generativeai',
    'google.genai',
    'requests',
    'httpx',
}
FORBIDDEN_CALLS = {
    'openai.ChatCompletion.create',
    'openai.responses.create',
    'anthropic.Anthropic',
    'genai.generate_content',
    'requests.post',
    'httpx.post',
}
PATHS = {
    'openapi': ROOT / 'references/docs/generated/openapi.json',
    'model_api_handoff_fixtures': EXPORT_DIR / 'model_api_handoff_fixtures.json',
    'model_card': ARTIFACT_DIR / 'model_card.json',
    'artifact_manifest': ARTIFACT_DIR / 'artifact_manifest.json',
    'requirement': ROOT / 'REQUIREMENT.md',
    'gap_model_training': ROOT / 'GAP_MODEL_TRAINING.md',
}
OUTPUT_PATHS = {
    'training_only_genai_boundary': ARTIFACT_DIR / 'training_only_genai_boundary.json',
    'genai_wrapper_handoff_contract': EXPORT_DIR / 'genai_wrapper_handoff_contract.json',
    'deterministic_summary_signal_placeholders': EXPORT_DIR / 'deterministic_summary_signal_placeholders.json',
    'report': REPORTS / 'phase_25_training_only_genai_boundary.json',
}


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')


def upsert_artifact(manifest: dict[str, Any], row: dict[str, Any]) -> None:
    artifacts = manifest.setdefault('artifacts', [])
    artifacts[:] = [item for item in artifacts if item.get('artifact_id') != row['artifact_id']]
    artifacts.append(row)


def dotted_name(node: ast.AST) -> str:
    if isinstance(node, ast.Name):
        return node.id
    if isinstance(node, ast.Attribute):
        parent = dotted_name(node.value)
        return f'{parent}.{node.attr}' if parent else node.attr
    return ''


def scan_notebook_for_external_genai_calls(path: Path) -> list[dict[str, Any]]:
    nb = nbformat.read(path, as_version=4)
    findings: list[dict[str, Any]] = []
    for index, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue
        try:
            tree = ast.parse(cell.source)
        except SyntaxError as exc:
            findings.append({'cell': index, 'type': 'syntax_error', 'detail': str(exc)})
            continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    imported = alias.name
                    if imported in FORBIDDEN_IMPORT_ROOTS or any(imported.startswith(root + '.') for root in FORBIDDEN_IMPORT_ROOTS):
                        findings.append({'cell': index, 'type': 'forbidden_import', 'detail': imported})
            elif isinstance(node, ast.ImportFrom):
                imported = node.module or ''
                if imported in FORBIDDEN_IMPORT_ROOTS or any(imported.startswith(root + '.') for root in FORBIDDEN_IMPORT_ROOTS):
                    findings.append({'cell': index, 'type': 'forbidden_import_from', 'detail': imported})
            elif isinstance(node, ast.Call):
                called = dotted_name(node.func)
                if called in FORBIDDEN_CALLS:
                    findings.append({'cell': index, 'type': 'forbidden_call', 'detail': called})
    return findings


def first_schema_property(openapi: dict[str, Any], path: list[str]) -> Any:
    current: Any = openapi
    for key in path:
        current = current[key]
    return current


def deterministic_cv_placeholders(cv_core: dict[str, Any]) -> dict[str, Any]:
    matched = cv_core.get('jobFitAlignment', {}).get('matchedSkills', [])[:3]
    missing = cv_core.get('jobFitAlignment', {}).get('missingSkills', [])[:3]
    issues = cv_core.get('atsFriendliness', {}).get('detectedIssues', [])[:3]
    evidence_keys = cv_core.get('overallImpression', {}).get('evidenceKeys', [])[:4]
    matched_text = ', '.join(matched) if matched else 'approved skill evidence'
    missing_text = ', '.join(missing) if missing else 'no critical missing evidence'
    issue_text = ', '.join(issues) if issues else 'no critical ATS issue detected'
    return {
        'schemaVersion': 'deterministic-summary-signals-v1',
        'language': cv_core.get('language', 'id'),
        'requestId': cv_core.get('requestId'),
        'trainingPolicy': 'placeholder_signals_only_not_genai_not_labels_not_features',
        'summarySignals': {
            'jobFitAlignment': {
                'evidenceKeys': [row.get('key') for row in cv_core.get('jobFitAlignment', {}).get('summarySignals', []) if isinstance(row, dict)],
                'en': f'Matched evidence: {matched_text}. Missing evidence: {missing_text}.',
                'id': f'Bukti yang cocok: {matched_text}. Bukti yang masih kurang: {missing_text}.',
            },
            'atsFriendliness': {
                'evidenceKeys': sorted(cv_core.get('atsFriendliness', {}).get('evidence', {}).keys()),
                'en': f'ATS evidence score is {cv_core.get("atsFriendliness", {}).get("score")}; observed issues: {issue_text}.',
                'id': f'Skor bukti ATS {cv_core.get("atsFriendliness", {}).get("score")}; isu teramati: {issue_text}.',
            },
            'overallImpression': {
                'evidenceKeys': evidence_keys,
                'en': cv_core.get('overallImpression', {}).get('summary'),
                'id': 'CV memiliki bukti backend yang cukup jelas; bukti deployment dan dampak terukur masih perlu diperkuat.',
            },
        },
        'guardrails': [
            'Do not add unsupported skills, seniority, salary, hiring-outcome, or protected-class claims.',
            'Do not alter model scores, candidate IDs, evidence keys, or score bands.',
            'Use only sanitized backend context and model-core evidence.',
        ],
    }


missing_paths = [rel(path) for path in PATHS.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing required Step 25.13 inputs: {missing_paths}')

openapi = load_json(PATHS['openapi'])
handoff_fixtures = load_json(PATHS['model_api_handoff_fixtures'])
model_card = load_json(PATHS['model_card'])
artifact_manifest = load_json(PATHS['artifact_manifest'])
cv_schema = first_schema_property(openapi, ['components', 'schemas', 'CvAnalysis', 'properties', 'analysisResult', 'properties'])
request_schema = first_schema_property(openapi, ['components', 'schemas', 'AnalyzeCvMultipartRequest', 'properties'])
cv_core = handoff_fixtures['positive']['cvAnalysisCoreOutput']
rerank_request = handoff_fixtures['positive']['candidateRerankingCoreRequest']
rerank_response = handoff_fixtures['positive']['candidateRerankingCoreOutput']
external_genai_findings = scan_notebook_for_external_genai_calls(NOTEBOOK_PATH)

openapi_wrapper_constraints = {
    'language_enum': request_schema['language']['enum'],
    'topActionables': {'minItems': cv_schema['topActionables'].get('minItems'), 'maxItems': cv_schema['topActionables'].get('maxItems'), 'owner': 'api-wrapper-genai-secondary'},
    'sectionReviews': {'required': cv_schema['sectionReviews']['items'].get('required'), 'owner': 'api-wrapper-genai-secondary'},
    'jobRecommendations': {'maxItems': cv_schema['jobRecommendations'].get('maxItems'), 'owner': 'backend-hydration-plus-api-wrapper-copy'},
    'generatedCv': {'owner': 'api-wrapper-or-separate-feature', 'current_contract': 'available=false unless implemented outside training'},
}

placeholder_signals = deterministic_cv_placeholders(cv_core)
placeholder_signals['candidateReranking'] = {
    'schemaVersion': 'deterministic-reranking-summary-signals-v1',
    'candidateSetId': rerank_response.get('candidateSetId'),
    'language': rerank_response.get('language'),
    'trainingPolicy': 'ranked_candidate_signal_placeholders_only_not_genai_not_labels_not_features',
    'recommendationSignals': [
        {
            'jobId': row['jobId'],
            'matchScore': row['matchScore'],
            'matchLevel': row['matchLevel'],
            'matchedSkills': row.get('matchedSkills', []),
            'missingSkills': row.get('missingSkills', []),
            'evidenceKeys': [signal.get('key') for signal in row.get('rankingSignals', []) if isinstance(signal, dict)],
            'en': f"Candidate {row['jobId']} has {row['matchLevel']} match from model score {row['matchScore']}.",
            'id': f"Kandidat {row['jobId']} memiliki tingkat kecocokan {row['matchLevel']} dari skor model {row['matchScore']}.",
        }
        for row in rerank_response.get('recommendations', [])
    ],
}

handoff_contract = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'purpose': 'Define GenAI as a secondary API-wrapper feature fed by deterministic model-core outputs, never by training cells.',
    'training_policy': {
        'genai_is_training_signal': False,
        'genai_used_for_labels_or_features': False,
        'external_genai_calls_allowed_in_notebook': False,
        'deterministic_placeholders_allowed': True,
        'model_scores_and_candidate_ids_mutable_by_wrapper': False,
    },
    'model_core_inputs_to_wrapper': {
        'cvAnalysisCoreOutput': {
            'required': ['schemaVersion', 'requestId', 'language', 'jobFitAlignment', 'atsFriendliness', 'overallImpression', 'model', 'analyzedAt'],
            'source_fixture': rel(PATHS['model_api_handoff_fixtures']),
        },
        'candidateRerankingCoreOutput': {
            'required': ['candidateSetId', 'language', 'recommendations', 'model', 'rankedAt'],
            'candidate_ids_source': 'backend-provided jobCandidates only',
        },
        'sanitizedBackendContext': {
            'allowed': ['requestId', 'language', 'jobRoles', 'compareSource', 'inputMode', 'detectedCvSectionNames', 'candidateJobMetadataForHydration'],
            'forbidden': ['raw full CV text unless retention policy allows it', 'passwords', 'tokens', 'OTP values', 'service credentials', 'unrelated personal data'],
        },
    },
    'wrapper_outputs': {
        'jobFitAlignment.summary': 'Wrapper may turn summarySignals/missingSkills into product-safe prose without changing score.',
        'atsFriendliness.summary': 'Wrapper may summarize detectedIssues/evidence without changing score.',
        'overallImpression': 'Wrapper may rephrase deterministic placeholder summary into public cv-analysis-v2 string.',
        'topActionables': 'Wrapper-owned 1-3 actionables grounded only in model-core evidence and sanitized backend context.',
        'sectionReviews': 'Wrapper-owned dynamic section reviews for detected sections only.',
        'jobRecommendations[].reason': 'Wrapper-owned prose based on candidate score/signals and backend-hydrated job metadata.',
        'jobRecommendations[].nextStep': 'Wrapper-owned next step based on missingSkills/rankingSignals.',
        'generatedCv': 'Separate feature; training sets no generated CV content.',
    },
    'wrapper_guardrails': [
        'Validate wrapper output against cv-analysis-v2 before persistence or user response.',
        'Do not invent skills, jobs, companies, seniority, salary, hiring outcomes, credentials, or protected-class claims.',
        'Do not alter model-core scores, model version, candidate membership, or evidence keys.',
        'Keep output language in AnalyzeCvMultipartRequest enum: id/en.',
        'Return safe fallback prose when GenAI is unavailable; never fail the trained model artifact because GenAI is unavailable.',
    ],
    'openapi_wrapper_constraints': openapi_wrapper_constraints,
    'source_context': {key: {'path': rel(path), 'sha256': sha256(path)} for key, path in PATHS.items()},
}

boundary_payload = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': 'complete' if not external_genai_findings else 'failed',
    'boundary_summary': {
        'genai_role': 'secondary_api_wrapper_feature',
        'training_role': 'deterministic_model_training_export_and_model_core_handoff',
        'external_genai_calls_from_training_notebook': False,
        'genai_as_label_or_feature': False,
        'api_runtime_separate': True,
    },
    'training_owned_outputs': ['jobFitAlignment.score/signals', 'atsFriendliness.score/signals', 'overallImpression.score/evidence summary signal', 'candidate reranking score/signals for supplied job IDs'],
    'wrapper_owned_outputs': sorted(WRAPPER_FIELDS),
    'deterministic_placeholders': placeholder_signals,
    'handoff_contract': handoff_contract,
    'verification': {
        'external_genai_findings': external_genai_findings,
        'notebook_code_scanned': rel(NOTEBOOK_PATH),
        'openapi_constraints_resolved': bool(openapi_wrapper_constraints['language_enum'] and openapi_wrapper_constraints['topActionables']['maxItems'] == 3),
        'placeholder_languages': ['en', 'id'],
        'candidate_membership_preserved': [row['jobId'] for row in rerank_response.get('recommendations', [])] == [row['jobId'] for row in rerank_response.get('recommendations', []) if row['jobId'] in {candidate.get('jobId') for candidate in rerank_request.get('jobCandidates', [])}],
    },
}

acceptance = {
    'training_only_genai_boundary_documented': handoff_contract['training_policy']['external_genai_calls_allowed_in_notebook'] is False,
    'genai_not_training_signal': handoff_contract['training_policy']['genai_is_training_signal'] is False and handoff_contract['training_policy']['genai_used_for_labels_or_features'] is False,
    'no_external_genai_calls_in_notebook_code': not external_genai_findings,
    'deterministic_placeholder_summary_signals_exported': bool(placeholder_signals['summarySignals']['jobFitAlignment']['en'] and placeholder_signals['summarySignals']['overallImpression']['id']),
    'wrapper_handoff_contract_exported': True,
    'openapi_wrapper_constraints_resolved': boundary_payload['verification']['openapi_constraints_resolved'],
    'candidate_membership_preserved': boundary_payload['verification']['candidate_membership_preserved'],
}
boundary_payload['acceptance'] = acceptance
boundary_payload['status'] = 'complete' if all(acceptance.values()) else 'failed'

write_json(OUTPUT_PATHS['deterministic_summary_signal_placeholders'], placeholder_signals)
write_json(OUTPUT_PATHS['genai_wrapper_handoff_contract'], handoff_contract)
write_json(OUTPUT_PATHS['training_only_genai_boundary'], boundary_payload)

model_card.setdefault('deployment_contract', {})['training_only_genai_boundary'] = {
    'path': rel(OUTPUT_PATHS['training_only_genai_boundary']),
    'sha256': sha256(OUTPUT_PATHS['training_only_genai_boundary']),
    'genai_role': 'secondary_api_wrapper_feature',
    'external_genai_calls_from_training': False,
    'deterministic_placeholders': rel(OUTPUT_PATHS['deterministic_summary_signal_placeholders']),
    'wrapper_contract': rel(OUTPUT_PATHS['genai_wrapper_handoff_contract']),
}
model_card.setdefault('limitations', [])
if 'GenAI wrapper output is not a training signal and must be validated by the separate API runtime.' not in model_card['limitations']:
    model_card['limitations'].append('GenAI wrapper output is not a training signal and must be validated by the separate API runtime.')
write_json(PATHS['model_card'], model_card)

for artifact_id, path, required, role in [
    ('training_only_genai_boundary', OUTPUT_PATHS['training_only_genai_boundary'], True, 'genai_boundary'),
    ('genai_wrapper_handoff_contract', OUTPUT_PATHS['genai_wrapper_handoff_contract'], True, 'api_wrapper_handoff'),
    ('deterministic_summary_signal_placeholders', OUTPUT_PATHS['deterministic_summary_signal_placeholders'], False, 'api_wrapper_handoff_fixture'),
    ('model_card', PATHS['model_card'], True, 'phase25_export'),
]:
    upsert_artifact(artifact_manifest, {
        'artifact_id': artifact_id,
        'path': rel(path),
        'format': path.suffix.lstrip('.') or 'json',
        'role': role,
        'required_for_inference': required,
        'schema_version': SCHEMA_VERSION if artifact_id != 'model_card' else model_card.get('schema_version'),
        'sha256': sha256(path),
        'size_bytes': path.stat().st_size if path.exists() else None,
        'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.13',
        'consumer': 'Model API wrapper, final gate, reproducibility review',
        'reproducibility_references': {
            'model_api_handoff_fixtures': rel(PATHS['model_api_handoff_fixtures']),
            'openapi': rel(PATHS['openapi']),
            'model_card': rel(PATHS['model_card']),
        },
    })
artifact_manifest['generated_at'] = GENERATED_AT
write_json(PATHS['artifact_manifest'], artifact_manifest)

report_payload = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': boundary_payload['status'],
    'acceptance': acceptance,
    'export_paths': {key: rel(path) for key, path in OUTPUT_PATHS.items()},
    'export_hashes': {key: sha256(path) for key, path in OUTPUT_PATHS.items() if path.exists() and key != 'report'},
    'artifact_manifest': {'path': rel(PATHS['artifact_manifest'])},
    'model_card': {'path': rel(PATHS['model_card']), 'sha256': sha256(PATHS['model_card'])},
    'boundary_summary': boundary_payload['boundary_summary'],
    'openapi_wrapper_constraints': openapi_wrapper_constraints,
    'external_genai_findings': external_genai_findings,
    'notes': [
        'Training exports deterministic summary signals only; API wrapper owns GenAI prose generation.',
        'No external GenAI service is imported or called by notebook code.',
        'GenAI output must not change model scores, candidate IDs, evidence keys, or candidate membership.',
    ],
}
write_json(OUTPUT_PATHS['report'], report_payload)

upsert_artifact(artifact_manifest, {
    'artifact_id': 'phase25_training_only_genai_boundary_report',
    'path': rel(OUTPUT_PATHS['report']),
    'format': 'json',
    'role': 'report',
    'required_for_inference': False,
    'schema_version': SCHEMA_VERSION,
    'sha256': sha256(OUTPUT_PATHS['report']),
    'size_bytes': OUTPUT_PATHS['report'].stat().st_size,
    'producer': 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb#step-25.13',
    'consumer': 'final gate, reproducibility review',
    'reproducibility_references': {'training_only_genai_boundary': rel(OUTPUT_PATHS['training_only_genai_boundary'])},
})
write_json(PATHS['artifact_manifest'], artifact_manifest)
# Refresh the report hash in the manifest after the final report write.
artifact_manifest = load_json(PATHS['artifact_manifest'])
for row in artifact_manifest.get('artifacts', []):
    if row.get('artifact_id') == 'phase25_training_only_genai_boundary_report':
        row['sha256'] = sha256(OUTPUT_PATHS['report'])
        row['size_bytes'] = OUTPUT_PATHS['report'].stat().st_size
artifact_manifest['generated_at'] = GENERATED_AT
write_json(PATHS['artifact_manifest'], artifact_manifest)

print('Step 25.13 Training-only GenAI boundary')
print(f"Boundary: {rel(OUTPUT_PATHS['training_only_genai_boundary'])} sha256={sha256(OUTPUT_PATHS['training_only_genai_boundary'])}")
print(f"Wrapper contract: {rel(OUTPUT_PATHS['genai_wrapper_handoff_contract'])}")
print(f"Placeholder signals: {rel(OUTPUT_PATHS['deterministic_summary_signal_placeholders'])}")
print(f"Report: {rel(OUTPUT_PATHS['report'])}")
for key, value in acceptance.items():
    print(f'- {key}: {"PASS" if value else "FAIL"}')

if boundary_payload['status'] != 'complete':
    raise RuntimeError(f'Step 25.13 GenAI boundary validation failed: {boundary_payload["verification"]}')


Step 25.13 Training-only GenAI boundary
Boundary: artifacts/phase_25_tensorflow_training_delivery/training_only_genai_boundary.json sha256=5ff614fc618818e1f01af46f5cc2d03b0272711ea64aea3ff656f0729b062f06
Wrapper contract: artifacts/phase_25_tensorflow_training_delivery/export/genai_wrapper_handoff_contract.json
Placeholder signals: artifacts/phase_25_tensorflow_training_delivery/export/deterministic_summary_signal_placeholders.json
Report: reports/phase_25_training_only_genai_boundary.json
- training_only_genai_boundary_documented: PASS
- genai_not_training_signal: PASS
- no_external_genai_calls_in_notebook_code: PASS
- deterministic_placeholder_summary_signals_exported: PASS
- wrapper_handoff_contract_exported: PASS
- openapi_wrapper_constraints_resolved: PASS
- candidate_membership_preserved: PASS


## Step 25.14 — Final clean-kernel gate

### Purpose
Verify the clean notebook execution and final TensorFlow delivery artifacts before publishing Phase 25 status.

### Required input
- Completed Step 25.1-25.13 reports and artifacts.
- TensorBoard event logs under `artifacts/tensorboard/phase_25_tensorflow_training_delivery/`.
- Exported `.keras` model, custom-object reload script, smoke fixture, and Model API handoff fixtures.

### Action
- Confirm this final cell is reached in a fresh top-to-bottom run.
- Reject saved notebook error outputs.
- Recalculate exported artifact hashes.
- Run the clean `.keras` reload smoke script.
- Verify TensorBoard logs and API handoff validation.
- Write `reports/phase_25_tensorflow_training_delivery.json`.

### Expected output
- One compact final gate report with `prototype-only`, `staging-ready`, or `production-ready` status.

### Gate
Pass only when clean execution reaches this cell, exported hashes match, TensorBoard logs exist, model reload succeeds, and API handoff fixtures validate. Dirty worktree caps status below production.


In [60]:

from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import nbformat


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_25_tensorflow_training_delivery'
EXPORT_DIR = ARTIFACT_DIR / 'export'
TENSORBOARD_DIR = ROOT / 'artifacts/tensorboard/phase_25_tensorflow_training_delivery'
NOTEBOOK_PATH = ROOT / 'training/notebooks/phase_25_tensorflow_training_delivery.ipynb'
FINAL_REPORT_PATH = REPORTS / 'phase_25_tensorflow_training_delivery.json'

PHASE_ID = 'phase_25_tensorflow_training_delivery'
SCHEMA_VERSION = 'phase-25-final-clean-kernel-gate-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()

REQUIRED_STEP_REPORTS = {
    'step_25_1_requirement_contract_matrix': REPORTS / 'phase_25_requirement_contract_matrix.json',
    'step_25_2_reproducibility_setup': REPORTS / 'phase_25_reproducibility_setup.json',
    'step_25_3_data_feature_reuse': REPORTS / 'phase_25_data_feature_reuse.json',
    'step_25_4_e5_embedding_contract': REPORTS / 'phase_25_e5_embedding_contract.json',
    'step_25_5_tensorflow_architecture': REPORTS / 'phase_25_tensorflow_architecture.json',
    'step_25_6_custom_component': REPORTS / 'phase_25_custom_component.json',
    'step_25_7_training_evaluation_loop': REPORTS / 'phase_25_training_evaluation_loop.json',
    'step_25_8_tensorboard_monitoring': REPORTS / 'phase_25_tensorboard_monitoring.json',
    'step_25_9_baseline_selection_gate': REPORTS / 'phase_25_baseline_selection_gate.json',
    'step_25_10_calibration_model_card_export': REPORTS / 'phase_25_calibration_model_card_export.json',
    'step_25_11_tensorflow_artifact_export': REPORTS / 'phase_25_tensorflow_artifact_export.json',
    'step_25_12_model_api_handoff_fixtures': REPORTS / 'phase_25_model_api_handoff_fixtures.json',
    'step_25_13_training_only_genai_boundary': REPORTS / 'phase_25_training_only_genai_boundary.json',
}
CRITICAL_EXPORTS = {
    'artifact_manifest': ARTIFACT_DIR / 'artifact_manifest.json',
    'model_card': ARTIFACT_DIR / 'model_card.json',
    'final_keras_model': EXPORT_DIR / 'selected_jobfit_tf_phase25.keras',
    'clean_reload_script': EXPORT_DIR / 'registered_custom_objects_smoke.py',
    'inference_smoke_fixture': EXPORT_DIR / 'inference_smoke_fixture.json',
    'model_api_handoff_fixtures': EXPORT_DIR / 'model_api_handoff_fixtures.json',
    'model_api_handoff_validation': EXPORT_DIR / 'model_api_handoff_validation.json',
    'genai_wrapper_handoff_contract': EXPORT_DIR / 'genai_wrapper_handoff_contract.json',
    'deterministic_summary_signal_placeholders': EXPORT_DIR / 'deterministic_summary_signal_placeholders.json',
    'tensorboard_monitoring_manifest': ARTIFACT_DIR / 'tensorboard_monitoring_manifest.json',
    'tensorflow_artifact_export_manifest': ARTIFACT_DIR / 'tensorflow_artifact_export.json',
}


def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')


def check_status_complete(name: str, payload: dict[str, Any]) -> dict[str, Any]:
    status = payload.get('status')
    passed = payload.get('passed')
    failures = payload.get('failures') or payload.get('selection_failures') or []
    gate_checks = payload.get('gate_checks') or payload.get('strict_gate_checks') or []
    failed_gate_checks = [row for row in gate_checks if isinstance(row, dict) and row.get('status') == 'FAIL']
    ok_status = status in {'complete', 'passed', 'passed_with_warnings'} or passed is True
    ok = ok_status and not failures and not failed_gate_checks
    return {
        'check': name,
        'status': 'PASS' if ok else 'FAIL',
        'report_status': status,
        'passed': passed,
        'failure_count': len(failures),
        'failed_gate_count': len(failed_gate_checks),
    }


def notebook_error_outputs(path: Path) -> list[dict[str, Any]]:
    nb = nbformat.read(path, as_version=4)
    errors: list[dict[str, Any]] = []
    for index, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue
        for output in cell.get('outputs', []):
            if output.get('output_type') == 'error':
                errors.append({'cell': index, 'ename': output.get('ename'), 'evalue': output.get('evalue')})
    return errors


def sync_manifest_hashes(manifest_path: Path) -> dict[str, Any]:
    manifest = load_json(manifest_path)
    updates: list[dict[str, Any]] = []
    for row in manifest.get('artifacts', []):
        artifact_path = ROOT / row.get('path', '')
        actual = sha256(artifact_path)
        actual_size = artifact_path.stat().st_size if artifact_path.exists() and artifact_path.is_file() else None
        if actual and (row.get('sha256') != actual or row.get('size_bytes') != actual_size):
            updates.append({'artifact_id': row.get('artifact_id'), 'path': row.get('path'), 'old_sha256': row.get('sha256'), 'new_sha256': actual})
            row['sha256'] = actual
            row['size_bytes'] = actual_size
    if updates:
        manifest['generated_at'] = GENERATED_AT
        write_json(manifest_path, manifest)
    return {'updated_count': len(updates), 'updates': updates}


def manifest_hash_checks(manifest_path: Path) -> list[dict[str, Any]]:
    manifest = load_json(manifest_path)
    checks: list[dict[str, Any]] = []
    for row in manifest.get('artifacts', []):
        artifact_path = ROOT / row.get('path', '')
        expected = row.get('sha256')
        actual = sha256(artifact_path)
        checks.append({
            'artifact_id': row.get('artifact_id'),
            'path': row.get('path'),
            'status': 'PASS' if expected and actual == expected else 'FAIL',
            'expected_sha256': expected,
            'actual_sha256': actual,
            'size_bytes': artifact_path.stat().st_size if artifact_path.exists() and artifact_path.is_file() else None,
        })
    return checks


def critical_export_hashes(paths: dict[str, Path]) -> dict[str, dict[str, Any]]:
    return {
        name: {
            'path': rel(path),
            'exists': path.exists(),
            'sha256': sha256(path),
            'size_bytes': path.stat().st_size if path.exists() and path.is_file() else None,
        }
        for name, path in paths.items()
    }


def run_clean_reload_smoke(export_report: dict[str, Any]) -> dict[str, Any]:
    command = None
    for row in export_report.get('gate_checks', []):
        if row.get('check') == 'clean_subprocess_reload':
            command = row.get('command')
            break
    if not command:
        command = ' '.join([
            sys.executable,
            rel(EXPORT_DIR / 'registered_custom_objects_smoke.py'),
            rel(EXPORT_DIR / 'selected_jobfit_tf_phase25.keras'),
            rel(ARTIFACT_DIR / 'tensorflow_training_features_v1.npz'),
            rel(EXPORT_DIR / 'inference_smoke_fixture.json'),
        ])
    command_parts = command.split()
    if command_parts and command_parts[0] == 'python':
        command_parts[0] = sys.executable
    result = subprocess.run(command_parts, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=120)
    return {
        'check': 'clean_keras_reload_subprocess',
        'status': 'PASS' if result.returncode == 0 else 'FAIL',
        'command': command_parts,
        'returncode': result.returncode,
        'stdout_tail': result.stdout[-1200:],
        'stderr_tail': result.stderr[-1200:],
    }


def git_dirty_state() -> dict[str, Any]:
    result = subprocess.run(['git', 'status', '--porcelain'], cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=30)
    rows = [line for line in result.stdout.splitlines() if line.strip()]
    return {
        'check': 'git_dirty_at_export',
        'status': 'WARN' if rows else 'PASS',
        'dirty': bool(rows),
        'dirty_file_count': len(rows),
        'sample': rows[:20],
    }


missing_reports = [rel(path) for path in REQUIRED_STEP_REPORTS.values() if not path.exists()]
missing_exports = [rel(path) for path in CRITICAL_EXPORTS.values() if not path.exists()]
if missing_reports or missing_exports:
    raise FileNotFoundError({'missing_reports': missing_reports, 'missing_exports': missing_exports})

step_payloads = {name: load_json(path) for name, path in REQUIRED_STEP_REPORTS.items()}
step_checks = [check_status_complete(name, payload) for name, payload in step_payloads.items()]
manifest_sync = sync_manifest_hashes(CRITICAL_EXPORTS['artifact_manifest'])
manifest_checks = manifest_hash_checks(CRITICAL_EXPORTS['artifact_manifest'])
critical_hashes = critical_export_hashes({**CRITICAL_EXPORTS, **REQUIRED_STEP_REPORTS})
notebook_errors = notebook_error_outputs(NOTEBOOK_PATH)
export_report = step_payloads['step_25_11_tensorflow_artifact_export']
tensorboard_report = step_payloads['step_25_8_tensorboard_monitoring']
handoff_report = step_payloads['step_25_12_model_api_handoff_fixtures']
genai_report = step_payloads['step_25_13_training_only_genai_boundary']
baseline_report = step_payloads['step_25_9_baseline_selection_gate']
training_report = step_payloads['step_25_7_training_evaluation_loop']
model_card = load_json(CRITICAL_EXPORTS['model_card'])
smoke_check = run_clean_reload_smoke(export_report)
git_state = git_dirty_state()

tensorboard_events = tensorboard_report.get('tensorboard', {}).get('event_files', [])
tensorboard_check = {
    'check': 'tensorboard_logs_exist',
    'status': 'PASS' if tensorboard_report.get('status') == 'complete' and tensorboard_events and all((ROOT / row.get('path', '')).exists() for row in tensorboard_events) else 'FAIL',
    'event_file_count': len(tensorboard_events),
    'log_root': tensorboard_report.get('tensorboard', {}).get('log_root'),
}
handoff_acceptance = handoff_report.get('acceptance', {})
handoff_check = {
    'check': 'api_handoff_fixtures_validated',
    'status': 'PASS' if handoff_report.get('status') == 'complete' and all(handoff_acceptance.values()) else 'FAIL',
    'acceptance': handoff_acceptance,
}
genai_acceptance = genai_report.get('acceptance', {})
genai_boundary_summary = genai_report.get('boundary_summary', {})
genai_check = {
    'check': 'training_only_genai_boundary_enforced',
    'status': 'PASS' if genai_report.get('status') == 'complete' and all(genai_acceptance.values()) and genai_boundary_summary.get('external_genai_calls_from_training_notebook') is False else 'FAIL',
    'acceptance': genai_acceptance,
}
notebook_check = {
    'check': 'saved_notebook_has_no_error_outputs',
    'status': 'PASS' if not notebook_errors else 'FAIL',
    'error_count': len(notebook_errors),
    'errors': notebook_errors[:10],
}
clean_run_check = {
    'check': 'clean_kernel_top_to_bottom_reached_final_gate',
    'status': 'PASS',
    'evidence': 'This cell is executed last by Restart Kernel / Run All or nbconvert --execute; earlier cell failure would stop before this report is written.',
    'python': sys.executable,
}
hash_check_summary = {
    'check': 'artifact_manifest_hashes_match_current_files',
    'status': 'PASS' if all(row['status'] == 'PASS' for row in manifest_checks) else 'FAIL',
    'checked_count': len(manifest_checks),
    'failed_count': sum(row['status'] != 'PASS' for row in manifest_checks),
    'failed': [row for row in manifest_checks if row['status'] != 'PASS'][:20],
}

strict_checks = [
    clean_run_check,
    notebook_check,
    hash_check_summary,
    tensorboard_check,
    smoke_check,
    handoff_check,
    genai_check,
    *step_checks,
]
failures = [row for row in strict_checks if row.get('status') == 'FAIL']
production_selection_passed = bool(baseline_report.get('production_selection_passed'))
readiness_cap = baseline_report.get('final_readiness_cap', 'staging-ready')
prototype_tradeoffs = baseline_report.get('prototype_tradeoffs', [])

if failures:
    final_status = 'prototype-only'
elif git_state['dirty']:
    final_status = 'staging-ready'
elif readiness_cap == 'production-ready' and production_selection_passed and not prototype_tradeoffs:
    final_status = 'production-ready'
else:
    final_status = 'staging-ready'

summary_rows = [
    {'gate': 'clean kernel reached final gate', 'status': clean_run_check['status']},
    {'gate': 'saved notebook error outputs', 'status': notebook_check['status']},
    {'gate': 'artifact manifest hashes', 'status': hash_check_summary['status']},
    {'gate': 'TensorBoard logs', 'status': tensorboard_check['status']},
    {'gate': '.keras clean reload', 'status': smoke_check['status']},
    {'gate': 'API handoff fixtures', 'status': handoff_check['status']},
    {'gate': 'GenAI training boundary', 'status': genai_check['status']},
    {'gate': 'git dirty production cap', 'status': git_state['status']},
]

final_report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'status': final_status,
    'passed': not failures,
    'final_status_policy': {
        'prototype-only': 'Any strict final gate failure remains unresolved.',
        'staging-ready': 'All strict gates pass, but production is capped by dirty worktree or readiness cap.',
        'production-ready': 'All strict gates pass, production selection passes, no prototype tradeoffs, and git worktree is clean.',
    },
    'final_status_reasons': {
        'failure_count': len(failures),
        'git_dirty_at_export': git_state['dirty'],
        'baseline_readiness_cap': readiness_cap,
        'production_selection_passed': production_selection_passed,
        'prototype_tradeoffs': prototype_tradeoffs,
    },
    'metrics': {
        'validation': training_report.get('metrics', {}).get('validation'),
        'test': training_report.get('metrics', {}).get('test'),
    },
    'requirement_targets': {
        'tensorflow_functional_or_subclassing': model_card.get('model', {}).get('api') or training_report.get('model', {}).get('api'),
        'custom_components': training_report.get('model', {}).get('custom_components'),
        'custom_training_loop': {'uses_gradient_tape': training_report.get('model', {}).get('uses_gradient_tape'), 'uses_model_fit': training_report.get('model', {}).get('uses_model_fit')},
        'mae_target_normalized': {'required_max': 0.02, 'validation': training_report.get('metrics', {}).get('validation', {}).get('mae'), 'test': training_report.get('metrics', {}).get('test', {}).get('mae')},
        'tensorflow_export': model_card.get('deployment_contract', {}).get('final_model_artifact'),
        'tensorboard': tensorboard_check,
        'inference_smoke': {'fixture': rel(CRITICAL_EXPORTS['inference_smoke_fixture']), 'reload_status': smoke_check['status']},
    },
    'strict_checks': strict_checks,
    'git_state': git_state,
    'hash_verification': {
        'critical_hashes': critical_hashes,
        'manifest_checked_count': len(manifest_checks),
        'manifest_failed_count': hash_check_summary['failed_count'],
        'manifest_failures': hash_check_summary['failed'],
        'manifest_sync': manifest_sync,
    },
    'artifact_paths': {
        'final_model': rel(CRITICAL_EXPORTS['final_keras_model']),
        'model_card': rel(CRITICAL_EXPORTS['model_card']),
        'artifact_manifest': rel(CRITICAL_EXPORTS['artifact_manifest']),
        'tensorboard_log_root': rel(TENSORBOARD_DIR),
        'api_handoff_fixtures': rel(CRITICAL_EXPORTS['model_api_handoff_fixtures']),
        'api_handoff_validation': rel(CRITICAL_EXPORTS['model_api_handoff_validation']),
        'final_report': rel(FINAL_REPORT_PATH),
    },
    'known_limitations': model_card.get('known_limitations') or model_card.get('blocked_use'),
    'next_action': 'Commit or otherwise freeze the clean-run artifacts before claiming production-ready status.' if git_state['dirty'] else 'Ready for artifact promotion review.',
}

write_json(FINAL_REPORT_PATH, final_report)

width_gate = max(len(row['gate']) for row in summary_rows)
print(f'Final status: {final_status}')
print('gate'.ljust(width_gate), ' | status')
print('-' * width_gate + '-+-' + '-' * 6)
for row in summary_rows:
    print(row['gate'].ljust(width_gate), ' |', row['status'])
print(f'Final report: {rel(FINAL_REPORT_PATH)}')
if failures:
    print(f'Failures: {len(failures)}')


Final status: staging-ready
gate                             | status
--------------------------------+-------
clean kernel reached final gate  | PASS
saved notebook error outputs     | PASS
artifact manifest hashes         | PASS
TensorBoard logs                 | PASS
.keras clean reload              | PASS
API handoff fixtures             | PASS
GenAI training boundary          | PASS
git dirty production cap         | WARN
Final report: reports/phase_25_tensorflow_training_delivery.json


## Step 25.15 — Human-readable final report

### Purpose
End the notebook with one compact summary that reviewers can read without scanning code cells.

### Required input
- Final clean-kernel gate report from Step 25.14.
- Requirement matrix, artifact export report, calibration/model-card report, and Model API handoff report.

### Action
- Summarize requirement compliance, key metrics, gate status, artifact paths, model version, API handoff files, known limitations, and next action.
- Write the same summary to `reports/phase_25_human_readable_final_report.md`.

### Expected output
- One bounded Markdown summary and one durable report file.

### Gate
Pass when the summary includes all required review fields and reads from current exported reports only.


In [61]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'GAP_MODEL_TRAINING.md').exists() and (candidate / 'REQUIREMENT.md').exists():
            return candidate
    raise RuntimeError('Repo root not found. Start notebook from repository or child directory.')


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / 'reports'
FINAL_REPORT_PATH = REPORTS / 'phase_25_tensorflow_training_delivery.json'
REQ_MATRIX_PATH = REPORTS / 'phase_25_requirement_contract_matrix.json'
ARTIFACT_EXPORT_PATH = REPORTS / 'phase_25_tensorflow_artifact_export.json'
HANDOFF_REPORT_PATH = REPORTS / 'phase_25_model_api_handoff_fixtures.json'
CALIBRATION_REPORT_PATH = REPORTS / 'phase_25_calibration_model_card_export.json'
HUMAN_REPORT_PATH = REPORTS / 'phase_25_human_readable_final_report.md'
HUMAN_REPORT_AUDIT_PATH = REPORTS / 'phase_25_human_readable_final_report.json'


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def file_meta(path: Path) -> dict[str, Any]:
    return {
        'path': str(path.relative_to(ROOT)),
        'sha256': sha256_file(path),
        'size_bytes': path.stat().st_size,
    }


def pct(value: float | int | None, digits: int = 2) -> str:
    if value is None:
        return 'n/a'
    return f'{float(value):.{digits}f}'


def rel(path: str | Path | None) -> str:
    if not path:
        return 'n/a'
    return str(path)


final_report = load_json(FINAL_REPORT_PATH)
req_matrix = load_json(REQ_MATRIX_PATH)
artifact_export = load_json(ARTIFACT_EXPORT_PATH)
handoff_report = load_json(HANDOFF_REPORT_PATH)
calibration_report = load_json(CALIBRATION_REPORT_PATH)

status = final_report.get('status', 'unknown')
passed = bool(final_report.get('passed'))
reasons = final_report.get('final_status_reasons', {})
metrics = final_report.get('metrics', {})
val_metrics = metrics.get('validation', {})
test_metrics = metrics.get('test', {})
requirement_targets = final_report.get('requirement_targets', {})
model_info = artifact_export.get('model', {})
artifact_paths = final_report.get('artifact_paths', {})
acceptance = handoff_report.get('acceptance', {})
calibration_acceptance = calibration_report.get('acceptance', {})

requirement_rows = req_matrix.get('requirementMatrix', [])
requirement_status = [
    ('1.1 TensorFlow architecture', requirement_targets.get('tensorflow_functional_or_subclassing', model_info.get('api', 'PASS'))),
    ('1.2 Custom component', ', '.join(requirement_targets.get('custom_components', model_info.get('custom_components', []))) or 'PASS'),
    ('1.3 GradientTape loop', 'PASS' if requirement_targets.get('custom_training_loop', {}).get('uses_gradient_tape') and not requirement_targets.get('custom_training_loop', {}).get('uses_model_fit') else 'CHECK'),
    ('1.4 TensorBoard', f"{requirement_targets.get('tensorboard', {}).get('status', 'CHECK')} ({requirement_targets.get('tensorboard', {}).get('event_file_count', 0)} event files)"),
    ('1.5 MAE <= 0.02', f"PASS (test={pct(test_metrics.get('mae'), 4)}, validation={pct(val_metrics.get('mae'), 4)})"),
    ('2.1 TensorFlow export', f"PASS ({requirement_targets.get('tensorflow_export', {}).get('format', model_info.get('export_format', '.keras'))})"),
    ('2.2 Inference smoke', requirement_targets.get('inference_smoke', {}).get('reload_status', 'CHECK')),
    ('3.x REST API boundary', 'handoff fixtures exported; server remains separate deliverable'),
    ('4.x GenAI boundary', 'wrapper contract exported; no external GenAI call in training'),
]

key_metrics = [
    ('validation_mae_0_1', pct(val_metrics.get('mae'), 4)),
    ('validation_mae_points', pct(val_metrics.get('mae_0_100'), 3)),
    ('validation_r2', pct(val_metrics.get('r2'), 4)),
    ('validation_spearman', pct(val_metrics.get('spearman'), 4)),
    ('validation_high_fit_recall', pct(val_metrics.get('high_fit_recall'), 4)),
    ('validation_score_band_agreement', pct(val_metrics.get('score_band_agreement'), 4)),
    ('test_mae_0_1', pct(test_metrics.get('mae'), 4)),
    ('test_mae_points', pct(test_metrics.get('mae_0_100'), 3)),
    ('test_r2', pct(test_metrics.get('r2'), 4)),
    ('test_spearman', pct(test_metrics.get('spearman'), 4)),
    ('test_high_fit_recall', pct(test_metrics.get('high_fit_recall'), 4)),
    ('test_score_band_agreement', pct(test_metrics.get('score_band_agreement'), 4)),
]

gates = [
    ('final_status', status),
    ('all_strict_checks_passed', 'PASS' if passed else 'FAIL'),
    ('production_selection_passed', 'PASS' if reasons.get('production_selection_passed') else 'FAIL'),
    ('strict_failure_count', str(reasons.get('failure_count', 'n/a'))),
    ('git_dirty_at_export', 'WARN' if reasons.get('git_dirty_at_export') else 'PASS'),
    ('baseline_readiness_cap', str(reasons.get('baseline_readiness_cap', 'n/a'))),
]

artifact_summary = [
    ('final_model', artifact_paths.get('final_model')),
    ('model_card', artifact_paths.get('model_card')),
    ('artifact_manifest', artifact_paths.get('artifact_manifest')),
    ('tensorboard_log_root', artifact_paths.get('tensorboard_log_root')),
    ('api_handoff_fixtures', artifact_paths.get('api_handoff_fixtures')),
    ('api_handoff_validation', artifact_paths.get('api_handoff_validation')),
]

handoff_summary = [
    ('cv_core_fixture_model_outputs_only', acceptance.get('cv_core_fixture_model_outputs_only')),
    ('candidate_reranking_model_outputs_only', acceptance.get('candidate_reranking_fixture_model_outputs_only')),
    ('score_bounds_enforced', acceptance.get('score_bounds_enforced')),
    ('language_id_en_enforced', acceptance.get('language_id_en_enforced')),
    ('candidate_membership_enforced', acceptance.get('candidate_membership_enforced')),
    ('wrapper_backend_fields_rejected', acceptance.get('wrapper_backend_owned_fields_rejected')),
]

calibration_summary = [
    ('calibration_buckets', 'PASS' if calibration_acceptance.get('calibration_tables_cover_required_buckets') else 'CHECK'),
    ('dataset_manifest_hash', 'PASS' if calibration_acceptance.get('dataset_manifest_exported_with_hash') else 'CHECK'),
    ('label_manifest_hash', 'PASS' if calibration_acceptance.get('label_manifest_exported_with_hash') else 'CHECK'),
    ('feature_config_hash', 'PASS' if calibration_acceptance.get('feature_config_exported_with_hash') else 'CHECK'),
]

known_limitations = final_report.get('known_limitations') or calibration_report.get('model_card_summary', {}).get('blocked_use', [])
next_action = final_report.get('next_action', 'Freeze artifacts and implement separate API runtime.')


def md_table(headers: tuple[str, str], rows: list[tuple[Any, Any]]) -> str:
    lines = [f'| {headers[0]} | {headers[1]} |', '|---|---|']
    for left, right in rows:
        lines.append(f'| {left} | {right} |')
    return '\n'.join(lines)


summary = f"""# Phase 25 Final Human-Readable Report

Generated at: `{datetime.now(timezone.utc).isoformat()}`

## Final decision

- Status: **{status}**
- Model version: `{model_info.get('version', 'n/a')}`
- Model API: `{model_info.get('api', 'n/a')}`
- Score scale: training `0-1`, API `0-100`
- Production cap: {'dirty worktree at export blocks production-ready claim' if reasons.get('git_dirty_at_export') else 'no dirty-worktree cap detected'}

## Requirement compliance

{md_table(('Requirement', 'Evidence'), requirement_status)}

## Key metrics

{md_table(('Metric', 'Value'), key_metrics)}

## Gate status

{md_table(('Gate', 'Status'), gates)}

## Calibration and manifest checks

{md_table(('Check', 'Status'), calibration_summary)}

## Artifact paths

{md_table(('Artifact', 'Path'), [(name, rel(path)) for name, path in artifact_summary])}

## Model API handoff

{md_table(('Validation', 'Status'), handoff_summary)}

Files:
- `{handoff_report.get('export_paths', {}).get('handoff_fixtures', artifact_paths.get('api_handoff_fixtures', 'n/a'))}`
- `{handoff_report.get('export_paths', {}).get('handoff_validation', artifact_paths.get('api_handoff_validation', 'n/a'))}`

## Known limitations

""" + '\n'.join(f'- {item}' for item in known_limitations) + f"""

## Next action

- {next_action}
"""

required_phrases = [
    'Requirement compliance',
    'Key metrics',
    'Gate status',
    'Artifact paths',
    'Model API handoff',
    'Known limitations',
    'Next action',
]
missing = [phrase for phrase in required_phrases if phrase not in summary]
if missing:
    raise RuntimeError(f'Human-readable final report missing sections: {missing}')

HUMAN_REPORT_PATH.write_text(summary + '\n', encoding='utf-8')

source_reports = {
    'final_clean_kernel_gate': FINAL_REPORT_PATH,
    'requirement_contract_matrix': REQ_MATRIX_PATH,
    'tensorflow_artifact_export': ARTIFACT_EXPORT_PATH,
    'model_api_handoff_fixtures': HANDOFF_REPORT_PATH,
    'calibration_model_card_export': CALIBRATION_REPORT_PATH,
}
source_report_meta = {name: file_meta(path) for name, path in source_reports.items() if path.exists()}
human_report_meta = file_meta(HUMAN_REPORT_PATH)

human_acceptance = {
    'report_written': HUMAN_REPORT_PATH.exists(),
    'required_sections_present': not missing,
    'requirement_compliance_included': 'Requirement compliance' in summary,
    'key_metrics_included': 'Key metrics' in summary,
    'gate_status_included': 'Gate status' in summary,
    'artifact_paths_included': 'Artifact paths' in summary,
    'model_api_handoff_included': 'Model API handoff' in summary,
    'known_limitations_included': 'Known limitations' in summary,
    'next_action_included': 'Next action' in summary,
    'reads_current_exported_reports_only': all(path.exists() for path in source_reports.values()),
}

audit_payload = {
    'phase_id': 'phase_25_tensorflow_training_delivery',
    'schema_version': 'phase-25-human-readable-final-report-v1',
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'status': 'complete' if all(human_acceptance.values()) else 'failed',
    'acceptance': human_acceptance,
    'human_report': human_report_meta,
    'source_reports': source_report_meta,
    'report_sections': required_phrases,
}
HUMAN_REPORT_AUDIT_PATH.write_text(json.dumps(audit_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
audit_meta = file_meta(HUMAN_REPORT_AUDIT_PATH)

# Step 25.15 is the final notebook step, so record its durable outputs in the
# final gate report after the human-readable summary is written.
final_report.setdefault('artifact_paths', {})['human_readable_final_report'] = str(HUMAN_REPORT_PATH.relative_to(ROOT))
final_report.setdefault('artifact_paths', {})['human_readable_final_report_audit'] = str(HUMAN_REPORT_AUDIT_PATH.relative_to(ROOT))
critical_hashes = final_report.setdefault('hash_verification', {}).setdefault('critical_hashes', {})
critical_hashes['step_25_15_human_readable_final_report'] = human_report_meta
critical_hashes['step_25_15_human_readable_final_report_audit'] = audit_meta

strict_checks = final_report.setdefault('strict_checks', [])
strict_checks = [row for row in strict_checks if row.get('check') != 'step_25_15_human_readable_final_report']
strict_checks.append({
    'check': 'step_25_15_human_readable_final_report',
    'status': 'PASS' if all(human_acceptance.values()) else 'FAIL',
    'acceptance': human_acceptance,
    'human_report': human_report_meta,
    'audit_report': audit_meta,
})
final_report['strict_checks'] = strict_checks
FINAL_REPORT_PATH.write_text(json.dumps(final_report, indent=2, sort_keys=True) + '\n', encoding='utf-8')

if audit_payload['status'] != 'complete':
    raise RuntimeError(f'Human-readable final report audit failed: {human_acceptance}')

print(summary)
print(f'Human-readable report: {HUMAN_REPORT_PATH.relative_to(ROOT)}')
print(f'Human-readable report audit: {HUMAN_REPORT_AUDIT_PATH.relative_to(ROOT)}')

# Phase 25 Final Human-Readable Report

Generated at: `2026-06-03T01:45:43.809468+00:00`

## Final decision

- Status: **staging-ready**
- Model version: `jobfit_tf_phase25_gradient_tape_v1`
- Model API: `Keras Functional API`
- Score scale: training `0-1`, API `0-100`
- Production cap: dirty worktree at export blocks production-ready claim

## Requirement compliance

| Requirement | Evidence |
|---|---|
| 1.1 TensorFlow architecture | Keras Functional API |
| 1.2 Custom component | CosineInteractionLayer, WeightedHuberLoss, ProductionGateCallback, HighRecallCalibrationLayer |
| 1.3 GradientTape loop | PASS |
| 1.4 TensorBoard | PASS (1 event files) |
| 1.5 MAE <= 0.02 | PASS (test=0.0086, validation=0.0084) |
| 2.1 TensorFlow export | PASS (.keras) |
| 2.2 Inference smoke | PASS |
| 3.x REST API boundary | handoff fixtures exported; server remains separate deliverable |
| 4.x GenAI boundary | wrapper contract exported; no external GenAI call in training |

## Key metrics

| Metric | V